In [ ]:

# ============================================================
# STEP 85A — UA-FLOODNET ABLATION READINESS AUDIT
# No training or checkpoint overwriting
# ============================================================

from pathlib import Path
import ast
import json
import zipfile
import torch
import os

from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
PROJECT = MOUNT / "MyDrive" / "UA_FloodNet"

if not PROJECT.is_dir():
    if MOUNT.exists() and any(MOUNT.iterdir()):
        raise RuntimeError(
            "Mount directory occupied; choose a new mount path."
        )
    MOUNT.mkdir(parents=True, exist_ok=True)
    drive.mount(str(MOUNT))

assert PROJECT.is_dir(), (
    "UA_FloodNet folder not found in MyDrive."
)

print("=" * 75)
print("UA-FLOODNET — ABLATION READINESS AUDIT")
print("=" * 75)
print("Project:", PROJECT)

# ------------------------------------------------------------
# 1. SEARCH ONLY PROJECT FOLDER
# ------------------------------------------------------------

script_names = [
    "ablation_model.py",
    "train_v4.py",
    "train_multiseed.py",
    "data.py"
]

for name in script_names:
    matches = sorted(PROJECT.rglob(name))
    print(f"\n{name}: {len(matches)} copies")

    for path in matches[:10]:
        print(" ", path.relative_to(PROJECT))

# ------------------------------------------------------------
# 2. INSPECT MODEL CLASS AND FUNCTIONS
# ------------------------------------------------------------

model_files = sorted(PROJECT.rglob("ablation_model.py"))

for path in model_files[:3]:
    print("\nMODEL SOURCE:", path.relative_to(PROJECT))

    try:
        tree = ast.parse(path.read_text(encoding="utf-8"))

        for node in tree.body:
            if isinstance(node, ast.ClassDef):
                methods = [
                    item.name for item in node.body
                    if isinstance(item, ast.FunctionDef)
                ]
                print(" Class:", node.name)
                print(" Methods:", methods)

            elif isinstance(node, ast.FunctionDef):
                print(" Function:", node.name)

    except Exception as exc:
        print(" Read error:", exc)

# ------------------------------------------------------------
# 3. FIND CHECKPOINTS
# ------------------------------------------------------------

checkpoints = sorted(PROJECT.rglob("*.pt"))

print("\nCHECKPOINT INVENTORY")
print("Total .pt files:", len(checkpoints))

for path in checkpoints:
    name = str(path.relative_to(PROJECT))
    if any(k in name.lower() for k in [
        "ablation", "multiseed", "control", "light",
        "full", "align", "reliability", "base"
    ]):
        print(
            name,
            "| MB:",
            round(path.stat().st_size / 1048576, 2)
        )

# ------------------------------------------------------------
# 4. READ SAFE CHECKPOINT METADATA
# ------------------------------------------------------------

interesting = [
    p for p in checkpoints
    if any(k in p.name.lower() for k in [
        "full_seed42",
        "align_seed42",
        "reliability_seed42",
        "base_seed42"
    ])
]

for path in interesting[:12]:

    print("\nCHECKPOINT:", path.relative_to(PROJECT))

    try:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        if isinstance(checkpoint, dict):
            print(" Top-level keys:", list(checkpoint)[:15])

            for key in [
                "epoch", "seed", "variant",
                "config", "best_val_iou"
            ]:
                if key in checkpoint:
                    value = checkpoint[key]
                    if isinstance(value, (str, int, float, bool)):
                        print(f" {key}:", value)
                    elif isinstance(value, dict):
                        print(f" {key}:", value)

            weights = checkpoint.get("model_state_dict")

            if weights is None:
                weights = checkpoint.get("state_dict")

            if weights is None and all(
                torch.is_tensor(v)
                for v in checkpoint.values()
            ):
                weights = checkpoint

            if isinstance(weights, dict):
                print(" Model tensors:", len(weights))
                print(" First keys:", list(weights)[:12])

        del checkpoint

    except Exception as exc:
        print(" Could not inspect safely:", repr(exc))

# ------------------------------------------------------------
# 5. SEARCH SOURCE ARCHIVES
# ------------------------------------------------------------

archives = sorted(PROJECT.rglob("*.zip"))

print("\nSOURCE ARCHIVES:", len(archives))

for archive in archives[:30]:
    print(" ", archive.relative_to(PROJECT))

    try:
        with zipfile.ZipFile(archive) as z:
            matches = [
                n for n in z.namelist()
                if any(n.endswith(s) for s in script_names)
            ]
            if matches:
                print("  SOURCE FILES:", matches[:15])
    except Exception:
        pass

# ------------------------------------------------------------
# 6. SAVE AUDIT
# ------------------------------------------------------------

OUTPUT = PROJECT / "v4" / "final_analysis" / "ablation_study"
OUTPUT.mkdir(parents=True, exist_ok=True)

report = {
    "experiment": "UA-FloodNet architectural ablation",
    "step": "85A",
    "script_inventory": {
        name: [
            str(p.relative_to(PROJECT))
            for p in PROJECT.rglob(name)
        ]
        for name in script_names
    },
    "checkpoint_inventory": [
        {
            "path": str(p.relative_to(PROJECT)),
            "size_bytes": p.stat().st_size
        }
        for p in checkpoints
    ],
    "source_archives": [
        str(p.relative_to(PROJECT))
        for p in archives
    ]
}

report_path = OUTPUT / "step85A_readiness_audit.json"

report_path.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 75)
print("STEP 85A COMPLETED")
print("=" * 75)
print("Audit saved:", report_path)
print("No training was started.")
print("No checkpoints were modified.")


In [ ]:

# ============================================================
# STEP 85B — ARCHITECTURAL ABLATION COMPATIBILITY CHECK
# UA-FloodNet | READ-ONLY INSPECTION
# ============================================================

from pathlib import Path
import torch
import ast
import json
import hashlib
import inspect
import sys
import importlib.util

ROOT = Path("/content/ua_ablation_drive/MyDrive/UA_FloodNet")
CODE = ROOT / "v4/code"

assert ROOT.is_dir(), "Drive project not found."
assert CODE.is_dir(), "v4/code directory not found."

print("=" * 75)
print("STEP 85B — ABLATION COMPATIBILITY")
print("=" * 75)

# ------------------------------------------------------------
# 1. CHECK SOURCE FILES AND HASHES
# ------------------------------------------------------------

scripts = [
    "ablation_model.py",
    "train_v4.py",
    "train_multiseed.py",
    "data.py"
]

print("\n1. SOURCE CODE")

for name in scripts:
    path = CODE / name
    assert path.is_file(), f"Missing: {path}"

    digest = hashlib.sha256(path.read_bytes()).hexdigest()

    print(f"\n{name}")
    print(" SHA256:", digest)
    print(" Bytes:", path.stat().st_size)

    tree = ast.parse(path.read_text(encoding="utf-8"))

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
            print(" ", node.__class__.__name__, node.name)

# ------------------------------------------------------------
# 2. INSPECT MODEL CONSTRUCTOR AND LOSSES
# ------------------------------------------------------------

sys.path.insert(0, str(CODE))

spec = importlib.util.spec_from_file_location(
    "ua_ablation_inspection",
    CODE / "ablation_model.py"
)

module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

print("\n2. MODEL IMPLEMENTATION")
print("UAFloodNet:", inspect.signature(module.UAFloodNet))
print("training_loss:", inspect.signature(module.training_loss))

print("\nModel constructor:")
print(inspect.getsource(module.UAFloodNet.__init__))

print("\nLoss function:")
print(inspect.getsource(module.training_loss))

# ------------------------------------------------------------
# 3. READ EXISTING ABLATION CHECKPOINTS
# ------------------------------------------------------------

names = {
    "full": "full_seed42.pt",
    "alignment": "align_seed42.pt",
    "reliability": "reliability_seed42.pt",
    "base": "base_seed42.pt"
}

records = []

print("\n3. EXISTING ABLATION CHECKPOINTS")

for label, filename in names.items():

    path = ROOT / "ablations" / filename

    assert path.is_file(), f"Missing checkpoint: {path}"

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    print("\n" + "-" * 70)
    print("CHECKPOINT:", filename)

    for field in [
        "variant",
        "seed",
        "epoch",
        "width",
        "use_alignment",
        "use_reliability",
        "metrics"
    ]:
        print(field, "=", ckpt.get(field, "NOT RECORDED"))

    weights = ckpt.get("model")

    assert isinstance(weights, dict), (
        f"{filename}: missing model state dictionary"
    )

    params = sum(
        v.numel() for v in weights.values()
        if torch.is_tensor(v)
    )

    print("Stored tensors:", len(weights))
    print("Stored elements:", params)

    records.append({
        "variant": label,
        "checkpoint": filename,
        "seed": ckpt.get("seed"),
        "epoch": ckpt.get("epoch"),
        "width": ckpt.get("width"),
        "use_alignment": ckpt.get("use_alignment"),
        "use_reliability": ckpt.get("use_reliability"),
        "stored_elements": params,
        "metrics": str(ckpt.get("metrics", {}))
    })

# ------------------------------------------------------------
# 4. CHECK FOR PREVIOUS TRAINING OPTIONS
# ------------------------------------------------------------

print("\n4. TRAINING CONFIGURATION EVIDENCE")

for filename in ["train_v4.py", "train_multiseed.py"]:

    path = CODE / filename
    source = path.read_text(encoding="utf-8")

    print("\n" + filename)

    relevant = [
        "argparse",
        "ArgumentParser",
        "add_argument",
        "shift_prob",
        "max_shift",
        "epochs",
        "batch_size",
        "learning_rate",
        "use_alignment",
        "use_reliability",
        "training_loss",
        "optimizer",
        "AdamW",
        "best.pt"
    ]

    for index, line in enumerate(source.splitlines(), 1):
        if any(term in line for term in relevant):
            print(f"{index:4}: {line[:180]}")

# ------------------------------------------------------------
# 5. CHECK DATASET AND FULL-MODEL CHECKPOINTS
# ------------------------------------------------------------

print("\n5. DATASET AND FINAL CHECKPOINTS")

paths = [
    ROOT / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip",
    ROOT / "v4/full/control_seed42_50ep.pt",
    ROOT / "v4/light/light_seed42_50ep.pt",
    ROOT / "v4/multiseed/light_seed7/best.pt",
    ROOT / "v4/multiseed/light_seed17/best.pt",
    ROOT / "v4/multiseed/light_seed37/best.pt"
]

for path in paths:
    print(
        "FOUND" if path.is_file() else "MISSING",
        path.relative_to(ROOT)
    )

# ------------------------------------------------------------
# 6. SAVE COMPATIBILITY AUDIT
# ------------------------------------------------------------

OUT = (
    ROOT / "v4/final_analysis"
    / "ablation_study"
)

OUT.mkdir(parents=True, exist_ok=True)

report_path = OUT / "step85B_compatibility.json"

report_path.write_text(
    json.dumps({
        "records": records,
        "source_files": {
            name: hashlib.sha256(
                (CODE / name).read_bytes()
            ).hexdigest()
            for name in scripts
        }
    }, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 75)
print("STEP 85B COMPLETED")
print("=" * 75)
print("Report:", report_path)
print("No training performed.")
print("Existing checkpoints unchanged.")


In [ ]:

from pathlib import Path
import ast
import zipfile

ROOT = Path("/content/ua_ablation_drive/MyDrive/UA_FloodNet")
CODE = ROOT / "v4/code"

train_path = CODE / "train_multiseed.py"
source = train_path.read_text(encoding="utf-8")
lines = source.splitlines()

print("=" * 76)
print("STEP 85C — MULTISEED ABLATION PREPARATION")
print("=" * 76)

print("\n1. FULL TRAINING SCRIPT")
for i, line in enumerate(lines, 1):
    print(f"{i:03d}: {line}")

print("\n2. CHECK ARCHITECTURE FLAGS")
tree = ast.parse(source)

for node in ast.walk(tree):
    if isinstance(node, ast.keyword):
        if node.arg in ("use_alignment", "use_reliability"):
            print(
                node.arg,
                "=",
                ast.unparse(node.value),
                "| line",
                node.lineno
            )

print("\n3. DATASET DIRECTORIES")

locations = [
    Path("/content/UA_FloodNet/dataset"),
    Path("/content/UA_FloodNet"),
    ROOT / "dataset",
    ROOT / "data"
]

for path in locations:
    if path.exists():
        print("FOUND:", path)
        for name in ("train", "val", "test"):
            folder = path / name
            if folder.is_dir():
                print(
                    " ",
                    name,
                    len(list(folder.glob("*.npz"))),
                    "NPZ files"
                )

print("\n4. DATASET BACKUP")

archive = ROOT / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"

assert archive.is_file(), "Dataset ZIP not found."

with zipfile.ZipFile(archive) as z:
    names = z.namelist()

    for part in ("train/", "val/", "test/"):
        count = sum(
            name.endswith(".npz") and part in name
            for name in names
        )
        print(part, count)

    print("\nArchive examples:")
    for name in names[:12]:
        print(" ", name)

print("\nSTEP 85C INSPECTION FINISHED")
print("No source files or checkpoints were modified.")


In [ ]:

# ============================================================
# STEP 85D-FIX — CORRECT IMPORT PATH AND RUN FIRST ABLATION
# No changes to original UA-FloodNet source files
# ============================================================

from pathlib import Path
import os
import sys
import subprocess
import torch

ROOT = Path("/content/ua_ablation_drive/MyDrive/UA_FloodNet")
CODE = ROOT / "v4/code"

OUT = ROOT / "v4/final_analysis/ablation_study"
SCRIPT = OUT / "train_ablation_multiseed.py"

DATA = Path("/content/UA_FloodNet/dataset")

assert CODE.is_dir(), "Source code directory missing."
assert SCRIPT.is_file(), "Ablation trainer missing."

for name in [
    "ablation_model.py",
    "data.py",
    "train_v4.py"
]:
    assert (CODE / name).is_file(), f"Missing dependency: {name}"

for partition, expected in [
    ("train", 499),
    ("val", 125),
    ("test", 70)
]:
    folder = DATA / partition

    assert folder.is_dir(), f"Dataset folder missing: {folder}"
    assert len(list(folder.glob("*.npz"))) == expected, (
        f"Unexpected number of {partition} patches"
    )

assert torch.cuda.is_available(), (
    "Enable GPU in Colab before training."
)

VARIANT = "no_alignment"
SEED = 7

RUN = OUT / "runs" / f"{VARIANT}_light_seed{SEED}"
RUN.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. FIX THE PYTHON MODULE SEARCH PATH
# ------------------------------------------------------------

env = os.environ.copy()

existing_pythonpath = env.get("PYTHONPATH", "")

env["PYTHONPATH"] = os.pathsep.join(
    part for part in [
        str(CODE),
        existing_pythonpath
    ]
    if part
)

# Verify imports before starting a long training run.
test_command = [
    sys.executable,
    "-c",
    (
        "from ablation_model import UAFloodNet, training_loss; "
        "from data import NPZFloodDataset; "
        "from train_v4 import augment_misregistration; "
        "print('All UA-FloodNet imports successful')"
    )
]

test = subprocess.run(
    test_command,
    cwd=str(CODE),
    env=env,
    capture_output=True,
    text=True
)

print(test.stdout)

if test.returncode != 0:
    print(test.stderr)
    raise RuntimeError("Dependency import test failed.")

# ------------------------------------------------------------
# 2. LAUNCH ABLATION TRAINING
# ------------------------------------------------------------

cmd = [
    sys.executable,
    "-u",
    str(SCRIPT),
    "--train", str(DATA / "train"),
    "--val", str(DATA / "val"),
    "--out-dir", str(RUN),
    "--seed", str(SEED),
    "--variant", VARIANT,
    "--epochs", "50",
    "--batch-size", "4",
    "--width", "24",
    "--lr", "0.001",
    "--shift-prob", "0.25",
    "--max-shift", "2"
]

print("=" * 75)
print("STEP 85D — CORRECTED ABLATION TRAINING")
print("=" * 75)
print("Variant:", VARIANT)
print("Seed:", SEED)
print("Device:", torch.cuda.get_device_name(0))
print("Output:", RUN)
print("Python path:", env["PYTHONPATH"])
print("=" * 75)

LOG = RUN / "training_console.log"

with LOG.open("a", encoding="utf-8") as log:

    process = subprocess.Popen(
        cmd,
        cwd=str(CODE),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Training failed with exit code {return_code}. "
        f"Check {LOG}"
    )

# ------------------------------------------------------------
# 3. VERIFY FINISHED TRAINING
# ------------------------------------------------------------

BEST = RUN / "best.pt"
LAST = RUN / "last.pt"

assert BEST.is_file(), "Best checkpoint missing."
assert LAST.is_file(), "Resume checkpoint missing."

best = torch.load(
    BEST,
    map_location="cpu",
    weights_only=True
)

last = torch.load(
    LAST,
    map_location="cpu",
    weights_only=True
)

assert best["variant"] == "no_alignment"
assert best["use_alignment"] is False
assert best["use_reliability"] is True
assert best["seed"] == 7

assert last["epoch"] == 50, (
    "Training did not reach epoch 50."
)

print("\n" + "=" * 75)
print("STEP 85D COMPLETED SUCCESSFULLY")
print("=" * 75)

print("Variant:", best["variant"])
print("Seed:", best["seed"])
print("Best epoch:", best["epoch"])
print("Best validation IoU:", best["metrics"]["flood_iou"])
print("Best validation F1:", best["metrics"]["flood_f1"])
print("Completed epochs:", last["epoch"])
print("Best checkpoint:", BEST)
print("Training log:", LOG)


In [ ]:

# ============================================================
# STEP 85E — COMPLETE NO-ALIGNMENT ABLATION
# Seeds 17, 37, 42 | 50 epochs each
# Preserves completed seed-7 run
# ============================================================

from pathlib import Path
import os
import sys
import subprocess
import csv
import torch

ROOT = Path("/content/ua_ablation_drive/MyDrive/UA_FloodNet")
CODE = ROOT / "v4/code"
OUT = ROOT / "v4/final_analysis/ablation_study"
SCRIPT = OUT / "train_ablation_multiseed.py"
DATA = Path("/content/UA_FloodNet/dataset")

VARIANT = "no_alignment"
SEEDS = [7, 17, 37, 42]
EPOCHS = 50

assert SCRIPT.is_file()
assert torch.cuda.is_available(), "GPU runtime required."

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    p for p in [str(CODE), env.get("PYTHONPATH", "")]
    if p
)

def read_checkpoint(path):
    return torch.load(
        path, map_location="cpu", weights_only=True
    )

def verify_completed_run(run, seed):
    best_path = run / "best.pt"
    last_path = run / "last.pt"

    if not best_path.exists() or not last_path.exists():
        return False

    last = read_checkpoint(last_path)
    best = read_checkpoint(best_path)

    if (
        last.get("epoch", -1) < EPOCHS
        or best.get("seed") != seed
        or best.get("variant") != VARIANT
        or best.get("use_alignment") is not False
        or best.get("use_reliability") is not True
    ):
        return False

    config = last.get("config", {})
    if (
        config.get("seed") != seed
        or config.get("variant") != VARIANT
        or config.get("shift_prob") != 0.25
        or config.get("max_shift") != 2
        or config.get("width") != 24
        or config.get("batch_size") != 4
    ):
        return False

    return True

for seed in SEEDS:

    run = OUT / "runs" / f"{VARIANT}_light_seed{seed}"
    run.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 75)
    print(f"NO ALIGNMENT — SEED {seed}")
    print("=" * 75)

    if verify_completed_run(run, seed):
        print("Already completed and verified. Skipping.")
        continue

    command = [
        sys.executable, "-u", str(SCRIPT),
        "--train", str(DATA / "train"),
        "--val", str(DATA / "val"),
        "--out-dir", str(run),
        "--seed", str(seed),
        "--variant", VARIANT,
        "--epochs", str(EPOCHS),
        "--batch-size", "4",
        "--width", "24",
        "--lr", "0.001",
        "--shift-prob", "0.25",
        "--max-shift", "2"
    ]

    log_path = run / "training_console.log"

    with log_path.open("a", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            cwd=str(CODE),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        exit_code = process.wait()

    if exit_code != 0:
        raise RuntimeError(
            f"Seed {seed} failed. See {log_path}"
        )

    assert verify_completed_run(run, seed), (
        f"Seed {seed} did not pass completion checks."
    )

# ------------------------------------------------------------
# BUILD FOUR-SEED VALIDATION SUMMARY
# ------------------------------------------------------------

rows = []

for seed in SEEDS:
    run = OUT / "runs" / f"{VARIANT}_light_seed{seed}"

    assert verify_completed_run(run, seed), (
        f"Incomplete or incompatible run: {seed}"
    )

    checkpoint = read_checkpoint(run / "best.pt")

    rows.append({
        "variant": VARIANT,
        "seed": seed,
        "best_epoch": checkpoint["epoch"],
        "validation_iou": checkpoint["metrics"]["flood_iou"],
        "validation_f1": checkpoint["metrics"]["flood_f1"]
    })

summary = OUT / "no_alignment_four_seed_summary.csv"

with summary.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

print("\n" + "=" * 75)
print("STEP 85E — FOUR-SEED SUMMARY")
print("=" * 75)

for row in rows:
    print(
        f"Seed {row['seed']}: "
        f"best epoch={row['best_epoch']}, "
        f"IoU={row['validation_iou']:.5f}, "
        f"F1={row['validation_f1']:.5f}"
    )

mean_iou = sum(r["validation_iou"] for r in rows) / 4

print(f"\nMean candidate-validation IoU: {mean_iou:.5f}")
print("Summary saved:", summary)
print("All four no-alignment seeds verified.")


In [ ]:

# ============================================================
# STEP 85E-FIX — RESTORE DATA AND RESUME TRAINING
# UA-FloodNet | No Alignment | Seeds 37 and 42
# ============================================================

from pathlib import Path
import os
import sys
import zipfile
import subprocessk
import torch

ROOT = Path("/content/ua_ablation_drive/MyDrive/UA_FloodNet")
CODE = ROOT / "v4/code"
OUT = ROOT / "v4/final_analysis/ablation_study"
SCRIPT = OUT / "train_ablation_multiseed.py"

ARCHIVE = ROOT / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"
DATA = Path("/content/UA_FloodNet/dataset")

assert ARCHIVE.is_file(), "Dataset backup ZIP missing."
assert SCRIPT.is_file(), "Ablation training script missing."
assert torch.cuda.is_available(), "Enable Colab GPU runtime."

print("=" * 75)
print("RECOVERING UA-FLOODNET ABLATION TRAINING")
print("=" * 75)

# ------------------------------------------------------------
# 1. RESTORE DATASET
# ------------------------------------------------------------

DATA.mkdir(parents=True, exist_ok=True)

expected = {
    "train": 499,
    "val": 125,
    "test": 70
}

with zipfile.ZipFile(ARCHIVE) as z:

    for partition, expected_count in expected.items():

        files = [
            info for info in z.infolist()
            if (
                not info.is_dir()
                and info.filename.endswith(".npz")
                and Path(info.filename).parts[0] == partition
            )
        ]

        assert len(files) == expected_count, (
            f"Incorrect {partition} count in ZIP."
        )

        folder = DATA / partition
        folder.mkdir(parents=True, exist_ok=True)

        for info in files:
            destination = DATA / info.filename

            assert destination.resolve().is_relative_to(
                DATA.resolve()
            ), "Unsafe ZIP path"

            # Restore files missing from temporary Colab storage.
            if (
                not destination.exists()
                or destination.stat().st_size != info.file_size
            ):
                z.extract(info, DATA)

        actual = len(list(folder.glob("*.npz")))

        assert actual == expected_count, (
            f"{partition}: expected {expected_count}, got {actual}"
        )

        print(
            f"{partition.upper()}: "
            f"{actual}/{expected_count} patches restored"
        )

# ------------------------------------------------------------
# 2. CHECK SAVED CHECKPOINTS
# ------------------------------------------------------------

def safe_load(path):
    return torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

variant = "no_alignment"

for seed in [7, 17, 37, 42]:

    run = OUT / "runs" / f"{variant}_light_seed{seed}"
    last_path = run / "last.pt"

    if last_path.exists():
        ckpt = safe_load(last_path)
        epoch = ckpt.get("epoch", 0)
        print(
            f"Seed {seed}: saved through epoch {epoch}"
        )
    else:
        print(f"Seed {seed}: no resume checkpoint")

# ------------------------------------------------------------
# 3. RESUME INCOMPLETE SEEDS
# ------------------------------------------------------------

env = os.environ.copy()

env["PYTHONPATH"] = os.pathsep.join(
    x for x in [
        str(CODE),
        env.get("PYTHONPATH", "")
    ] if x
)

for seed in [37, 42]:

    run = OUT / "runs" / f"{variant}_light_seed{seed}"
    run.mkdir(parents=True, exist_ok=True)

    last_path = run / "last.pt"

    if last_path.exists():

        ckpt = safe_load(last_path)
        saved_epoch = ckpt["epoch"]

        if saved_epoch >= 50:
            print(
                f"\nSeed {seed} already completed. Skipping."
            )
            continue

        print(
            f"\nSeed {seed}: resuming at "
            f"epoch {saved_epoch + 1}"
        )

    else:
        print(f"\nSeed {seed}: starting at epoch 1")

    command = [
        sys.executable,
        "-u",
        str(SCRIPT),
        "--train", str(DATA / "train"),
        "--val", str(DATA / "val"),
        "--out-dir", str(run),
        "--seed", str(seed),
        "--variant", variant,
        "--epochs", "50",
        "--batch-size", "4",
        "--width", "24",
        "--lr", "0.001",
        "--shift-prob", "0.25",
        "--max-shift", "2"
    ]

    log_path = run / "training_console.log"

    with log_path.open("a", encoding="utf-8") as log:

        process = subprocess.Popen(
            command,
            cwd=str(CODE),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        exit_code = process.wait()

    if exit_code != 0:
        raise RuntimeError(
            f"Seed {seed} failed again. "
            f"See {log_path}"
        )

    finished = safe_load(last_path)

    assert finished["epoch"] == 50, (
        f"Seed {seed} did not finish 50 epochs."
    )

    best = safe_load(run / "best.pt")

    print("\n" + "=" * 65)
    print(f"SEED {seed} COMPLETED")
    print("=" * 65)
    print("Best epoch:", best["epoch"])
    print(
        "Best validation IoU:",
        best["metrics"]["flood_iou"]
    )
    print(
        "Best validation F1:",
        best["metrics"]["flood_f1"]
    )

print("\nAll requested recovery runs finished.")


In [ ]:

from pathlib import Path
import torch

from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")

if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = (
    MOUNT / "MyDrive/UA_FloodNet/v4/"
    "final_analysis/ablation_study/runs"
)

print("=" * 65)
print("CHECKING SAVED ABLATION PROGRESS")
print("=" * 65)

for seed in [7, 17, 37, 42]:
    run = ROOT / f"no_alignment_light_seed{seed}"

    last = run / "last.pt"
    best = run / "best.pt"

    print(f"\nSEED {seed}")

    if last.exists():
        try:
            checkpoint = torch.load(
                last,
                map_location="cpu",
                weights_only=True
            )

            epoch = checkpoint.get("epoch", 0)

            print("Last saved epoch:", epoch)
            print("Resume from epoch:", epoch + 1)

        except Exception as error:
            print("Checkpoint read error:", error)
    else:
        print("No resume checkpoint found")

    if best.exists():
        try:
            checkpoint = torch.load(
                best,
                map_location="cpu",
                weights_only=True
            )

            print("Best epoch:", checkpoint.get("epoch"))
            print(
                "Best validation IoU:",
                checkpoint.get("metrics", {}).get("flood_iou")
            )
        except Exception as error:
            print("Best checkpoint read error:", error)
    else:
        print("No best checkpoint found")


In [ ]:

from pathlib import Path
import os
import sys
import zipfile
import subprocess
import torch

ROOT = Path("/content/ua_ablation_drive/MyDrive/UA_FloodNet")
CODE = ROOT / "v4/code"
OUT = ROOT / "v4/final_analysis/ablation_study"
SCRIPT = OUT / "train_ablation_multiseed.py"
DATA = Path("/content/UA_FloodNet/dataset")
ARCHIVE = ROOT / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"

# Restore dataset if Colab temporary files disappeared.
expected = {"train": 499, "val": 125, "test": 70}

with zipfile.ZipFile(ARCHIVE) as z:
    for partition, count in expected.items():
        folder = DATA / partition
        folder.mkdir(parents=True, exist_ok=True)
        files = [
            info for info in z.infolist()
            if info.filename.startswith(partition + "/")
            and info.filename.endswith(".npz")
        ]
        assert len(files) == count

        for info in files:
            target = DATA / info.filename
            assert target.resolve().is_relative_to(DATA.resolve())
            if not target.exists() or target.stat().st_size != info.file_size:
                z.extract(info, DATA)

        assert len(list(folder.glob("*.npz"))) == count
        print(partition, ":", count, "patches ready")

SEED = 42
VARIANT = "no_alignment"
RUN = OUT / "runs" / f"{VARIANT}_light_seed{SEED}"

last_path = RUN / "last.pt"
assert last_path.is_file(), "Seed 42 resume checkpoint missing."

last = torch.load(
    last_path, map_location="cpu", weights_only=True
)

assert last["config"]["seed"] == SEED
assert last["config"]["variant"] == VARIANT
assert last["config"]["shift_prob"] == 0.25
assert last["config"]["max_shift"] == 2

saved_epoch = last["epoch"]

print("\nSaved epoch:", saved_epoch)
print("Next epoch:", saved_epoch + 1)

if saved_epoch < 50:
    assert torch.cuda.is_available(), "Enable GPU runtime."

    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join(
        p for p in [str(CODE), env.get("PYTHONPATH", "")]
        if p
    )

    command = [
        sys.executable, "-u", str(SCRIPT),
        "--train", str(DATA / "train"),
        "--val", str(DATA / "val"),
        "--out-dir", str(RUN),
        "--seed", "42",
        "--variant", "no_alignment",
        "--epochs", "50",
        "--batch-size", "4",
        "--width", "24",
        "--lr", "0.001",
        "--shift-prob", "0.25",
        "--max-shift", "2"
    ]

    with (RUN / "training_console.log").open("a") as log:
        process = subprocess.Popen(
            command,
            cwd=str(CODE),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        code = process.wait()

    if code != 0:
        raise RuntimeError("Seed 42 training failed.")

finished = torch.load(
    last_path, map_location="cpu", weights_only=True
)
best = torch.load(
    RUN / "best.pt", map_location="cpu", weights_only=True
)

assert finished["epoch"] == 50

print("\n" + "=" * 60)
print("ALL FOUR NO-ALIGNMENT RUNS COMPLETED")
print("=" * 60)
print("Seed 42 best epoch:", best["epoch"])
print("Seed 42 best IoU:", best["metrics"]["flood_iou"])
print("Seed 42 best F1:", best["metrics"]["flood_f1"])


In [ ]:

# ============================================================
# STEP 86 — NO-RELIABILITY ABLATION
# Four seeds | 50 epochs | Resume supported
# ============================================================

from pathlib import Path
import os
import sys
import csv
import zipfile
import subprocess
import torch

from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
CODE = ROOT / "v4/code"
OUT = ROOT / "v4/final_analysis/ablation_study"
SCRIPT = OUT / "train_ablation_multiseed.py"
ARCHIVE = ROOT / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"
DATA = Path("/content/UA_FloodNet/dataset")

VARIANT = "no_reliability"
SEEDS = [7, 17, 37, 42]
EPOCHS = 50

assert SCRIPT.is_file(), "Ablation trainer missing"
assert ARCHIVE.is_file(), "Dataset archive missing"
assert torch.cuda.is_available(), "Enable a GPU runtime"

# ------------------------------------------------------------
# 1. RESTORE DATASET
# ------------------------------------------------------------

expected = {"train": 499, "val": 125, "test": 70}

with zipfile.ZipFile(ARCHIVE) as z:
    for part, count in expected.items():
        members = [
            m for m in z.infolist()
            if m.filename.startswith(part + "/")
            and m.filename.endswith(".npz")
            and not m.is_dir()
        ]
        assert len(members) == count

        folder = DATA / part
        folder.mkdir(parents=True, exist_ok=True)

        for member in members:
            target = DATA / member.filename
            assert target.resolve().is_relative_to(DATA.resolve())
            if (
                not target.exists()
                or target.stat().st_size != member.file_size
            ):
                z.extract(member, DATA)

        assert len(list(folder.glob("*.npz"))) == count
        print(f"{part}: {count} patches ready")

# ------------------------------------------------------------
# 2. PREPARE IMPORT PATH
# ------------------------------------------------------------

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    p for p in [str(CODE), env.get("PYTHONPATH", "")]
    if p
)

def load(path):
    return torch.load(
        path, map_location="cpu", weights_only=True
    )

def completed(run, seed):
    last_file = run / "last.pt"
    best_file = run / "best.pt"

    if not last_file.is_file() or not best_file.is_file():
        return False

    last = load(last_file)
    best = load(best_file)
    config = last.get("config", {})

    return (
        last.get("epoch") == EPOCHS
        and config.get("variant") == VARIANT
        and config.get("seed") == seed
        and config.get("shift_prob") == 0.25
        and config.get("max_shift") == 2
        and best.get("variant") == VARIANT
        and best.get("seed") == seed
        and best.get("use_alignment") is True
        and best.get("use_reliability") is False
    )

# ------------------------------------------------------------
# 3. TRAIN OR RESUME EACH SEED
# ------------------------------------------------------------

for seed in SEEDS:
    run = OUT / "runs" / f"{VARIANT}_light_seed{seed}"
    run.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 70)
    print(f"NO RELIABILITY | SEED {seed}")
    print("=" * 70)

    if completed(run, seed):
        print("Already completed; skipping.")
        continue

    last_file = run / "last.pt"

    if last_file.is_file():
        saved = load(last_file)
        config = saved.get("config", {})
        assert config.get("variant") == VARIANT
        assert config.get("seed") == seed
        print("Resuming from epoch:", saved["epoch"] + 1)
    else:
        print("Starting from epoch 1")

    command = [
        sys.executable, "-u", str(SCRIPT),
        "--train", str(DATA / "train"),
        "--val", str(DATA / "val"),
        "--out-dir", str(run),
        "--seed", str(seed),
        "--variant", VARIANT,
        "--epochs", str(EPOCHS),
        "--batch-size", "4",
        "--width", "24",
        "--lr", "0.001",
        "--shift-prob", "0.25",
        "--max-shift", "2"
    ]

    with (run / "training_console.log").open(
        "a", encoding="utf-8"
    ) as log:

        process = subprocess.Popen(
            command,
            cwd=str(CODE),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Seed {seed} failed; inspect training_console.log"
        )

    assert completed(run, seed)
    print(f"Seed {seed} completed successfully.")

# ------------------------------------------------------------
# 4. CREATE RESULTS SUMMARY
# ------------------------------------------------------------

rows = []

for seed in SEEDS:
    run = OUT / "runs" / f"{VARIANT}_light_seed{seed}"
    assert completed(run, seed)

    best = load(run / "best.pt")

    rows.append({
        "variant": VARIANT,
        "seed": seed,
        "best_epoch": best["epoch"],
        "validation_iou": best["metrics"]["flood_iou"],
        "validation_f1": best["metrics"]["flood_f1"]
    })

summary = OUT / "no_reliability_four_seed_summary.csv"

with summary.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

print("\n" + "=" * 70)
print("NO-RELIABILITY ABLATION COMPLETED")
print("=" * 70)

for row in rows:
    print(
        f"Seed {row['seed']} | "
        f"Best epoch {row['best_epoch']} | "
        f"IoU {row['validation_iou']:.5f} | "
        f"F1 {row['validation_f1']:.5f}"
    )

print(
    "\nMean validation IoU:",
    round(
        sum(row["validation_iou"] for row in rows)
        / len(rows), 5
    )
)

print("Results saved:", summary)


In [ ]:

# ============================================================
# STEP 87 — BASE ARCHITECTURAL ABLATION
# Alignment OFF | Reliability OFF
# Seeds 7, 17, 37, 42 | 50 epochs
# ============================================================

from pathlib import Path
import os, sys, csv, zipfile, subprocess, torch
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
CODE = ROOT / "v4/code"
OUT = ROOT / "v4/final_analysis/ablation_study"
SCRIPT = OUT / "train_ablation_multiseed.py"
ARCHIVE = ROOT / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"
DATA = Path("/content/UA_FloodNet/dataset")

VARIANT = "base"
SEEDS = [7, 17, 37, 42]
EPOCHS = 50

assert SCRIPT.is_file(), "Ablation trainer not found"
assert ARCHIVE.is_file(), "Dataset archive not found"
assert torch.cuda.is_available(), "Enable GPU in Colab"

# ------------------------------------------------------------
# 1. RESTORE FROZEN DATASET
# ------------------------------------------------------------

expected = {"train": 499, "val": 125, "test": 70}

with zipfile.ZipFile(ARCHIVE) as z:
    for split, count in expected.items():
        members = [
            item for item in z.infolist()
            if (
                not item.is_dir()
                and item.filename.endswith(".npz")
                and Path(item.filename).parts[0] == split
            )
        ]
        assert len(members) == count

        folder = DATA / split
        folder.mkdir(parents=True, exist_ok=True)

        for item in members:
            target = DATA / item.filename
            assert target.resolve().is_relative_to(DATA.resolve())

            if (
                not target.is_file()
                or target.stat().st_size != item.file_size
            ):
                z.extract(item, DATA)

        assert len(list(folder.glob("*.npz"))) == count
        print(f"{split}: {count} patches ready")

# ------------------------------------------------------------
# 2. VERIFY CHECKPOINTS
# ------------------------------------------------------------

def load(path):
    return torch.load(
        path, map_location="cpu", weights_only=True
    )

def verify(run, seed):
    best_path = run / "best.pt"
    last_path = run / "last.pt"

    if not (best_path.is_file() and last_path.is_file()):
        return False

    best = load(best_path)
    last = load(last_path)
    config = last.get("config", {})

    return (
        last.get("epoch") == EPOCHS
        and config.get("seed") == seed
        and config.get("variant") == VARIANT
        and config.get("shift_prob") == 0.25
        and config.get("max_shift") == 2
        and config.get("batch_size") == 4
        and config.get("width") == 24
        and config.get("lr") == 0.001
        and best.get("seed") == seed
        and best.get("variant") == VARIANT
        and best.get("use_alignment") is False
        and best.get("use_reliability") is False
    )

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    p for p in [str(CODE), env.get("PYTHONPATH", "")]
    if p
)

# ------------------------------------------------------------
# 3. TRAIN OR RESUME EACH SEED
# ------------------------------------------------------------

for seed in SEEDS:

    RUN = OUT / "runs" / f"base_light_seed{seed}"
    RUN.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 70)
    print(f"BASE MODEL — SEED {seed}")
    print("=" * 70)

    if verify(RUN, seed):
        print("Already completed — skipping.")
        continue

    last_path = RUN / "last.pt"

    if last_path.is_file():
        last = load(last_path)
        config = last.get("config", {})

        assert config.get("variant") == VARIANT
        assert config.get("seed") == seed
        assert config.get("shift_prob") == 0.25
        assert config.get("max_shift") == 2
        assert config.get("batch_size") == 4
        assert config.get("width") == 24
        assert config.get("lr") == 0.001

        print("Resuming from epoch:", last["epoch"] + 1)
    else:
        print("Starting from epoch 1")

    command = [
        sys.executable, "-u", str(SCRIPT),
        "--train", str(DATA / "train"),
        "--val", str(DATA / "val"),
        "--out-dir", str(RUN),
        "--seed", str(seed),
        "--variant", VARIANT,
        "--epochs", str(EPOCHS),
        "--batch-size", "4",
        "--width", "24",
        "--lr", "0.001",
        "--shift-prob", "0.25",
        "--max-shift", "2"
    ]

    LOG = RUN / "training_console.log"

    with LOG.open("a", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            cwd=str(CODE),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Seed {seed} failed. Inspect {LOG}"
        )

    assert verify(RUN, seed), (
        f"Seed {seed} failed checkpoint verification"
    )

    best = load(RUN / "best.pt")
    print(
        f"Seed {seed} complete: "
        f"best epoch {best['epoch']}, "
        f"IoU {best['metrics']['flood_iou']:.5f}"
    )

# ------------------------------------------------------------
# 4. GENERATE FOUR-SEED SUMMARY
# ------------------------------------------------------------

rows = []

for seed in SEEDS:
    RUN = OUT / "runs" / f"base_light_seed{seed}"
    assert verify(RUN, seed)

    best = load(RUN / "best.pt")

    rows.append({
        "variant": VARIANT,
        "seed": seed,
        "best_epoch": best["epoch"],
        "validation_iou": best["metrics"]["flood_iou"],
        "validation_f1": best["metrics"]["flood_f1"]
    })

SUMMARY = OUT / "base_four_seed_summary.csv"

with SUMMARY.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print("\n" + "=" * 70)
print("STEP 87 — BASE ABLATION COMPLETED")
print("=" * 70)

for row in rows:
    print(
        f"Seed {row['seed']} | "
        f"Best epoch {row['best_epoch']} | "
        f"IoU {row['validation_iou']:.5f} | "
        f"F1 {row['validation_f1']:.5f}"
    )

print(
    "\nMean validation IoU:",
    round(
        sum(r["validation_iou"] for r in rows) / 4,
        5
    )
)
print("Results saved:", SUMMARY)


In [ ]:

# ============================================================
# STEP 88A — VERIFY ALL 16 ARCHITECTURE CHECKPOINTS
# Read-only: does not retrain or modify models
# ============================================================

from pathlib import Path
import torch

ROOT = Path("/content/ua_ablation_drive/MyDrive/UA_FloodNet")
OUT = ROOT / "v4/final_analysis/ablation_study"

SEEDS = [7, 17, 37, 42]

VARIANTS = {
    "full": (True, True),
    "no_alignment": (False, True),
    "no_reliability": (True, False),
    "base": (False, False)
}

def checkpoint_path(variant, seed):
    if variant == "full":
        if seed == 42:
            return ROOT / "v4/light/light_seed42_50ep.pt"
        return ROOT / f"v4/multiseed/light_seed{seed}/best.pt"

    return (
        OUT / "runs" /
        f"{variant}_light_seed{seed}" / "best.pt"
    )

all_valid = True

print("=" * 76)
print("STEP 88A — CHECKPOINT VERIFICATION")
print("=" * 76)

for variant, (expected_align, expected_reliability) in VARIANTS.items():
    print(f"\n{variant.upper()}")

    for seed in SEEDS:
        path = checkpoint_path(variant, seed)

        if not path.is_file():
            print(f"Seed {seed}: MISSING — {path}")
            all_valid = False
            continue

        try:
            ckpt = torch.load(
                path, map_location="cpu", weights_only=True
            )

            flags_ok = (
                ckpt.get("use_alignment") == expected_align
                and ckpt.get("use_reliability") == expected_reliability
            )

            seed_ok = ckpt.get("seed") == seed
            has_weights = isinstance(ckpt.get("model"), dict)

            valid = flags_ok and seed_ok and has_weights

            print(
                f"Seed {seed}: "
                f"{'PASS' if valid else 'REVIEW'} | "
                f"epoch={ckpt.get('epoch')} | "
                f"alignment={ckpt.get('use_alignment')} | "
                f"reliability={ckpt.get('use_reliability')} | "
                f"validation={ckpt.get('metrics', {})}"
            )

            if not valid:
                all_valid = False

        except Exception as exc:
            print(f"Seed {seed}: ERROR — {exc}")
            all_valid = False

print("\n" + "=" * 76)

if all_valid:
    print("ALL 16 CHECKPOINTS VERIFIED")
    print("Ready for common-protocol evaluation.")
else:
    print("Some checkpoints need review before evaluation.")
    print("Do not retrain automatically.")


In [ ]:

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Ready for UA-FloodNet training!")
else:
    print("No GPU detected. Change the Colab runtime.")


In [ ]:

# ================================================================
# STEP 88B — UA-FLOODNET COMMON-PROTOCOL ABLATION EVALUATION
# 4 architectures x 4 seeds x 13 displacement conditions
# Candidate validation + official held-out test
# READ-ONLY MODEL EVALUATION — NO RETRAINING
# ================================================================

from pathlib import Path
import os, sys, csv, zipfile, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
CODE = ROOT / "v4/code"
OUT = ROOT / "v4/final_analysis/ablation_study"
RESULTS = OUT / "evaluation_88B"
RESULTS.mkdir(parents=True, exist_ok=True)

DATA = Path("/content/UA_FloodNet/dataset")
ARCHIVE = ROOT / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"

sys.path.insert(0, str(CODE))
from ablation_model import UAFloodNet
from data import NPZFloodDataset

SEEDS = [7, 17, 37, 42]

VARIANTS = {
    "full": (True, True),
    "no_alignment": (False, True),
    "no_reliability": (True, False),
    "base": (False, False)
}

# (horizontal pixels, vertical pixels)
SHIFTS = [
    (0, 0),
    (-1, 0), (1, 0),
    (-2, 0), (2, 0),
    (-4, 0), (4, 0),
    (-8, 0), (8, 0),
    (0, -2), (0, 2),
    (0, -4), (0, 4)
]

CROP = 20
BATCH_SIZE = 4
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("STEP 88B — COMMON-PROTOCOL EVALUATION")
print("=" * 72)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ---------------------------------------------------------------
# 1. RESTORE FROZEN DATASET
# ---------------------------------------------------------------

expected = {"train": 499, "val": 125, "test": 70}

with zipfile.ZipFile(ARCHIVE) as z:
    for split, count in expected.items():
        members = [
            m for m in z.infolist()
            if not m.is_dir()
            and m.filename.endswith(".npz")
            and Path(m.filename).parts[0] == split
        ]
        assert len(members) == count

        folder = DATA / split
        folder.mkdir(parents=True, exist_ok=True)

        for m in members:
            target = DATA / m.filename
            assert target.resolve().is_relative_to(
                DATA.resolve()
            )
            if (
                not target.is_file()
                or target.stat().st_size != m.file_size
            ):
                z.extract(m, DATA)

        assert len(list(folder.glob("*.npz"))) == count
        print(f"{split}: {count} patches ready")

# ---------------------------------------------------------------
# 2. CHECKPOINT LOCATIONS
# ---------------------------------------------------------------

def checkpoint_path(variant, seed):
    if variant == "full":
        if seed == 42:
            return ROOT / "v4/light/light_seed42_50ep.pt"
        return (
            ROOT / "v4/multiseed"
            / f"light_seed{seed}" / "best.pt"
        )

    return (
        OUT / "runs"
        / f"{variant}_light_seed{seed}" / "best.pt"
    )

# ---------------------------------------------------------------
# 3. EXACT ZERO-PADDED INTEGER TRANSLATION
# ---------------------------------------------------------------

def translate_zero(x, dx, dy):
    """
    dx > 0 moves pixels right.
    dy > 0 moves pixels down.
    Uncovered pixels are zero padded.
    No wraparound or interpolation.
    """
    if dx == 0 and dy == 0:
        return x

    h, w = x.shape[-2:]
    y = torch.zeros_like(x)

    if abs(dx) >= w or abs(dy) >= h:
        return y

    sx0 = max(0, -dx)
    sx1 = min(w, w - dx)
    sy0 = max(0, -dy)
    sy1 = min(h, h - dy)

    tx0 = sx0 + dx
    tx1 = sx1 + dx
    ty0 = sy0 + dy
    ty1 = sy1 + dy

    y[..., ty0:ty1, tx0:tx1] = (
        x[..., sy0:sy1, sx0:sx1]
    )

    return y

# Verify displacement direction.
probe = torch.zeros(1, 1, 9, 9)
probe[..., 4, 4] = 1
assert translate_zero(probe, 2, 0)[0, 0, 4, 6] == 1
assert translate_zero(probe, 0, -2)[0, 0, 2, 4] == 1

# ---------------------------------------------------------------
# 4. LOAD EXACT ARCHITECTURE
# ---------------------------------------------------------------

def load_model(variant, seed):
    path = checkpoint_path(variant, seed)

    assert path.is_file(), f"Missing: {path}"

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    align, reliability = VARIANTS[variant]

    assert ckpt.get("seed") == seed
    assert ckpt.get("use_alignment") == align
    assert ckpt.get("use_reliability") == reliability

    model = UAFloodNet(
        width=ckpt.get("width", 24),
        use_alignment=align,
        use_reliability=reliability
    )

    model.load_state_dict(
        ckpt["model"],
        strict=True
    )

    model = model.to(DEVICE)
    model.eval()
    return model

# ---------------------------------------------------------------
# 5. EVALUATE ONE DISPLACEMENT
# ---------------------------------------------------------------

@torch.inference_mode()
def evaluate_shift(model, loader, dx, dy):

    tp = fp = fn = 0

    for batch in loader:
        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        ms_pre = translate_zero(batch["ms_pre"], dx, dy)
        ms_post = translate_zero(batch["ms_post"], dx, dy)

        logits = model(
            batch["sar_pre"],
            batch["sar_post"],
            ms_pre,
            ms_post
        )["logits"]

        pred = logits.sigmoid() >= 0.5
        truth = batch["mask"] >= 0.5

        # Consistent 20-pixel interior crop.
        pred = pred[..., CROP:-CROP, CROP:-CROP]
        truth = truth[..., CROP:-CROP, CROP:-CROP]

        tp += int((pred & truth).sum().item())
        fp += int((pred & ~truth).sum().item())
        fn += int((~pred & truth).sum().item())

    union = tp + fp + fn
    iou = tp / union if union else 1.0

    denom = 2 * tp + fp + fn
    f1 = 2 * tp / denom if denom else 1.0

    return {
        "iou": float(iou),
        "f1": float(f1),
        "tp": tp,
        "fp": fp,
        "fn": fn
    }

# ---------------------------------------------------------------
# 6. RESUMABLE RESULT STORAGE
# ---------------------------------------------------------------

DETAIL = RESULTS / "all_displacement_results.csv"

FIELDS = [
    "partition", "variant", "seed", "dx", "dy",
    "iou", "f1", "tp", "fp", "fn"
]

def read_results():
    if DETAIL.exists():
        df = pd.read_csv(DETAIL)
        assert set(FIELDS).issubset(df.columns)
        assert not df.duplicated(
            ["partition", "variant", "seed", "dx", "dy"]
        ).any(), "Duplicate evaluation records detected"
        return df[FIELDS]
    return pd.DataFrame(columns=FIELDS)

def save_results(df):
    temp = DETAIL.with_suffix(".tmp")
    df.to_csv(temp, index=False)
    os.replace(temp, DETAIL)

df = read_results()

loaders = {
    name: DataLoader(
        NPZFloodDataset(str(DATA / name)),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )
    for name in ("val", "test")
}

total_expected = 2 * 4 * 4 * len(SHIFTS)
print("Expected evaluation records:", total_expected)
print("Previously completed:", len(df))

# ---------------------------------------------------------------
# 7. RUN ALL MISSING EVALUATIONS
# ---------------------------------------------------------------

for partition, loader in loaders.items():
    for variant in VARIANTS:
        for seed in SEEDS:

            existing = set(
                zip(
                    df["partition"],
                    df["variant"],
                    df["seed"],
                    df["dx"],
                    df["dy"]
                )
            )

            missing = [
                (dx, dy) for dx, dy in SHIFTS
                if (partition, variant, seed, dx, dy)
                not in existing
            ]

            if not missing:
                print(
                    f"SKIP {partition} / {variant} / seed {seed}"
                )
                continue

            print(
                f"\n{partition.upper()} | "
                f"{variant} | seed {seed} | "
                f"{len(missing)} conditions remaining",
                flush=True
            )

            model = load_model(variant, seed)

            for dx, dy in missing:
                result = evaluate_shift(
                    model, loader, dx, dy
                )

                row = {
                    "partition": partition,
                    "variant": variant,
                    "seed": seed,
                    "dx": dx,
                    "dy": dy,
                    **result
                }

                df = pd.concat(
                    [df, pd.DataFrame([row])],
                    ignore_index=True
                )

                save_results(df)

                print(
                    f"  ({dx:+d},{dy:+d}) "
                    f"IoU={result['iou']:.5f} "
                    f"F1={result['f1']:.5f} "
                    f"[{len(df)}/{total_expected}]",
                    flush=True
                )

            del model

            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

# ---------------------------------------------------------------
# 8. VERIFY COVERAGE AND GENERATE SUMMARY
# ---------------------------------------------------------------

df = read_results()

assert len(df) == total_expected, (
    f"Expected {total_expected} records, got {len(df)}"
)

rows = []

for (partition, variant, seed), group in df.groupby(
    ["partition", "variant", "seed"]
):
    assert len(group) == 13

    clean = group[
        (group["dx"] == 0) & (group["dy"] == 0)
    ]["iou"].iloc[0]

    shifted = group[
        ~((group["dx"] == 0) & (group["dy"] == 0))
    ]["iou"]

    rows.append({
        "partition": partition,
        "variant": variant,
        "seed": int(seed),
        "clean_iou": float(clean),
        "mean_shifted_iou": float(shifted.mean()),
        "iou_degradation": float(clean - shifted.mean()),
        "worst_shifted_iou": float(shifted.min())
    })

summary = pd.DataFrame(rows).sort_values(
    ["partition", "variant", "seed"]
)

SUMMARY_PATH = RESULTS / "four_seed_architecture_summary.csv"
summary.to_csv(SUMMARY_PATH, index=False)

means = (
    summary.groupby(["partition", "variant"])[
        [
            "clean_iou",
            "mean_shifted_iou",
            "iou_degradation",
            "worst_shifted_iou"
        ]
    ]
    .mean()
    .round(5)
)

print("\n" + "=" * 72)
print("STEP 88B — COMPLETED")
print("=" * 72)
print("\nFOUR-SEED MEAN PERFORMANCE")
print(means.to_string())

print("\nSaved detailed results:", DETAIL)
print("Saved four-seed summary:", SUMMARY_PATH)
print("All 416 architecture/seed/shift/partition")
print("evaluation records verified.")


In [ ]:

# ============================================================
# STEP 89 — UA-FLOODNET ARCHITECTURAL ABLATION STATISTICS
# Four matched seeds | Candidate validation + official test
# No model training or changes to checkpoint files
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")

if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
EVAL = (
    ROOT / "v4/final_analysis"
    / "ablation_study/evaluation_88B"
)

OUT = EVAL / "step89_statistics"
OUT.mkdir(parents=True, exist_ok=True)

SOURCE = EVAL / "four_seed_architecture_summary.csv"
DETAIL = EVAL / "all_displacement_results.csv"

assert SOURCE.is_file(), "Step 88B summary missing."
assert DETAIL.is_file(), "Step 88B detailed results missing."

df = pd.read_csv(SOURCE)
detail = pd.read_csv(DETAIL)

seeds = [7, 17, 37, 42]
variants = [
    "full",
    "no_alignment",
    "no_reliability",
    "base"
]
partitions = ["val", "test"]

metrics = [
    "clean_iou",
    "mean_shifted_iou",
    "iou_degradation",
    "worst_shifted_iou"
]

labels = {
    "full": "Full UA-FloodNet",
    "no_alignment": "No alignment",
    "no_reliability": "No reliability",
    "base": "Base model"
}

print("=" * 75)
print("STEP 89 — ARCHITECTURAL ABLATION STATISTICS")
print("=" * 75)

# ------------------------------------------------------------
# 1. DATA INTEGRITY CHECKS
# ------------------------------------------------------------

assert len(df) == 32, "Expected 32 summary records."
assert len(detail) == 416, "Expected 416 displacement records."

assert not df.duplicated(
    ["partition", "variant", "seed"]
).any()

assert not detail.duplicated(
    ["partition", "variant", "seed", "dx", "dy"]
).any()

for part in partitions:
    for variant in variants:
        sub = df[
            (df.partition == part)
            & (df.variant == variant)
        ]
        assert sorted(sub.seed.tolist()) == seeds

assert df[metrics].notna().all().all()

print("Integrity checks passed.")
print("Summary records:", len(df))
print("Displacement records:", len(detail))

# ------------------------------------------------------------
# 2. FOUR-SEED DESCRIPTIVE STATISTICS
# ------------------------------------------------------------

descriptive = (
    df.groupby(["partition", "variant"])[metrics]
    .agg(["mean", "std"])
)

descriptive.to_csv(
    OUT / "Table_Ab1_Descriptive_Statistics.csv"
)

print("\nFOUR-SEED DESCRIPTIVE STATISTICS")
print(descriptive.round(5).to_string())

# ------------------------------------------------------------
# 3. PAIRED STUDENT T TESTS
# Difference = ablated model minus full model
# ------------------------------------------------------------

rows = []

for part in partitions:

    reference = (
        df[
            (df.partition == part)
            & (df.variant == "full")
        ]
        .set_index("seed")
        .loc[seeds]
    )

    for variant in variants[1:]:

        comparison = (
            df[
                (df.partition == part)
                & (df.variant == variant)
            ]
            .set_index("seed")
            .loc[seeds]
        )

        for metric in metrics:

            difference = (
                comparison[metric].to_numpy(dtype=float)
                - reference[metric].to_numpy(dtype=float)
            )

            n = len(difference)
            mean = float(difference.mean())
            sd = float(difference.std(ddof=1))
            se = sd / np.sqrt(n)

            if np.isclose(sd, 0, atol=1e-14):
                if np.isclose(mean, 0, atol=1e-14):
                    p = 1.0
                    ci_low = ci_high = mean
                else:
                    p = 0.0
                    ci_low = ci_high = mean
            else:
                test = stats.ttest_1samp(difference, 0)
                p = float(test.pvalue)

                critical = stats.t.ppf(0.975, df=n-1)
                ci_low = mean - critical * se
                ci_high = mean + critical * se

            rows.append({
                "partition": part,
                "comparison": variant + " minus full",
                "variant": variant,
                "metric": metric,
                "n_seeds": n,
                "mean_difference": mean,
                "sd_difference": sd,
                "ci95_lower": ci_low,
                "ci95_upper": ci_high,
                "p_unadjusted": p
            })

paired = pd.DataFrame(rows)

# Holm correction across all 24 exploratory tests.
paired["p_holm"] = multipletests(
    paired["p_unadjusted"].to_numpy(),
    alpha=0.05,
    method="holm"
)[1]

paired["holm_significant_005"] = paired["p_holm"] < 0.05

paired.to_csv(
    OUT / "Table_Ab2_Paired_Comparisons.csv",
    index=False
)

print("\nPAIRED COMPARISONS — OFFICIAL TEST")
test_rows = paired[paired.partition == "test"]

print(
    test_rows[
        [
            "comparison",
            "metric",
            "mean_difference",
            "ci95_lower",
            "ci95_upper",
            "p_unadjusted",
            "p_holm"
        ]
    ].round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 4. FIGURE — ABSOLUTE SEGMENTATION PERFORMANCE
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False
})

test = df[df.partition == "test"]

summary = (
    test.groupby("variant")[
        ["clean_iou", "mean_shifted_iou"]
    ].mean().reindex(variants)
)

x = np.arange(len(variants))
width = 0.36

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    x - width/2,
    summary["clean_iou"],
    width,
    label="Clean IoU"
)

ax.bar(
    x + width/2,
    summary["mean_shifted_iou"],
    width,
    label="Mean shifted IoU"
)

ax.set_xticks(x)
ax.set_xticklabels([labels[v] for v in variants])
ax.set_ylabel("Flood intersection-over-union")
ax.set_ylim(0, 0.85)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.18)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig(
    OUT / "Figure_Ab1_Architecture_Performance.png",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 5. FIGURE — ROBUSTNESS / IOU DEGRADATION
# ------------------------------------------------------------

degradation = (
    test.groupby("variant")["iou_degradation"]
    .mean().reindex(variants)
)

fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    [labels[v] for v in variants],
    degradation.values
)

ax.set_ylabel(
    "Mean IoU degradation under optical displacement"
)

ax.grid(axis="y", alpha=0.18)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig(
    OUT / "Figure_Ab2_Robustness_Degradation.png",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 6. FIGURE — PAIRED EFFECTS WITH 95% CI
# ------------------------------------------------------------

effect = paired[
    (paired.partition == "test")
    & (paired.metric == "clean_iou")
].set_index("variant").loc[variants[1:]]

positions = np.arange(len(effect))

means = effect["mean_difference"].to_numpy()

left = (
    means - effect["ci95_lower"].to_numpy()
)
right = (
    effect["ci95_upper"].to_numpy() - means
)

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.errorbar(
    means,
    positions,
    xerr=np.vstack([left, right]),
    fmt="o",
    capsize=5
)

ax.axvline(0, linestyle="--", linewidth=1)

ax.set_yticks(positions)
ax.set_yticklabels([
    labels[v] for v in variants[1:]
])

ax.invert_yaxis()
ax.set_xlabel(
    "Clean IoU difference relative to full UA-FloodNet"
)

ax.grid(axis="x", alpha=0.18)

fig.tight_layout()
fig.savefig(
    OUT / "Figure_Ab3_Paired_Effects_95CI.png",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 7. DISPLACEMENT SENSITIVITY CURVES
# ------------------------------------------------------------

test_detail = detail[
    detail.partition == "test"
].copy()

curve = (
    test_detail.groupby(
        ["variant", "dx", "dy"],
        as_index=False
    )["iou"].mean()
)

order = [
    (-8,0),(-4,0),(-2,0),(-1,0),
    (0,-4),(0,-2),(0,0),(0,2),(0,4),
    (1,0),(2,0),(4,0),(8,0)
]

fig, ax = plt.subplots(figsize=(10, 5.5))

for variant in variants:

    part = curve[curve.variant == variant]
    lookup = {
        (int(r.dx), int(r.dy)): r.iou
        for r in part.itertuples()
    }

    y = [lookup[p] for p in order]

    ax.plot(
        range(len(order)),
        y,
        marker="o",
        linewidth=1.5,
        markersize=4,
        label=labels[variant]
    )

ax.set_xticks(range(len(order)))
ax.set_xticklabels(
    [f"({a},{b})" for a,b in order],
    rotation=45,
    ha="right"
)

ax.set_ylabel("Mean test IoU")
ax.set_xlabel("Optical displacement (dx, dy), pixels")
ax.legend(frameon=False)
ax.grid(alpha=0.18)

fig.tight_layout()
fig.savefig(
    OUT / "Figure_Ab4_Displacement_Sensitivity.png",
    dpi=600,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 8. COMPLETION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("STEP 89 COMPLETED")
print("=" * 75)

print("Paired comparisons:", len(paired))
print(
    "Holm-significant exploratory comparisons:",
    int(paired["holm_significant_005"].sum())
)

print("\nSaved output directory:", OUT)

for path in sorted(OUT.iterdir()):
    print(" ", path.name)

print("\nReady for Step 90 manuscript integration.")


In [ ]:

# ==============================================================
# STEP 89B — ISPRS-STYLE ABLATION FIGURES
# Paired observations, confidence intervals, displacement curves
# No retraining or synthetic data
# ==============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
EVAL = ROOT / "v4/final_analysis/ablation_study/evaluation_88B"
SOURCE = EVAL / "four_seed_architecture_summary.csv"
DETAIL = EVAL / "all_displacement_results.csv"
OUT = EVAL / "step89_statistics/ISPRS_redesigned"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SOURCE)
detail = pd.read_csv(DETAIL)

assert len(df) == 32
assert len(detail) == 416

variants = [
    "full", "no_alignment", "no_reliability", "base"
]
names = [
    "Full model", "No alignment",
    "No reliability", "Base model"
]
seeds = [7, 17, 37, 42]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

def save_figure(fig, name):
    for extension in ["png", "pdf", "svg"]:
        kwargs = (
            {"dpi": 600}
            if extension == "png" else {}
        )
        fig.savefig(
            OUT / f"{name}.{extension}",
            bbox_inches="tight",
            pad_inches=0.08,
            **kwargs
        )
    plt.show()
    plt.close(fig)

# --------------------------------------------------------------
# FIGURE A — MATCHED SEED PERFORMANCE
# --------------------------------------------------------------

test = df[df.partition == "test"].copy()

fig, axes = plt.subplots(
    1, 2, figsize=(10, 4.1),
    constrained_layout=True
)

plot_metrics = [
    ("clean_iou", "(a) Aligned imagery"),
    ("mean_shifted_iou", "(b) Displaced imagery")
]

markers = ["o", "s", "^", "D"]

for ax, (metric, title) in zip(axes, plot_metrics):

    for seed, marker in zip(seeds, markers):
        part = (
            test[test.seed == seed]
            .set_index("variant")
            .loc[variants]
        )

        ax.plot(
            range(4),
            part[metric].to_numpy(),
            linewidth=0.9,
            alpha=0.55,
            marker=marker,
            markersize=4,
            label=f"Seed {seed}"
        )

    means = (
        test.groupby("variant")[metric]
        .mean()
        .reindex(variants)
    )

    ax.scatter(
        range(4),
        means,
        marker="_",
        s=240,
        linewidths=3,
        color="black",
        zorder=5,
        label="Four-seed mean"
    )

    ax.set_xticks(range(4))
    ax.set_xticklabels(names, rotation=20, ha="right")
    ax.set_ylabel("Flood IoU")
    ax.set_title(title, loc="left")
    ax.grid(axis="y", alpha=0.15)
    ax.spines[["top", "right"]].set_visible(False)

handles, labels_ = axes[0].get_legend_handles_labels()

fig.legend(
    handles, labels_,
    loc="outside upper center",
    ncol=5,
    frameon=False
)

save_figure(fig, "Fig_A_Matched_Seed_Performance")

# --------------------------------------------------------------
# FIGURE B — PAIRED EFFECTS AND 95% CONFIDENCE INTERVALS
# Difference = ablated variant minus full model
# --------------------------------------------------------------

metrics = [
    ("clean_iou", "Clean IoU"),
    ("mean_shifted_iou", "Mean shifted IoU"),
    ("iou_degradation", "IoU degradation"),
    ("worst_shifted_iou", "Worst-shift IoU")
]

fig, axes = plt.subplots(
    2, 2, figsize=(10, 6.5),
    constrained_layout=True
)

reference = (
    test[test.variant == "full"]
    .set_index("seed").loc[seeds]
)

for idx, (ax, (metric, label)) in enumerate(
    zip(axes.flat, metrics)
):
    for j, variant in enumerate(variants[1:]):

        other = (
            test[test.variant == variant]
            .set_index("seed").loc[seeds]
        )

        differences = (
            other[metric].to_numpy()
            - reference[metric].to_numpy()
        )

        mean = differences.mean()
        se = differences.std(ddof=1) / np.sqrt(4)
        margin = t.ppf(0.975, 3) * se

        ax.errorbar(
            mean, j,
            xerr=margin,
            fmt="o",
            markersize=5,
            capsize=4,
            linewidth=1.3
        )

        ax.text(
            mean, j - 0.19,
            f"{mean:+.4f}",
            fontsize=7.5,
            ha="center"
        )

    ax.axvline(
        0,
        color="black",
        linestyle="--",
        linewidth=0.9
    )

    ax.set_yticks(range(3))
    ax.set_yticklabels(names[1:])
    ax.set_ylim(2.6, -0.6)
    ax.set_title(
        f"({chr(97 + idx)}) {label}",
        loc="left"
    )
    ax.set_xlabel("Difference relative to full model")
    ax.grid(axis="x", alpha=0.15)
    ax.spines[["top", "right"]].set_visible(False)

save_figure(fig, "Fig_B_Paired_Effects_95CI")

# --------------------------------------------------------------
# FIGURE C — DIRECTION-SPECIFIC MISREGISTRATION
# Proper numeric horizontal/vertical displacement axes
# --------------------------------------------------------------

sub = detail[detail.partition == "test"].copy()

fig, axes = plt.subplots(
    1, 2, figsize=(11, 4.4),
    constrained_layout=True
)

panels = [
    ("dx", "dy", "(a) Horizontal optical displacement"),
    ("dy", "dx", "(b) Vertical optical displacement")
]

for ax, (axis_key, fixed_key, title) in zip(
    axes, panels
):

    for variant, name in zip(variants, names):

        selected = sub[
            (sub.variant == variant)
            & (sub[fixed_key] == 0)
        ]

        pivot = selected.pivot(
            index=axis_key,
            columns="seed",
            values="iou"
        ).sort_index()

        assert list(pivot.columns) == seeds

        x = pivot.index.to_numpy(dtype=float)
        values = pivot.to_numpy(dtype=float)

        mean = values.mean(axis=1)
        sd = values.std(axis=1, ddof=1)

        line, = ax.plot(
            x, mean,
            marker="o",
            linewidth=1.7,
            markersize=3.5,
            label=name
        )

        ax.fill_between(
            x,
            mean - sd,
            mean + sd,
            color=line.get_color(),
            alpha=0.08,
            linewidth=0
        )

    ax.axvline(
        0, linestyle=":",
        linewidth=0.8, color="black"
    )

    ax.set_xticks(sorted(sub[axis_key].unique()))
    ax.set_xlabel("Optical displacement (pixels)")
    ax.set_ylabel("Mean flood IoU")
    ax.set_title(title, loc="left")
    ax.grid(alpha=0.15)
    ax.spines[["top", "right"]].set_visible(False)

axes[0].legend(
    frameon=False,
    fontsize=8,
    loc="lower center"
)

save_figure(fig, "Fig_C_Directional_Misregistration")

print("\n" + "=" * 65)
print("STEP 89B COMPLETED")
print("=" * 65)

for path in sorted(OUT.iterdir()):
    print(path.name)

print("\nSaved to:", OUT)


In [ ]:

# ============================================================
# STEP 91A — STANDARD U-NET BENCHMARK
# OMBRIA | 4 seeds | 50 epochs | Resumable
# Does not overwrite UA-FloodNet checkpoints
# ============================================================

from pathlib import Path
import os, sys, json, random, zipfile, subprocess, textwrap
import torch
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
CODE = ROOT / "v4/code"
OUT = ROOT / "v4/final_analysis/benchmark_study"
OUT.mkdir(parents=True, exist_ok=True)

ARCHIVE = ROOT / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"
DATA = Path("/content/UA_FloodNet/dataset")

assert ARCHIVE.is_file()
assert torch.cuda.is_available(), "Enable GPU runtime."

# ------------------------------------------------------------
# 1. RESTORE FROZEN OMBRIA DATASET
# ------------------------------------------------------------

expected = {"train": 499, "val": 125, "test": 70}

with zipfile.ZipFile(ARCHIVE) as z:
    for split, count in expected.items():
        members = [
            m for m in z.infolist()
            if not m.is_dir()
            and m.filename.endswith(".npz")
            and Path(m.filename).parts[0] == split
        ]
        assert len(members) == count

        folder = DATA / split
        folder.mkdir(parents=True, exist_ok=True)

        for member in members:
            target = DATA / member.filename
            assert target.resolve().is_relative_to(
                DATA.resolve()
            )
            if (
                not target.exists()
                or target.stat().st_size != member.file_size
            ):
                z.extract(member, DATA)

        assert len(list(folder.glob("*.npz"))) == count
        print(f"{split}: {count} patches verified")

# ------------------------------------------------------------
# 2. CREATE A SEPARATE BENCHMARK TRAINER
# ------------------------------------------------------------

SCRIPT = OUT / "train_unet_benchmark.py"

SOURCE = r'''
import argparse, json, random, os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from data import NPZFloodDataset
from train_v4 import augment_misregistration


def block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True)
    )


class UNet(nn.Module):
    def __init__(self, width=24):
        super().__init__()
        w = width
        self.e1 = block(8, w)
        self.e2 = block(w, w*2)
        self.e3 = block(w*2, w*4)
        self.e4 = block(w*4, w*8)

        self.middle = block(w*8, w*16)

        self.d4 = block(w*16 + w*8, w*8)
        self.d3 = block(w*8 + w*4, w*4)
        self.d2 = block(w*4 + w*2, w*2)
        self.d1 = block(w*2 + w, w)

        self.head = nn.Conv2d(w, 1, 1)

    def forward(self, batch):
        x = torch.cat([
            batch["sar_pre"], batch["sar_post"],
            batch["ms_pre"], batch["ms_post"]
        ], dim=1)

        pool = F.max_pool2d

        a = self.e1(x)
        b = self.e2(pool(a, 2))
        c = self.e3(pool(b, 2))
        d = self.e4(pool(c, 2))
        m = self.middle(pool(d, 2))

        def up(x, skip):
            return torch.cat([
                F.interpolate(
                    x, size=skip.shape[-2:],
                    mode="bilinear",
                    align_corners=False
                ), skip
            ], dim=1)

        x = self.d4(up(m, d))
        x = self.d3(up(x, c))
        x = self.d2(up(x, b))
        x = self.d1(up(x, a))

        return self.head(x)


def segmentation_loss(logits, truth):
    truth = truth.float()

    bce = F.binary_cross_entropy_with_logits(
        logits, truth
    )

    probabilities = logits.sigmoid()

    dice = 1 - (
        2 * (probabilities * truth).sum((1, 2, 3)) + 1
    ) / (
        (probabilities + truth).sum((1, 2, 3)) + 1
    )

    return bce + dice.mean()


@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    tp = fp = fn = 0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        prediction = model(batch).sigmoid() >= 0.5
        truth = batch["mask"] >= 0.5

        tp += int((prediction & truth).sum())
        fp += int((prediction & ~truth).sum())
        fn += int((~prediction & truth).sum())

    return {
        "flood_iou": tp / max(tp + fp + fn, 1),
        "flood_f1": 2*tp / max(2*tp + fp + fn, 1)
    }


def save_atomic(value, path):
    path = Path(path)
    tmp = path.with_suffix(".tmp")
    torch.save(value, tmp)
    os.replace(tmp, path)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--train", required=True)
    p.add_argument("--val", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--seed", type=int, required=True)
    p.add_argument("--epochs", type=int, default=50)
    args = p.parse_args()

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    config = {
        "model": "standard_unet",
        "seed": args.seed,
        "train": str(Path(args.train).resolve()),
        "val": str(Path(args.val).resolve()),
        "width": 24,
        "batch_size": 4,
        "lr": 0.001,
        "shift_prob": 0.25,
        "max_shift": 2
    }

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)

    generator = torch.Generator()
    generator.manual_seed(args.seed)

    device = torch.device("cuda")
    model = UNet(width=24).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=0.001)

    train_loader = DataLoader(
        NPZFloodDataset(args.train), batch_size=4,
        shuffle=True, generator=generator, num_workers=0
    )
    val_loader = DataLoader(
        NPZFloodDataset(args.val), batch_size=4,
        shuffle=False, num_workers=0
    )

    last_file = out / "last.pt"
    best_file = out / "best.pt"

    start = 1
    best_iou = -1.0
    history = []

    if last_file.exists():
        ck = torch.load(
            last_file,
            map_location="cpu",
            weights_only=True
        )

        assert ck["config"] == config, (
            "Checkpoint configuration mismatch"
        )

        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["optimizer"])

        for state in opt.state.values():
            for key, value in state.items():
                if torch.is_tensor(value):
                    state[key] = value.to(device)

        generator.set_state(ck["shuffle_rng"])
        torch.set_rng_state(ck["torch_rng"])
        torch.cuda.set_rng_state_all(ck["cuda_rng"])

        if "python_rng" in ck:
            random.setstate(ck["python_rng"])

        if "numpy_rng" in ck:
            ns = ck["numpy_rng"]
            np.random.set_state((
                ns[0], ns[1].numpy(),
                int(ns[2]), int(ns[3]), float(ns[4])
            ))

        start = ck["epoch"] + 1
        best_iou = ck["best_iou"]
        history = ck["history"]
        print("Resuming from epoch", start, flush=True)

    if start > args.epochs:
        print("Already finished", flush=True)
        return

    print(
        "Parameters:",
        sum(p.numel() for p in model.parameters()),
        flush=True
    )

    for epoch in range(start, args.epochs + 1):
        model.train()
        losses = []

        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            batch = augment_misregistration(
                batch, 0.25, 2
            )

            opt.zero_grad(set_to_none=True)

            logits = model(batch)
            loss = segmentation_loss(logits, batch["mask"])

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 5.0
            )
            opt.step()

            losses.append(float(loss.detach()))

        metrics = evaluate(model, val_loader, device)

        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            **metrics
        })

        if metrics["flood_iou"] > best_iou:
            best_iou = metrics["flood_iou"]

            save_atomic({
                "model": model.state_dict(),
                "architecture": "standard_unet",
                "seed": args.seed,
                "epoch": epoch,
                "width": 24,
                "metrics": metrics,
                "config": config
            }, best_file)

        ns = np.random.get_state()

        save_atomic({
            "model": model.state_dict(),
            "optimizer": opt.state_dict(),
            "epoch": epoch,
            "best_iou": best_iou,
            "history": history,
            "config": config,
            "shuffle_rng": generator.get_state(),
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all(),
            "python_rng": random.getstate(),
            "numpy_rng": (
                ns[0], torch.from_numpy(ns[1].copy()),
                ns[2], ns[3], ns[4]
            )
        }, last_file)

        (out / "history.json").write_text(
            json.dumps(history, indent=2)
        )

        print(
            f"epoch={epoch} "
            f"loss={np.mean(losses):.4f} "
            f"val_iou={metrics['flood_iou']:.5f} "
            f"best_iou={best_iou:.5f}",
            flush=True
        )

    print(
        f"FINISHED seed={args.seed} "
        f"best_iou={best_iou:.5f}",
        flush=True
    )


if __name__ == "__main__":
    main()
'''

if SCRIPT.exists():
    assert SCRIPT.read_text() == SOURCE, (
        "Existing benchmark trainer differs; not overwriting."
    )
else:
    SCRIPT.write_text(SOURCE)

print("Benchmark trainer created:", SCRIPT)

# ------------------------------------------------------------
# 3. TRAIN FOUR MATCHED SEEDS
# ------------------------------------------------------------

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    p for p in [str(CODE), env.get("PYTHONPATH", "")]
    if p
)

for seed in [7, 17, 37, 42]:

    RUN = OUT / "standard_unet" / f"seed{seed}"
    RUN.mkdir(parents=True, exist_ok=True)

    if (RUN / "last.pt").exists():
        last = torch.load(
            RUN / "last.pt",
            map_location="cpu",
            weights_only=True
        )

        if last["epoch"] == 50:
            assert (RUN / "best.pt").is_file()
            assert last["config"]["seed"] == seed
            assert last["config"]["model"] == "standard_unet"
            print(f"Seed {seed} complete — skipping.")
            continue

        print(
            f"Seed {seed}: resuming from "
            f"epoch {last['epoch'] + 1}"
        )

    else:
        print(f"Seed {seed}: starting training")

    command = [
        sys.executable, "-u", str(SCRIPT),
        "--train", str(DATA / "train"),
        "--val", str(DATA / "val"),
        "--out", str(RUN),
        "--seed", str(seed),
        "--epochs", "50"
    ]

    LOG = RUN / "training_console.log"

    with LOG.open("a", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            cwd=str(CODE),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        code = process.wait()

    if code != 0:
        raise RuntimeError(
            f"Seed {seed} failed. Inspect {LOG}"
        )

    best = torch.load(
        RUN / "best.pt",
        map_location="cpu",
        weights_only=True
    )

    print(
        f"SEED {seed} COMPLETE | "
        f"Best epoch={best['epoch']} | "
        f"IoU={best['metrics']['flood_iou']:.5f}"
    )

print("\nSTEP 91A — STANDARD U-NET TRAINING COMPLETE")


In [ ]:

# ============================================================
# STEP 92 — SIAMESE AND TWO-STAGE FUSION U-NET BENCHMARKS
# 2 architectures x 4 seeds x 50 epochs
# Preserves existing models; resumes saved checkpoints
# ============================================================

from pathlib import Path
import os
import sys
import hashlib
import zipfile
import subprocess
import torch
import pandas as pd
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
CODE = ROOT / "v4/code"
OUT = ROOT / "v4/final_analysis/benchmark_study"
DATA = Path("/content/UA_FloodNet/dataset")
ARCHIVE = ROOT / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"

ORIGINAL = OUT / "train_unet_benchmark.py"

assert ORIGINAL.is_file(), "Step 91A trainer not found."
assert ARCHIVE.is_file(), "OMBRIA backup not found."
assert torch.cuda.is_available(), "Enable GPU runtime."

# ------------------------------------------------------------
# 1. RESTORE FROZEN DATASET
# ------------------------------------------------------------

expected = {"train": 499, "val": 125, "test": 70}

with zipfile.ZipFile(ARCHIVE) as z:
    for split, count in expected.items():
        members = [
            m for m in z.infolist()
            if not m.is_dir()
            and m.filename.endswith(".npz")
            and Path(m.filename).parts[0] == split
        ]
        assert len(members) == count

        folder = DATA / split
        folder.mkdir(parents=True, exist_ok=True)

        for member in members:
            target = DATA / member.filename
            assert target.resolve().is_relative_to(
                DATA.resolve()
            )
            if (
                not target.is_file()
                or target.stat().st_size != member.file_size
            ):
                z.extract(member, DATA)

        assert len(list(folder.glob("*.npz"))) == count
        print(f"{split}: {count} patches ready")

# ------------------------------------------------------------
# 2. ARCHITECTURE DEFINITIONS
# ------------------------------------------------------------

# Both models expose UNet(width=24) and forward(batch),
# matching the Step 91A trainer interface.

SIAMESE = r'''
class UNet(nn.Module):
    """
    Shared-weight, multiscale temporal Siamese U-Net.
    Each date has SAR(1)+optical(3) = 4 channels.
    Shared encoder weights across pre/post dates.
    """

    def __init__(self, width=24):
        super().__init__()
        w = width

        self.e1 = block(4, w)
        self.e2 = block(w, 2*w)
        self.e3 = block(2*w, 4*w)
        self.e4 = block(4*w, 8*w)
        self.mid = block(8*w, 8*w)

        # Pre, post, and absolute temporal difference.
        self.temporal = block(24*w, 8*w)
        self.d4 = block(8*w + 24*w, 8*w)
        self.d3 = block(8*w + 12*w, 4*w)
        self.d2 = block(4*w + 6*w, 2*w)
        self.d1 = block(2*w + 3*w, w)

        self.head = nn.Conv2d(w, 1, 1)

    def encode(self, x):
        a = self.e1(x)
        b = self.e2(F.max_pool2d(a, 2))
        c = self.e3(F.max_pool2d(b, 2))
        d = self.e4(F.max_pool2d(c, 2))
        m = self.mid(F.max_pool2d(d, 2))
        return [a, b, c, d, m]

    @staticmethod
    def combine(a, b):
        return torch.cat([a, b, (b-a).abs()], dim=1)

    @staticmethod
    def up(x, skip):
        x = F.interpolate(
            x, size=skip.shape[-2:],
            mode="bilinear", align_corners=False
        )
        return torch.cat([x, skip], dim=1)

    def forward(self, batch):
        pre = torch.cat(
            [batch["sar_pre"], batch["ms_pre"]], dim=1
        )
        post = torch.cat(
            [batch["sar_post"], batch["ms_post"]], dim=1
        )

        a = self.encode(pre)
        b = self.encode(post)

        skips = [
            self.combine(x, y)
            for x, y in zip(a, b)
        ]

        x = self.temporal(skips[4])
        x = self.d4(self.up(x, skips[3]))
        x = self.d3(self.up(x, skips[2]))
        x = self.d2(self.up(x, skips[1]))
        x = self.d1(self.up(x, skips[0]))

        return self.head(x)
'''

TWO_STAGE = r'''
class SensorEncoder(nn.Module):
    def __init__(self, in_channels, width):
        super().__init__()
        w = width
        self.e1 = block(in_channels, w)
        self.e2 = block(w, 2*w)
        self.e3 = block(2*w, 4*w)
        self.e4 = block(4*w, 8*w)

    def forward(self, x):
        a = self.e1(x)
        b = self.e2(F.max_pool2d(a, 2))
        c = self.e3(F.max_pool2d(b, 2))
        d = self.e4(F.max_pool2d(c, 2))
        return [a, b, c, d]


class UNet(nn.Module):
    """
    Two-stage fusion:
      1. Separate SAR and optical feature extraction,
         shared between acquisition dates.
      2. Same-date feature fusion at each scale.
      3. Pre/post temporal fusion and decoding.

    No learned geometric alignment or reliability weights.
    """

    def __init__(self, width=24):
        super().__init__()
        w = width

        self.sar = SensorEncoder(1, w)
        self.opt = SensorEncoder(3, w)

        self.fuse = nn.ModuleList([
            block(2*c, c)
            for c in [w, 2*w, 4*w, 8*w]
        ])

        self.temporal = block(24*w, 8*w)
        self.d3 = block(8*w + 12*w, 4*w)
        self.d2 = block(4*w + 6*w, 2*w)
        self.d1 = block(2*w + 3*w, w)

        self.head = nn.Conv2d(w, 1, 1)

    def encode_date(self, sar, opt):
        s = self.sar(sar)
        o = self.opt(opt)

        return [
            f(torch.cat([a, b], dim=1))
            for f, a, b in zip(self.fuse, s, o)
        ]

    @staticmethod
    def combine(a, b):
        return torch.cat([a, b, (b-a).abs()], dim=1)

    @staticmethod
    def up(x, skip):
        x = F.interpolate(
            x, size=skip.shape[-2:],
            mode="bilinear", align_corners=False
        )
        return torch.cat([x, skip], dim=1)

    def forward(self, batch):
        pre = self.encode_date(
            batch["sar_pre"], batch["ms_pre"]
        )
        post = self.encode_date(
            batch["sar_post"], batch["ms_post"]
        )

        skips = [
            self.combine(a, b)
            for a, b in zip(pre, post)
        ]

        x = self.temporal(skips[3])
        x = self.d3(self.up(x, skips[2]))
        x = self.d2(self.up(x, skips[1]))
        x = self.d1(self.up(x, skips[0]))

        return self.head(x)
'''

ARCHITECTURES = {
    "siamese_unet": SIAMESE,
    "two_stage_fusion_unet": TWO_STAGE
}

# ------------------------------------------------------------
# 3. CREATE BENCHMARK SCRIPTS
# ------------------------------------------------------------

source = ORIGINAL.read_text(encoding="utf-8")

start_marker = "class UNet(nn.Module):"
end_marker = "\ndef segmentation_loss("

assert source.count(start_marker) == 1
assert source.count(end_marker) == 1
assert source.count("standard_unet") >= 2

start = source.index(start_marker)
end = source.index(end_marker, start)

script_paths = {}

for name, definition in ARCHITECTURES.items():

    new_source = (
        source[:start]
        + definition.strip()
        + "\n\n"
        + source[end:]
    )

    new_source = new_source.replace(
        "standard_unet", name
    )

    compile(new_source, f"train_{name}.py", "exec")

    script = OUT / f"train_{name}.py"

    if script.exists():
        assert script.read_text(encoding="utf-8") == new_source, (
            f"Existing script differs: {script}. "
            "No file has been overwritten."
        )
    else:
        script.write_text(new_source, encoding="utf-8")

    script_paths[name] = script

    print(
        "\nCreated/verified:", script.name,
        "\nSHA256:",
        hashlib.sha256(script.read_bytes()).hexdigest()
    )

# ------------------------------------------------------------
# 4. RESTORE IMPORT PATH AND RUN SHAPE TESTS
# ------------------------------------------------------------

env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    p for p in [str(CODE), env.get("PYTHONPATH", "")]
    if p
)

# Test architecture tensor shapes before full training.
import importlib.util

for name, script in script_paths.items():
    spec = importlib.util.spec_from_file_location(
        f"benchmark_{name}", str(script)
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    model = module.UNet(width=24).eval()

    sample = {
        "sar_pre": torch.rand(1, 1, 64, 64),
        "sar_post": torch.rand(1, 1, 64, 64),
        "ms_pre": torch.rand(1, 3, 64, 64),
        "ms_post": torch.rand(1, 3, 64, 64)
    }

    with torch.inference_mode():
        output = model(sample)

    assert output.shape == (1, 1, 64, 64)

    count = sum(p.numel() for p in model.parameters())

    print(f"{name}: shape PASS; parameters={count:,}")

    del model, module

# ------------------------------------------------------------
# 5. TRAIN FOUR SEEDS PER ARCHITECTURE
# ------------------------------------------------------------

SEEDS = [7, 17, 37, 42]

def load_checkpoint(path):
    return torch.load(
        path, map_location="cpu", weights_only=True
    )

for name, script in script_paths.items():

    print("\n" + "=" * 72)
    print("BENCHMARK:", name)
    print("=" * 72)

    for seed in SEEDS:
        run = OUT / name / f"seed{seed}"
        run.mkdir(parents=True, exist_ok=True)

        last_path = run / "last.pt"
        best_path = run / "best.pt"

        if last_path.is_file():
            ck = load_checkpoint(last_path)
            cfg = ck.get("config", {})

            assert cfg.get("model") == name
            assert cfg.get("seed") == seed
            assert cfg.get("batch_size") == 4
            assert cfg.get("shift_prob") == 0.25
            assert cfg.get("max_shift") == 2

            if ck["epoch"] >= 50:
                assert best_path.is_file()
                print(f"{name} seed {seed}: COMPLETE, skipping")
                continue

            print(
                f"{name} seed {seed}: "
                f"resuming from epoch {ck['epoch'] + 1}"
            )
        else:
            print(f"{name} seed {seed}: starting at epoch 1")

        cmd = [
            sys.executable, "-u", str(script),
            "--train", str(DATA / "train"),
            "--val", str(DATA / "val"),
            "--out", str(run),
            "--seed", str(seed),
            "--epochs", "50"
        ]

        log_path = run / "training_console.log"

        with log_path.open("a", encoding="utf-8") as log:
            process = subprocess.Popen(
                cmd,
                cwd=str(CODE),
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1
            )

            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()

            exit_code = process.wait()

        if exit_code != 0:
            raise RuntimeError(
                f"{name} seed {seed} failed. See {log_path}"
            )

        best = load_checkpoint(best_path)

        print(
            f"COMPLETED {name} / seed {seed}: "
            f"best epoch={best['epoch']}, "
            f"validation IoU="
            f"{best['metrics']['flood_iou']:.5f}"
        )

# ------------------------------------------------------------
# 6. WRITE BENCHMARK SUMMARY
# ------------------------------------------------------------

rows = []

for name in ["standard_unet", *ARCHITECTURES]:

    for seed in SEEDS:
        path = OUT / name / f"seed{seed}/best.pt"
        assert path.is_file(), f"Missing checkpoint: {path}"

        ck = load_checkpoint(path)

        rows.append({
            "architecture": name,
            "seed": seed,
            "best_epoch": ck["epoch"],
            "validation_iou": ck["metrics"]["flood_iou"],
            "validation_f1": ck["metrics"]["flood_f1"]
        })

summary = pd.DataFrame(rows)

summary_path = OUT / "three_baseline_validation_summary.csv"
summary.to_csv(summary_path, index=False)

print("\n" + "=" * 72)
print("STEP 92 COMPLETED")
print("=" * 72)

print(summary.to_string(index=False))
print("\nFour-seed mean validation IoU:")
print(
    summary.groupby("architecture")["validation_iou"]
    .mean().round(5).to_string()
)

print("\nSaved:", summary_path)
print("Original UA-FloodNet checkpoints unchanged.")


In [ ]:

# ============================================================
# STEP 93 — COMMON-PROTOCOL U-NET BENCHMARK EVALUATION
# 3 baselines x 4 seeds x 2 partitions x 13 conditions
# 312 expected evaluation records
# ============================================================

from pathlib import Path
import sys, os, zipfile, importlib.util
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
CODE = ROOT / "v4/code"
BASE = ROOT / "v4/final_analysis/benchmark_study"
ABLATION = ROOT / "v4/final_analysis/ablation_study"
RESULTS = BASE / "evaluation_93"
RESULTS.mkdir(parents=True, exist_ok=True)

DATA = Path("/content/UA_FloodNet/dataset")
ARCHIVE = ROOT / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"

sys.path.insert(0, str(CODE))
from data import NPZFloodDataset

ARCHITECTURES = [
    "standard_unet",
    "siamese_unet",
    "two_stage_fusion_unet"
]
SEEDS = [7, 17, 37, 42]
SHIFTS = [
    (0,0), (-1,0), (1,0), (-2,0), (2,0),
    (-4,0), (4,0), (-8,0), (8,0),
    (0,-2), (0,2), (0,-4), (0,4)
]
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
CROP = 20

# 1. Restore missing dataset files
expected = {"train":499, "val":125, "test":70}
with zipfile.ZipFile(ARCHIVE) as z:
    for split, count in expected.items():
        members = [
            m for m in z.infolist()
            if not m.is_dir()
            and len(Path(m.filename).parts) == 2
            and Path(m.filename).parts[0] == split
            and m.filename.endswith(".npz")
        ]
        assert len(members) == count
        folder = DATA / split
        folder.mkdir(parents=True, exist_ok=True)
        for m in members:
            target = DATA / m.filename
            assert target.resolve().is_relative_to(DATA.resolve())
            if not target.exists() or target.stat().st_size != m.file_size:
                z.extract(m, DATA)
        assert len(list(folder.glob("*.npz"))) == count
        print(split, count, "patches ready")

# 2. Import original architecture classes, not new substitutes
model_classes = {}
for name in ARCHITECTURES:
    script = BASE / f"train_{name if name != 'standard_unet' else 'unet_benchmark'}.py"
    assert script.is_file(), script
    spec = importlib.util.spec_from_file_location(
        "benchmark_eval_" + name, str(script)
    )
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    model_classes[name] = mod.UNet

# 3. Zero-padded optical translation
def translate_zero(x, dx, dy):
    if dx == 0 and dy == 0:
        return x
    h, w = x.shape[-2:]
    result = torch.zeros_like(x)
    if abs(dx) >= w or abs(dy) >= h:
        return result
    sx0, sx1 = max(0,-dx), min(w,w-dx)
    sy0, sy1 = max(0,-dy), min(h,h-dy)
    result[...,sy0+dy:sy1+dy,sx0+dx:sx1+dx] = (
        x[...,sy0:sy1,sx0:sx1]
    )
    return result

def load_model(name, seed):
    path = BASE / name / f"seed{seed}" / "best.pt"
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    assert ckpt["seed"] == seed
    assert ckpt["architecture"] == name
    assert ckpt["config"]["model"] == name
    assert ckpt["config"]["shift_prob"] == 0.25
    assert ckpt["config"]["max_shift"] == 2

    model = model_classes[name](width=ckpt["width"])
    model.load_state_dict(ckpt["model"], strict=True)
    return model.to(DEVICE).eval()

@torch.inference_mode()
def evaluate(model, loader, dx, dy):
    tp = fp = fn = 0
    for batch in loader:
        b = {k:v.to(DEVICE) for k,v in batch.items()}
        b["ms_pre"] = translate_zero(b["ms_pre"],dx,dy)
        b["ms_post"] = translate_zero(b["ms_post"],dx,dy)
        logits = model(b)
        pred = (logits.sigmoid() >= 0.5)[
            ...,CROP:-CROP,CROP:-CROP
        ]
        truth = (b["mask"] >= 0.5)[
            ...,CROP:-CROP,CROP:-CROP
        ]
        tp += int((pred & truth).sum().item())
        fp += int((pred & ~truth).sum().item())
        fn += int((~pred & truth).sum().item())
    union = tp+fp+fn
    denom = 2*tp+fp+fn
    return {
        "iou": tp/union if union else 1.0,
        "f1": 2*tp/denom if denom else 1.0,
        "tp":tp, "fp":fp, "fn":fn
    }

# 4. Resume from existing CSV
DETAIL = RESULTS / "baseline_displacement_results.csv"
FIELDS = [
    "partition","architecture","seed","dx","dy",
    "iou","f1","tp","fp","fn"
]
if DETAIL.exists():
    df = pd.read_csv(DETAIL)
    assert set(FIELDS).issubset(df.columns)
    assert not df.duplicated(
        ["partition","architecture","seed","dx","dy"]
    ).any()
    df = df[FIELDS]
else:
    df = pd.DataFrame(columns=FIELDS)

loaders = {
    split: DataLoader(
        NPZFloodDataset(str(DATA/split)),
        batch_size=4, shuffle=False, num_workers=0
    )
    for split in ("val","test")
}

print("Existing:",len(df),"/312")

for split,loader in loaders.items():
    for name in ARCHITECTURES:
        for seed in SEEDS:
            done = set(zip(
                df.partition,df.architecture,df.seed,
                df.dx,df.dy
            ))
            missing = [
                (dx,dy) for dx,dy in SHIFTS
                if (split,name,seed,dx,dy) not in done
            ]
            if not missing:
                print("SKIP",split,name,seed)
                continue
            print("EVALUATE",split,name,seed,flush=True)
            model = load_model(name,seed)
            for dx,dy in missing:
                metrics = evaluate(model,loader,dx,dy)
                record = {
                    "partition":split,"architecture":name,
                    "seed":seed,"dx":dx,"dy":dy,**metrics
                }
                df = pd.concat(
                    [df,pd.DataFrame([record])],
                    ignore_index=True
                )
                tmp = DETAIL.with_suffix(".tmp")
                df.to_csv(tmp,index=False)
                os.replace(tmp,DETAIL)
                print(
                    f"  ({dx:+d},{dy:+d}) "
                    f"IoU={metrics['iou']:.5f} "
                    f"[{len(df)}/312]",
                    flush=True
                )
            del model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

# 5. Final checks and four-seed summaries
df = pd.read_csv(DETAIL)
assert len(df) == 312
assert not df.duplicated(
    ["partition","architecture","seed","dx","dy"]
).any()

rows = []
for (split,name,seed),g in df.groupby(
    ["partition","architecture","seed"]
):
    assert len(g) == 13
    clean = float(g[(g.dx==0)&(g.dy==0)].iou.iloc[0])
    shifted = g[~((g.dx==0)&(g.dy==0))].iou
    rows.append({
        "partition":split,"architecture":name,
        "seed":int(seed),
        "clean_iou":clean,
        "mean_shifted_iou":float(shifted.mean()),
        "iou_degradation":float(clean-shifted.mean()),
        "worst_shifted_iou":float(shifted.min())
    })

summary = pd.DataFrame(rows)
SUMMARY = RESULTS / "baseline_four_seed_summary.csv"
summary.to_csv(SUMMARY,index=False)

print("\nSTEP 93 COMPLETED")
print(summary.groupby(["partition","architecture"])[[
    "clean_iou","mean_shifted_iou",
    "iou_degradation","worst_shifted_iou"
]].mean().round(5).to_string())
print("\nSaved:",DETAIL)
print("Saved:",SUMMARY)


In [ ]:

# ================================================================
# STEP 93B — BENCHMARK STATISTICS AND FIGURES
# Four matched seeds | Official OMBRIA test partition
# No model training
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
FINAL = ROOT / "v4/final_analysis"

ABL = FINAL / "ablation_study/evaluation_88B"
BEN = FINAL / "benchmark_study/evaluation_93"

OUT = BEN / "step93B_statistics"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. LOAD VERIFIED EVALUATIONS
# ------------------------------------------------------------

full = pd.read_csv(
    ABL / "four_seed_architecture_summary.csv"
)
baseline = pd.read_csv(
    BEN / "baseline_four_seed_summary.csv"
)

full = full[full["variant"] == "full"].copy()
full["architecture"] = "UA-FloodNet"

baseline = baseline.copy()

name_map = {
    "standard_unet": "Standard U-Net",
    "siamese_unet": "Siamese U-Net",
    "two_stage_fusion_unet": "Two-stage fusion"
}

baseline["architecture"] = (
    baseline["architecture"].map(name_map)
)

columns = [
    "partition", "architecture", "seed",
    "clean_iou", "mean_shifted_iou",
    "iou_degradation", "worst_shifted_iou"
]

all_data = pd.concat(
    [full[columns], baseline[columns]],
    ignore_index=True
)

SEEDS = [7, 17, 37, 42]
BASELINES = list(name_map.values())
METRICS = [
    "clean_iou", "mean_shifted_iou",
    "iou_degradation", "worst_shifted_iou"
]

assert len(all_data) == 32
assert not all_data.duplicated(
    ["partition", "architecture", "seed"]
).any()

for partition in ["val", "test"]:
    for model in ["UA-FloodNet", *BASELINES]:
        matched = all_data[
            (all_data.partition == partition)
            & (all_data.architecture == model)
        ]
        assert sorted(matched.seed.tolist()) == SEEDS
        assert matched[METRICS].notna().all().all()

# ------------------------------------------------------------
# 2. OFFICIAL TEST DESCRIPTIVE STATISTICS
# ------------------------------------------------------------

test = all_data[
    all_data.partition == "test"
].copy()

descriptive = (
    test.groupby("architecture")[METRICS]
    .agg(["mean", "std"])
)

descriptive.to_csv(
    OUT / "Table_B1_Benchmark_Descriptive.csv"
)

print("\nOFFICIAL TEST — FOUR-SEED DESCRIPTIVES")
print(descriptive.round(5).to_string())

# ------------------------------------------------------------
# 3. PAIRED STATISTICAL COMPARISONS
# Difference = baseline minus UA-FloodNet
# ------------------------------------------------------------

reference = (
    test[test.architecture == "UA-FloodNet"]
    .set_index("seed").loc[SEEDS]
)

rows = []

for model in BASELINES:
    comparison = (
        test[test.architecture == model]
        .set_index("seed").loc[SEEDS]
    )

    for metric in METRICS:
        delta = (
            comparison[metric].to_numpy(float)
            - reference[metric].to_numpy(float)
        )

        n = len(delta)
        estimate = delta.mean()
        sd = delta.std(ddof=1)
        se = sd / np.sqrt(n)

        if sd == 0:
            p = 1.0 if estimate == 0 else 0.0
            low = high = estimate
        else:
            t_result = stats.ttest_1samp(delta, 0)
            p = float(t_result.pvalue)

            margin = stats.t.ppf(
                0.975, n - 1
            ) * se

            low = estimate - margin
            high = estimate + margin

        rows.append({
            "comparison": model + " - UA-FloodNet",
            "architecture": model,
            "metric": metric,
            "n_seeds": n,
            "mean_difference": estimate,
            "ci95_lower": low,
            "ci95_upper": high,
            "p_unadjusted": p
        })

paired = pd.DataFrame(rows)

# One adjustment family: 3 models x 4 test metrics.
paired["p_holm"] = multipletests(
    paired["p_unadjusted"],
    method="holm"
)[1]

paired["holm_significant"] = paired["p_holm"] < 0.05

paired.to_csv(
    OUT / "Table_B2_Benchmark_Paired_Statistics.csv",
    index=False
)

print("\nPAIRED TEST COMPARISONS")
print(
    paired[
        ["comparison", "metric", "mean_difference",
         "ci95_lower", "ci95_upper",
         "p_unadjusted", "p_holm"]
    ].round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 4. PAIRED-EFFECT FOREST PLOT
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

metric_titles = {
    "clean_iou": "Clean IoU",
    "mean_shifted_iou": "Mean shifted IoU",
    "iou_degradation": "IoU degradation",
    "worst_shifted_iou": "Worst-shift IoU"
}

fig, axes = plt.subplots(
    2, 2,
    figsize=(10, 6.5),
    constrained_layout=True
)

for ax, metric in zip(axes.flat, METRICS):
    subset = (
        paired[paired.metric == metric]
        .set_index("architecture")
        .loc[BASELINES]
    )

    y = np.arange(len(BASELINES))

    estimate = subset.mean_difference.to_numpy()
    low = subset.ci95_lower.to_numpy()
    high = subset.ci95_upper.to_numpy()

    for i in range(len(y)):
        ax.plot(
            [low[i], high[i]], [y[i], y[i]],
            color="0.25", linewidth=1.4
        )
        ax.plot(
            [low[i], low[i]], [y[i]-.065, y[i]+.065],
            color="0.25", linewidth=1
        )
        ax.plot(
            [high[i], high[i]], [y[i]-.065, y[i]+.065],
            color="0.25", linewidth=1
        )
        ax.scatter(
            estimate[i], y[i],
            color="black", marker="D",
            s=27, zorder=4
        )

    ax.axvline(
        0, color="0.5", linestyle="--",
        linewidth=0.9
    )

    ax.set_yticks(y, BASELINES)
    ax.invert_yaxis()
    ax.set_title(
        metric_titles[metric], loc="left",
        fontsize=10
    )
    ax.set_xlabel("Baseline minus UA-FloodNet")
    ax.grid(axis="x", alpha=0.12)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)

for ext in ["png", "pdf", "svg"]:
    options = {"dpi": 600} if ext == "png" else {}
    fig.savefig(
        OUT / f"Figure_B1_Paired_Benchmark_Effects.{ext}",
        bbox_inches="tight",
        **options
    )

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 5. HORIZONTAL AND VERTICAL DISPLACEMENT CURVES
# ------------------------------------------------------------

ua_detail = pd.read_csv(
    ABL / "all_displacement_results.csv"
)
ua_detail = ua_detail[
    (ua_detail.partition == "test")
    & (ua_detail.variant == "full")
].copy()
ua_detail["architecture"] = "UA-FloodNet"

base_detail = pd.read_csv(
    BEN / "baseline_displacement_results.csv"
)
base_detail = base_detail[
    base_detail.partition == "test"
].copy()
base_detail["architecture"] = (
    base_detail["architecture"].map(name_map)
)

curves = pd.concat(
    [
        ua_detail[
            ["architecture", "seed", "dx", "dy", "iou"]
        ],
        base_detail[
            ["architecture", "seed", "dx", "dy", "iou"]
        ]
    ],
    ignore_index=True
)

assert len(curves) == 208

fig, axes = plt.subplots(
    1, 2,
    figsize=(11, 4.3),
    constrained_layout=True
)

models = ["UA-FloodNet", *BASELINES]

for ax, shift_col, fixed_col, title in [
    (axes[0], "dx", "dy", "(a) Horizontal displacement"),
    (axes[1], "dy", "dx", "(b) Vertical displacement")
]:
    for model in models:
        part = curves[
            (curves.architecture == model)
            & (curves[fixed_col] == 0)
        ]

        pivot = part.pivot(
            index=shift_col, columns="seed",
            values="iou"
        ).sort_index()

        assert list(pivot.columns) == SEEDS

        x = pivot.index.to_numpy()
        values = pivot.to_numpy()

        mean = values.mean(axis=1)
        sd = values.std(axis=1, ddof=1)

        line, = ax.plot(
            x, mean, marker="o",
            markersize=3.5,
            linewidth=1.5, label=model
        )

        ax.fill_between(
            x, mean - sd, mean + sd,
            color=line.get_color(),
            alpha=0.07, linewidth=0
        )

    ax.axvline(0, color="0.4", linestyle=":", linewidth=1)
    ax.set_xlabel("Optical displacement (pixels)")
    ax.set_ylabel("Mean flood IoU")
    ax.set_title(title, loc="left")
    ax.grid(alpha=0.12)
    ax.spines[["top", "right"]].set_visible(False)

axes[1].legend(
    frameon=False,
    loc="lower center",
    fontsize=8
)

for ext in ["png", "pdf", "svg"]:
    options = {"dpi": 600} if ext == "png" else {}
    fig.savefig(
        OUT / f"Figure_B2_Benchmark_Displacement.{ext}",
        bbox_inches="tight",
        **options
    )

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 6. FINISH
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("STEP 93B COMPLETED")
print("=" * 68)

print("Paired comparisons:", len(paired))
print(
    "Holm-significant comparisons:",
    int(paired.holm_significant.sum())
)
print("Output directory:", OUT)

for path in sorted(OUT.iterdir()):
    print(" ", path.name)


In [ ]:

# ============================================================
# STEP 94A — FLOOD EVENT PROVENANCE AND DATASET AUDIT
# Read-only inspection; no training or data modification
# ============================================================

from pathlib import Path
from collections import Counter
import zipfile
import numpy as np
import json
import csv
import re
import hashlib

from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
ARCHIVE = ROOT / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"
OUT = ROOT / "v4/final_analysis/generalization_study"

OUT.mkdir(parents=True, exist_ok=True)

assert ARCHIVE.is_file(), "OMBRIA dataset archive missing."

print("=" * 72)
print("STEP 94A — GENERALIZATION FEASIBILITY AUDIT")
print("=" * 72)

report = {
    "archive": str(ARCHIVE),
    "splits": {},
    "sample_structures": {},
    "candidate_metadata_files": [],
    "event_provenance_verified": False
}

# ------------------------------------------------------------
# 1. INSPECT ARCHIVE CONTENTS AND PATCH IDENTIFIERS
# ------------------------------------------------------------

with zipfile.ZipFile(ARCHIVE) as z:
    names = [
        n for n in z.namelist()
        if n.lower().endswith(".npz")
    ]

    print("\n1. DATASET SPLITS")

    for split in ["train", "val", "test"]:
        selected = sorted(
            n for n in names
            if Path(n).parts[0] == split
        )

        ids = [Path(n).stem for n in selected]

        print(f"\n{split.upper()}: {len(selected)} patches")
        print("Examples:", ids[:12])

        report["splits"][split] = {
            "count": len(selected),
            "example_ids": ids[:12]
        }

    # --------------------------------------------------------
    # 2. INSPECT NUMERIC ARRAYS AND METADATA KEYS
    # --------------------------------------------------------

    print("\n2. NPZ FILE CONTENTS")

    for split in ["train", "val", "test"]:
        selected = sorted(
            n for n in names
            if Path(n).parts[0] == split
        )

        structures = []

        # Inspect three examples from each split.
        for filename in selected[:3]:
            with z.open(filename) as f:
                with np.load(f, allow_pickle=False) as npz:
                    fields = {}

                    for key in npz.files:
                        a = npz[key]
                        fields[key] = {
                            "shape": list(a.shape),
                            "dtype": str(a.dtype)
                        }

                    structures.append({
                        "file": filename,
                        "fields": fields
                    })

                    print("\n", filename)
                    for key, description in fields.items():
                        print(
                            " ",
                            key,
                            description["shape"],
                            description["dtype"]
                        )

        report["sample_structures"][split] = structures

# ------------------------------------------------------------
# 3. IDENTIFY POTENTIAL PROVENANCE DOCUMENTS
# ------------------------------------------------------------

print("\n3. PROJECT METADATA INVENTORY")

keywords = [
    "event",
    "metadata",
    "manifest",
    "catalog",
    "source",
    "provenance",
    "split",
    "scene",
    "location",
    "ombria"
]

extensions = {
    ".csv", ".json", ".geojson",
    ".txt", ".xlsx", ".parquet"
}

candidate_files = []

for path in ROOT.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in extensions:
        continue

    if any(k in path.name.lower() for k in keywords):
        candidate_files.append(path)

candidate_files = sorted(candidate_files)

for path in candidate_files[:80]:
    relative = str(path.relative_to(ROOT))
    print(relative)
    report["candidate_metadata_files"].append(relative)

print("\nMetadata candidates:", len(candidate_files))

# ------------------------------------------------------------
# 4. CHECK FOR EXPLICIT EVENT IDENTIFIERS
# ------------------------------------------------------------

print("\n4. EVENT-INDEPENDENCE ASSESSMENT")

print(
    "NPZ patch counts alone cannot establish "
    "flood-event or geographic independence."
)

print(
    "Inspect the candidate metadata files for "
    "original event IDs, scene IDs, spatial footprints, "
    "acquisition dates, and source-image identifiers."
)

print(
    "Event provenance is NOT marked as verified "
    "without an explicit patch-to-event mapping."
)

# ------------------------------------------------------------
# 5. SAVE REPORT
# ------------------------------------------------------------

destination = OUT / "step94A_dataset_provenance_audit.json"

destination.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 72)
print("STEP 94A COMPLETED")
print("=" * 72)
print("Report:", destination)
print("No checkpoints or datasets were modified.")


In [ ]:

# ================================================================
# STEP 93C — COLORED PAIRED-EFFECT ESTIMATION PLOT
# Target: ISPRS Journal of Photogrammetry and Remote Sensing
# 4 matched seeds | 95% paired Student-t confidence intervals
# Color-coded architectures | 600 dpi PNG + PDF + SVG
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import t
from google.colab import drive

# ------------------------------------------------------------
# 1. LOAD THE ORIGINAL VERIFIED RESULTS
# ------------------------------------------------------------

MOUNT = Path("/content/ua_ablation_drive")

if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
FINAL = ROOT / "v4/final_analysis"

ABL = FINAL / "ablation_study/evaluation_88B"
BEN = FINAL / "benchmark_study/evaluation_93"
STAT = BEN / "step93B_statistics"

OUT = STAT / "advanced_figures"
OUT.mkdir(parents=True, exist_ok=True)

ua = pd.read_csv(
    ABL / "four_seed_architecture_summary.csv"
)

baselines = pd.read_csv(
    BEN / "baseline_four_seed_summary.csv"
)

tests = pd.read_csv(
    STAT / "Table_B2_Benchmark_Paired_Statistics.csv"
)

ua = (
    ua[
        (ua["partition"] == "test") &
        (ua["variant"] == "full")
    ]
    .set_index("seed")
)

baselines = baselines[
    baselines["partition"] == "test"
].copy()

# ------------------------------------------------------------
# 2. MODEL COLORS AND VISUAL DESIGN
# ------------------------------------------------------------

MODELS = [
    ("standard_unet", "Standard U-Net", "#1769AA"),
    ("siamese_unet", "Siamese U-Net", "#E47722"),
    ("two_stage_fusion_unet", "Two-stage fusion", "#159765")
]

METRICS = [
    ("clean_iou", "(a) Clean IoU"),
    ("mean_shifted_iou", "(b) Mean shifted IoU"),
    ("iou_degradation", "(c) IoU degradation"),
    ("worst_shifted_iou", "(d) Worst-shift IoU")
]

SEEDS = [7, 17, 37, 42]

SEED_MARKERS = ["o", "s", "^", "D"]
SEED_OFFSETS = [-0.18, -0.06, 0.06, 0.18]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "axes.linewidth": 0.85,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

fig, axes = plt.subplots(
    2, 2,
    figsize=(12.4, 8.0),
    constrained_layout=False
)

# ------------------------------------------------------------
# 3. CREATE THE FOUR ESTIMATION PANELS
# ------------------------------------------------------------

for ax, (metric, panel_title) in zip(axes.flat, METRICS):

    all_values = [0.0]

    for i, (key, model_name, color) in enumerate(MODELS):

        comparison = (
            baselines[
                baselines["architecture"] == key
            ]
            .set_index("seed")
            .loc[SEEDS]
        )

        reference = ua.loc[SEEDS]

        differences = (
            comparison[metric].to_numpy(dtype=float)
            - reference[metric].to_numpy(dtype=float)
        )

        estimate = float(differences.mean())

        se = float(
            differences.std(ddof=1) / np.sqrt(len(SEEDS))
        )

        ci = float(
            t.ppf(0.975, len(SEEDS) - 1) * se
        )

        # Draw individual seed observations in model color.
        for j, difference in enumerate(differences):
            ax.scatter(
                i + SEED_OFFSETS[j],
                difference,
                marker=SEED_MARKERS[j],
                s=47,
                facecolor=color,
                edgecolor="white",
                linewidth=0.85,
                alpha=0.88,
                zorder=4
            )

        # Draw confidence interval and mean point.
        ax.errorbar(
            i + 0.29,
            estimate,
            yerr=ci,
            fmt="o",
            color=color,
            ecolor=color,
            markerfacecolor=color,
            markeredgecolor="white",
            markeredgewidth=1.0,
            markersize=8.5,
            capsize=5,
            capthick=1.7,
            elinewidth=2.1,
            zorder=6
        )

        # Use saved Holm-adjusted significance values.
        record = tests[
            (tests["architecture"] == model_name) &
            (tests["metric"] == metric)
        ]

        assert len(record) == 1, (
            f"Missing statistical record: {model_name}, {metric}"
        )

        saved_difference = float(
            record.iloc[0]["mean_difference"]
        )

        assert np.isclose(
            estimate, saved_difference, atol=1e-9
        ), "Effect mismatch with saved statistics"

        p_holm = float(record.iloc[0]["p_holm"])

        # Add asterisk only for Holm-significant results.
        if p_holm < 0.05:
            all_values.append(estimate + ci + 0.006)
            ax.annotate(
                "*",
                xy=(i + 0.29, estimate + ci),
                xytext=(0, 7),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=16,
                fontweight="bold",
                color=color,
                clip_on=False
            )

        all_values.extend(differences.tolist())
        all_values.extend([
            estimate - ci,
            estimate + ci
        ])

    # Zero represents identical performance.
    ax.axhline(
        0,
        color="#64748B",
        linestyle=(0, (4, 3)),
        linewidth=1.15,
        zorder=1
    )

    ax.set_xlim(-0.45, 2.62)

    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels([
        "Standard\nU-Net",
        "Siamese\nU-Net",
        "Two-stage\nfusion"
    ])

    ax.set_title(
        panel_title,
        loc="left",
        fontsize=11,
        fontweight="bold",
        pad=11
    )

    ax.set_ylabel(
        "Paired difference\n(baseline − UA-FloodNet)"
    )

    ax.grid(
        axis="y",
        color="#DDE4EB",
        linestyle="-",
        linewidth=0.65,
        alpha=0.7
    )

    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Automatically preserve data and confidence interval limits.
    ymin = min(all_values)
    ymax = max(all_values)

    span = ymax - ymin
    margin = max(span * 0.16, 0.0015)

    ax.set_ylim(
        ymin - margin,
        ymax + margin
    )

# ------------------------------------------------------------
# 4. PROFESSIONAL LEGEND
# ------------------------------------------------------------

legend_items = [
    Line2D(
        [0], [0],
        color=color,
        marker="o",
        markersize=7,
        linewidth=2,
        label=name
    )
    for _, name, color in MODELS
]

legend_items += [
    Line2D(
        [0], [0],
        marker=marker,
        color="#64748B",
        linestyle="None",
        markersize=7,
        label=f"Seed {seed}"
    )
    for seed, marker in zip(SEEDS, SEED_MARKERS)
]

fig.legend(
    handles=legend_items,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.995),
    ncol=4,
    frameon=False,
    fontsize=9,
    columnspacing=1.7,
    handletextpad=0.5
)

fig.subplots_adjust(
    left=0.09,
    right=0.985,
    top=0.88,
    bottom=0.09,
    wspace=0.27,
    hspace=0.34
)

# ------------------------------------------------------------
# 5. SAVE PUBLICATION-QUALITY FIGURES
# ------------------------------------------------------------

NAME = "Figure_B1_Advanced_Paired_Estimation"

for ext in ["png", "pdf", "svg"]:

    kwargs = {"dpi": 600} if ext == "png" else {}

    destination = OUT / f"{NAME}.{ext}"

    fig.savefig(
        destination,
        bbox_inches="tight",
        pad_inches=0.12,
        facecolor="white",
        **kwargs
    )

    print("Saved:", destination.name)

plt.show()
plt.close(fig)

print("\n" + "=" * 72)
print("STEP 93C — COLORED ESTIMATION FIGURE COMPLETED")
print("=" * 72)
print("Colors: blue, orange, green")
print("Individual observations: four matched seeds")
print("Error bars: 95% paired Student-t confidence intervals")
print("Asterisks: Holm-adjusted p < 0.05")
print("Output:", OUT)
print("Original statistical results unchanged.")


In [ ]:

# ============================================================
# STEP 94B — OMBRIA EVENT-PROVENANCE INVESTIGATION
# Read-only metadata inspection
# No training, changes to splits, or checkpoint modification
# ============================================================

from pathlib import Path
import pandas as pd
import json
import re

from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")

if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"

FILES = {
    "candidate_split": (
        ROOT / "dataset_backup/candidate_split_seed42.csv"
    ),
    "conversion_metadata": (
        ROOT / "dataset_backup/conversion_metadata.json"
    ),
    "dataset_provenance": (
        ROOT / "v4/final_analysis/"
        "manuscript_package_step81/tables/"
        "Table_04_Dataset_Provenance.csv"
    ),
    "reproducibility_manifest": (
        ROOT / "v4/final_analysis/"
        "manuscript_package_step81/"
        "reproducibility_manifest.json"
    ),
    "source_notebook_audit": (
        ROOT / "v4/audit/"
        "step73d_source_notebook_audit.json"
    )
}

KEYWORDS = [
    "event", "flood", "location", "region",
    "scene", "source", "date", "time",
    "latitude", "longitude", "coordinate",
    "tile", "image", "country",
    "sentinel", "acquisition", "patch",
    "geographic", "original", "id"
]

print("=" * 76)
print("STEP 94B — OMBRIA PROVENANCE INVESTIGATION")
print("=" * 76)

def inspect_csv(path, label):
    df = pd.read_csv(path)

    print(f"\n{label.upper()}")
    print("-" * 65)
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("\nFirst 8 records:")
    print(df.head(8).to_string(index=False, max_colwidth=75))

    relevant = [
        col for col in df.columns
        if any(word in col.lower() for word in KEYWORDS)
    ]

    print("\nPotential provenance columns:", relevant)

    for col in relevant:
        print(
            f"{col}: "
            f"{df[col].nunique(dropna=True)} "
            "unique non-null values"
        )
        print(
            "Examples:",
            df[col].dropna().astype(str).unique()[:8]
        )

    return df

def inspect_json(path, label):
    data = json.loads(path.read_text(encoding="utf-8"))

    print(f"\n{label.upper()}")
    print("-" * 65)

    if isinstance(data, dict):
        print("Top-level keys:", list(data.keys()))

        for key, value in data.items():
            print(f"\n{key}:")

            if isinstance(value, dict):
                print(
                    json.dumps(
                        dict(list(value.items())[:8]),
                        indent=2,
                        default=str
                    )[:1800]
                )

            elif isinstance(value, list):
                print("List length:", len(value))
                print(
                    json.dumps(
                        value[:2],
                        indent=2,
                        default=str
                    )[:1600]
                )

            else:
                print(str(value)[:1000])

    else:
        print(
            json.dumps(data, indent=2, default=str)[:4000]
        )

    return data

results = {}

for name, path in FILES.items():

    if not path.is_file():
        print("\nMISSING:", name, path)
        continue

    print("\nFILE:", path.relative_to(ROOT))

    try:
        if path.suffix.lower() == ".csv":
            results[name] = inspect_csv(path, name)
        else:
            results[name] = inspect_json(path, name)

    except Exception as exc:
        print("Inspection failed:", repr(exc))

# ------------------------------------------------------------
# CHECK WHETHER SPLIT HAS EXPLICIT EVENT IDENTIFIERS
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("EVENT-IDENTIFIER ASSESSMENT")
print("=" * 76)

candidate = results.get("candidate_split")

if isinstance(candidate, pd.DataFrame):

    event_columns = [
        col for col in candidate.columns
        if any(
            keyword in col.lower()
            for keyword in [
                "event", "flood_id",
                "scene", "source_image",
                "location", "region",
                "acquisition"
            ]
        )
    ]

    if event_columns:
        print("Potential event-provenance columns:")
        print(event_columns)
        print(
            "\nThese must be checked against the "
            "original data documentation before use."
        )

    else:
        print(
            "No explicit event or scene fields "
            "identified in candidate split columns."
        )

print("\n" + "=" * 76)
print("STEP 94B INSPECTION FINISHED")
print("=" * 76)
print("No data, checkpoints, or existing splits were changed.")


In [ ]:

# ============================================================
# STEP 94C — RECOVER ORIGINAL OMBRIA PROVENANCE
# Read-only: inspect archived source paths and metadata
# ============================================================

from pathlib import Path
import zipfile
import json
import pandas as pd
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")

if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"

print("=" * 72)
print("STEP 94C — ORIGINAL OMBRIA SOURCE INVESTIGATION")
print("=" * 72)

# ------------------------------------------------------------
# 1. DISCOVER ORIGINAL DATA SOURCES
# ------------------------------------------------------------

terms = [
    "ombria", "sentinel", "source", "original",
    "dataset", "tfrecord", "metadata",
    "manifest", "geo", "event"
]

candidates = []

for path in ROOT.rglob("*"):
    if not path.is_file():
        continue

    name = path.name.lower()

    if any(term in name for term in terms):
        candidates.append(path)

print("\nPOTENTIAL ORIGINAL SOURCES")

for path in candidates[:100]:
    print(path.relative_to(ROOT))

print("\nTotal matching files:", len(candidates))

# ------------------------------------------------------------
# 2. INSPECT ORIGINAL ZIP ARCHIVES
# ------------------------------------------------------------

archives = sorted(ROOT.rglob("*.zip"))

print("\nARCHIVE INVENTORY")

for archive in archives:
    print("\nArchive:", archive.relative_to(ROOT))

    try:
        with zipfile.ZipFile(archive) as z:
            members = z.namelist()

            print("Entries:", len(members))
            print("First 30 paths:")

            for name in members[:30]:
                print(" ", name)

            event_like = [
                name for name in members
                if any(term in name.lower() for term in [
                    "event", "location", "ems",
                    "country", "manifest", "metadata"
                ])
            ]

            if event_like:
                print("\nPotential event metadata:")
                for name in event_like[:30]:
                    print(" ", name)

    except Exception as exc:
        print("Archive inspection failed:", exc)

# ------------------------------------------------------------
# 3. ANALYZE CANDIDATE SPLIT RELATIONSHIPS
# ------------------------------------------------------------

split_path = (
    ROOT / "dataset_backup/candidate_split_seed42.csv"
)

split = pd.read_csv(split_path)

print("\nORIGINAL SPLIT RELATIONSHIPS")

print(
    pd.crosstab(
        split["source_split"],
        split["split"]
    ).to_string()
)

print("\nIdentifier examples by partition")

for partition, group in split.groupby("split"):
    print(
        partition,
        sorted(group["identifier"].tolist())[:15]
    )

# ------------------------------------------------------------
# 4. REPORT PROVENANCE LIMITATION
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("PROVENANCE ASSESSMENT")
print("=" * 72)

print("Original source files identified:", len(candidates))
print("ZIP archives inspected:", len(archives))

print(
    "\nIMPORTANT: Original patch identifiers are not "
    "automatically geographic or event identifiers."
)

print(
    "We need an explicit source-image, scene, or "
    "event mapping before creating event-held-out folds."
)

print("\nSTEP 94C INSPECTION COMPLETED")
print("No files or checkpoints modified.")


In [ ]:

# ============================================================
# STEP 94D — ORIGINAL OMBRIA ARCHIVE INVESTIGATION
# Download source archive; inspect paths without extraction
# No training or modification of existing data
# ============================================================

from pathlib import Path
from collections import Counter
import requests
import zipfile
import json
import re

from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
OUT = ROOT / "v4/final_analysis/generalization_study"
OUT.mkdir(parents=True, exist_ok=True)

LOCAL = Path("/content/ombria_source")
LOCAL.mkdir(parents=True, exist_ok=True)

ARCHIVE = LOCAL / "OMBRIA-master.zip"

# ------------------------------------------------------------
# 1. GET AUTHORITATIVE ZENODO DOWNLOAD LINK
# ------------------------------------------------------------

API = "https://zenodo.org/api/records/6777870"

response = requests.get(API, timeout=60)
response.raise_for_status()

record = response.json()

files = record.get("files", [])

candidates = [
    item for item in files
    if item.get("key", "").lower().endswith(".zip")
]

assert candidates, "No ZIP archive found in Zenodo record."

source_file = candidates[0]
url = (
    source_file.get("links", {}).get("self")
    or source_file.get("links", {}).get("download")
)

assert url, "Zenodo file download link not found."

print("=" * 72)
print("STEP 94D — ORIGINAL OMBRIA SOURCE AUDIT")
print("=" * 72)

print("Source:", source_file["key"])
print("Expected size (MB):",
      round(source_file.get("size", 0) / 1048576, 2))

# ------------------------------------------------------------
# 2. DOWNLOAD ORIGINAL ARCHIVE IF NEEDED
# ------------------------------------------------------------

expected_size = source_file.get("size", 0)

if not ARCHIVE.is_file() or (
    expected_size and ARCHIVE.stat().st_size != expected_size
):
    print("\nDownloading original source archive.")

    temporary = ARCHIVE.with_suffix(".partial")

    with requests.get(url, stream=True, timeout=90) as r:
        r.raise_for_status()

        with temporary.open("wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)

    if expected_size:
        assert temporary.stat().st_size == expected_size, (
            "Downloaded archive size mismatch."
        )

    temporary.replace(ARCHIVE)

print("Archive available:", ARCHIVE)

# ------------------------------------------------------------
# 3. INSPECT THE FOLDER STRUCTURE
# ------------------------------------------------------------

with zipfile.ZipFile(ARCHIVE) as z:

    assert z.testzip() is None, "Archive integrity failed."

    names = z.namelist()

    print("\nARCHIVE CONTENTS")
    print("Total entries:", len(names))

    print("\nFirst 70 paths:")

    for name in names[:70]:
        print(" ", name)

    # Identify the first three directory levels.
    folders = Counter()

    for name in names:
        parts = Path(name).parts

        if len(parts) >= 3:
            folders["/".join(parts[:3])] += 1

    print("\nSOURCE DIRECTORY GROUPS")

    for folder, count in folders.most_common(40):
        print(f"{count:5}  {folder}")

    # Search filenames for location or scene clues.
    search_terms = [
        "event", "region", "emsr", "location",
        "metadata", "manifest", "scene",
        "coordinates", "country", "tfrecord"
    ]

    matches = [
        name for name in names
        if any(
            term in name.lower()
            for term in search_terms
        )
    ]

    print("\nPOTENTIAL PROVENANCE FILES")

    for name in matches[:80]:
        print(" ", name)

    # Save path-only inventory, not extracted imagery.
    report = {
        "source_record": 6777870,
        "archive": source_file["key"],
        "entry_count": len(names),
        "directory_groups": dict(folders),
        "provenance_candidates": matches,
        "event_independence_verified": False
    }

REPORT = OUT / "step94D_original_archive_inventory.json"

REPORT.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 72)
print("STEP 94D COMPLETED")
print("=" * 72)
print("Saved inventory:", REPORT)
print("No original files extracted.")
print("No training performed.")


In [ ]:

# ============================================================
# STEP 94E — ORIGINAL OMBRIA SOURCE CORRESPONDENCE
# Inspect archive structure, pixel matches, PNG metadata
# No training or changes to existing datasets
# ============================================================

from pathlib import Path
from collections import Counter
from io import BytesIO
import zipfile
import numpy as np
from PIL import Image
import json

ROOT = Path(
    "/content/ua_ablation_drive/MyDrive/UA_FloodNet"
)

SOURCE = Path(
    "/content/ombria_source/OMBRIA-master.zip"
)

CONVERTED = (
    ROOT / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"
)

OUT = (
    ROOT / "v4/final_analysis/generalization_study"
)
OUT.mkdir(parents=True, exist_ok=True)

assert SOURCE.is_file(), "Original archive missing"
assert CONVERTED.is_file(), "Converted archive missing"

print("=" * 72)
print("STEP 94E — ORIGINAL OMBRIA CORRESPONDENCE")
print("=" * 72)

# ------------------------------------------------------------
# 1. SUMMARIZE ORIGINAL FILE TYPES
# ------------------------------------------------------------

with zipfile.ZipFile(SOURCE) as z:

    entries = [
        n for n in z.namelist()
        if not n.endswith("/")
    ]

    print("\nORIGINAL FILE TYPES")

    types = Counter(
        Path(n).suffix.lower() for n in entries
    )

    for extension, count in sorted(types.items()):
        print(extension or "(none)", count)

    non_images = [
        n for n in entries
        if Path(n).suffix.lower()
        not in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
    ]

    print("\nNON-IMAGE FILES")
    for name in non_images[:40]:
        print(name)

    # --------------------------------------------------------
    # 2. INSPECT IDENTIFIERS AND IMAGE GROUPS
    # --------------------------------------------------------

    print("\nIMAGE GROUPS")

    groups = Counter()

    for name in entries:
        p = Path(name).parts

        if len(p) >= 5:
            groups["/".join(p[1:4])] += 1

    for key, count in sorted(groups.items()):
        print(f"{key}: {count}")

    # --------------------------------------------------------
    # 3. INSPECT ORIGINAL PNG METADATA
    # --------------------------------------------------------

    print("\nPNG METADATA INSPECTION")

    candidates = [
        "OMBRIA-master/OmbriaS1/test/AFTER/S1_after_0013.png",
        "OMBRIA-master/OmbriaS2/test/AFTER/S2_after_0013.png"
    ]

    for name in candidates:

        if name not in z.namelist():
            print("Not found:", name)
            continue

        with z.open(name) as f:
            image = Image.open(BytesIO(f.read()))
            image.load()

            print("\n", name)
            print("Size:", image.size)
            print("Mode:", image.mode)
            print("Embedded metadata keys:", list(image.info))

            for key, value in image.info.items():
                print(key, ":", str(value)[:300])

# ------------------------------------------------------------
# 4. VERIFY ORIGINAL SOURCE-TO-NPZ PIXELS
# ------------------------------------------------------------

print("\nSOURCE-TO-NPZ PIXEL COMPARISON")

sample_ids = [
    ("test", "0013"),
    ("test", "0408"),
    ("train", "0001"),
    ("val", "0002")
]

comparisons = []

with zipfile.ZipFile(SOURCE) as original, \
     zipfile.ZipFile(CONVERTED) as converted:

    original_names = set(original.namelist())

    for split, identifier in sample_ids:

        source_split = (
            "test" if split == "test" else "train"
        )

        npz_name = f"{split}/{identifier}.npz"

        # These original filenames are checked rather than assumed.
        sar_name = (
            f"OMBRIA-master/OmbriaS1/"
            f"{source_split}/AFTER/"
            f"S1_after_{identifier}.png"
        )

        opt_name = (
            f"OMBRIA-master/OmbriaS2/"
            f"{source_split}/AFTER/"
            f"S2_after_{identifier}.png"
        )

        print(f"\n{split}/{identifier}")

        with converted.open(npz_name) as f:
            with np.load(f, allow_pickle=False) as arrays:
                sar_npz = arrays["sar_post"].copy()
                opt_npz = arrays["ms_post"].copy()

        for modality, name, target in [
            ("SAR", sar_name, sar_npz),
            ("Optical", opt_name, opt_npz)
        ]:

            if name not in original_names:
                print(modality, "source image not found")
                continue

            with original.open(name) as f:
                img = Image.open(BytesIO(f.read()))
                pixels = np.asarray(img).copy()

            # Convert source image to channel-first structure.
            if pixels.ndim == 2:
                pixels = pixels[None, :, :]
            elif pixels.ndim == 3:
                pixels = pixels.transpose(2, 0, 1)

            print(
                modality,
                "Original shape:", pixels.shape,
                "NPZ shape:", target.shape
            )

            if pixels.shape != target.shape:
                print("Shape mismatch; comparison skipped")
                continue

            normalized = pixels.astype(np.float32) / 255.0

            difference = np.abs(normalized - target)

            maximum_error = float(difference.max())
            mean_error = float(difference.mean())

            print("Maximum difference:", maximum_error)
            print("Mean difference:", mean_error)

            matches = bool(
                np.allclose(
                    normalized,
                    target,
                    rtol=0,
                    atol=1e-6
                )
            )

            print("Exact normalized match:", matches)

            comparisons.append({
                "split": split,
                "identifier": identifier,
                "modality": modality,
                "source_file": name,
                "max_absolute_difference": maximum_error,
                "mean_absolute_difference": mean_error,
                "normalized_pixel_match": matches
            })

# ------------------------------------------------------------
# 5. SAVE AUDIT REPORT
# ------------------------------------------------------------

REPORT = OUT / "step94E_source_correspondence.json"

REPORT.write_text(
    json.dumps({
        "comparisons": comparisons,
        "note": (
            "Matching pixels establish source correspondence "
            "for inspected samples only. They do not establish "
            "geographic or event independence."
        )
    }, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 72)
print("STEP 94E COMPLETED")
print("=" * 72)
print("Report:", REPORT)
print("No datasets or checkpoints modified.")


In [ ]:

# ============================================================
# STEP 95A — GEOID-FLOOD METADATA AND COMPATIBILITY AUDIT
# Metadata only | No imagery downloads | No training
# ============================================================

import sys, subprocess, json, re
from pathlib import Path
from collections import Counter
import pandas as pd

# 1. Install metadata dependencies
subprocess.run(
    [
        sys.executable, "-m", "pip", "-q", "install",
        "huggingface_hub", "pyarrow"
    ],
    check=True
)

from huggingface_hub import (
    HfApi, hf_hub_download
)
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
OUT = ROOT / "v4/final_analysis/generalization_study"
OUT.mkdir(parents=True, exist_ok=True)

REPO = "links-ads/geoid-flood"

print("=" * 75)
print("STEP 95A — GEOID-FLOOD EXTERNAL DATA AUDIT")
print("=" * 75)

# ------------------------------------------------------------
# 2. FETCH REPOSITORY INVENTORY — NO RASTERS
# ------------------------------------------------------------

api = HfApi()
repo_files = api.list_repo_files(
    repo_id=REPO,
    repo_type="dataset"
)

print("\nREPOSITORY INVENTORY")
print("Total files:", len(repo_files))

extensions = Counter(
    Path(f).suffix.lower()
    for f in repo_files
)

print("\nFile extensions:")
for suffix, count in extensions.most_common():
    print(suffix or "(none)", count)

metadata_candidates = [
    f for f in repo_files
    if Path(f).name in {
        "data_tiles_s256_st128.csv",
        "data_tiles_s256_st128_sample.csv",
        "tile_catalog.parquet"
    }
]

print("\nMETADATA FILES:")
for f in metadata_candidates:
    print(" ", f)

# ------------------------------------------------------------
# 3. DOWNLOAD ONLY METADATA FILES
# ------------------------------------------------------------

MAX_METADATA_BYTES = 150 * 1024 * 1024

repo_info = api.repo_info(
    repo_id=REPO,
    repo_type="dataset",
    files_metadata=True
)

size_lookup = {
    s.rfilename: s.size
    for s in (repo_info.siblings or [])
}

downloaded = {}
download_errors = {}

for name in metadata_candidates:

    size = size_lookup.get(name)

    if size is not None and size > MAX_METADATA_BYTES:
        print(
            "Skipping oversized metadata:",
            name, size
        )
        continue

    print("\nFetching metadata:", name)

    try:
        path = hf_hub_download(
            repo_id=REPO,
            repo_type="dataset",
            filename=name
        )
        downloaded[name] = Path(path)
        print("Saved in HF cache:", path)

    except Exception as exc:
        download_errors[name] = str(exc)
        print("Could not fetch:", str(exc)[:400])

# ------------------------------------------------------------
# 4. INSPECT METADATA AND EVENT INFORMATION
# ------------------------------------------------------------

EVENT_PATTERN = re.compile(
    r"EMSR\d+(?:-\d+)?",
    flags=re.IGNORECASE
)

tables = []
table_reports = []

for name, path in downloaded.items():

    print("\n" + "-" * 75)
    print("METADATA:", name)

    try:
        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path)
        elif path.suffix.lower() == ".parquet":
            df = pd.read_parquet(path)
        else:
            continue

    except Exception as exc:
        print("Read error:", exc)
        continue

    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("\nFirst three records:")
    print(df.head(3).to_string(index=False, max_colwidth=40))

    # Find split and geographic/event candidate fields
    important = [
        c for c in df.columns
        if any(
            word in str(c).lower()
            for word in [
                "event", "ems", "aoi", "activation",
                "country", "split", "location",
                "path", "tile", "geometry", "valid"
            ]
        )
    ]

    print("\nPotential provenance columns:", important)

    if "split" in df.columns:
        print("\nSplit counts:")
        print(
            df["split"].value_counts(
                dropna=False
            ).to_string()
        )

    # Extract explicit event identifiers from available
    # text fields. Keep inferred identifiers separate.
    text_columns = [
        c for c in df.columns
        if (
            pd.api.types.is_string_dtype(df[c])
            or df[c].dtype == object
        )
        and "geometry" not in str(c).lower()
    ]

    candidate_event_col = [
        c for c in text_columns
        if any(
            key in str(c).lower()
            for key in ["event", "aoi", "activation"]
        )
    ]

    if candidate_event_col:
        source_col = candidate_event_col[0]
        event_series = df[source_col].astype(str)
    else:
        source_col = None
        event_series = pd.Series(
            [""] * len(df),
            index=df.index
        )

        # Look only for explicit EMSR patterns.
        for c in text_columns[:12]:
            extracted = df[c].astype(str).str.extract(
                r"(EMSR\d+(?:-\d+)?)",
                flags=re.IGNORECASE,
                expand=False
            )
            event_series = event_series.mask(
                (event_series == "") & extracted.notna(),
                extracted
            )

    event_series = event_series.fillna("").astype(str)
    event_series = event_series.replace(
        {"nan": "", "None": ""}
    )

    print("\nEvent field used:", source_col or "EMSR pattern")
    print(
        "Rows with event-like identifiers:",
        int((event_series != "").sum())
    )
    print(
        "Distinct event-like identifiers:",
        event_series[event_series != ""].nunique()
    )

    preview_ids = (
        event_series[event_series != ""]
        .drop_duplicates()
        .head(12)
        .tolist()
    )
    print("Examples:", preview_ids)

    # Check whether detected event IDs span splits.
    overlap = []

    if "split" in df.columns:
        tmp = pd.DataFrame({
            "event": event_series,
            "split": df["split"].astype(str)
        })
        tmp = tmp[tmp.event != ""]

        if not tmp.empty:
            by_event = tmp.groupby("event")["split"].nunique()
            overlap = (
                by_event[by_event > 1].index.tolist()
            )

        print(
            "Event IDs appearing in multiple splits:",
            len(overlap)
        )

        if overlap:
            print("First overlaps:", overlap[:12])

    table_reports.append({
        "metadata_file": name,
        "rows": len(df),
        "columns": [str(c) for c in df.columns],
        "event_field": source_col,
        "event_pattern_count": int(
            (event_series != "").sum()
        ),
        "event_id_count": int(
            event_series[event_series != ""].nunique()
        ),
        "event_cross_split_count": len(overlap),
        "event_cross_split_examples": overlap[:20]
    })

    tables.append((name, df))

# ------------------------------------------------------------
# 5. CHECK RASTER SHARD STRUCTURE WITHOUT DOWNLOADING
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("EXTERNAL IMAGERY AVAILABILITY")
print("=" * 75)

layers = [
    "s1grd", "s1rtc", "s2l2a",
    "label", "dem", "validity"
]

layer_counts = {}

for layer in layers:
    paths = [
        f for f in repo_files
        if f"/{layer}/" in f.lower()
    ]
    layer_counts[layer] = len(paths)
    print(layer, "shard files:", len(paths))

print("\nSample raster paths:")
for f in [
    f for f in repo_files
    if f.endswith(".tar")
][:12]:
    print(" ", f)

# ------------------------------------------------------------
# 6. SAVE COMPATIBILITY REPORT
# ------------------------------------------------------------

compatibility = {
    "source": REPO,
    "dataset_type": "external_event_aware_flood",
    "expected_model_inputs": {
        "sar_pre": [1, 256, 256],
        "sar_post": [1, 256, 256],
        "ms_pre": [3, 256, 256],
        "ms_post": [3, 256, 256]
    },
    "external_modalities": {
        "s1grd": "2-band pre/post SAR; linear sigma0",
        "s2l2a": "12-band pre-event optical only",
        "label": "0 background, 1 permanent water, "
                 "2 flooded water, 255 ignore"
    },
    "compatibility_findings": {
        "SAR_channels": "adaptation_required",
        "SAR_normalization": "adaptation_required",
        "optical_RGB": "band_selection_required",
        "post_event_optical": "unavailable",
        "label_mapping": "required",
        "geographic_independence": (
            "metadata_assessment_pending"
        ),
        "zero_shot_input_equivalence": False
    },
    "metadata_tables": table_reports,
    "repository_file_count": len(repo_files),
    "raster_shard_counts": layer_counts,
    "metadata_download_errors": download_errors
}

REPORT = OUT / "step95A_geoid_compatibility_audit.json"

REPORT.write_text(
    json.dumps(compatibility, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 75)
print("STEP 95A COMPLETED")
print("=" * 75)
print("Saved:", REPORT)
print("No imagery shards downloaded.")
print("No training or checkpoint modification performed.")

print("\nNEXT DECISION:")
print(
    "Select an independent event subset, verify complete "
    "SAR/optical/label availability, then design and validate "
    "an explicit missing-modality adaptation."
)


In [ ]:

from pathlib import Path
import pandas as pd

from huggingface_hub import hf_hub_download

repo = "links-ads/geoid-flood"

main = pd.read_parquet(
    hf_hub_download(
        repo_id=repo,
        filename="geoid-flood/tile_catalog.parquet",
        repo_type="dataset",
        local_files_only=True
    )
)

heldout = pd.read_parquet(
    hf_hub_download(
        repo_id=repo,
        filename="geoid-flood-heldout/tile_catalog.parquet",
        repo_type="dataset",
        local_files_only=True
    )
)

main_events = set(main.event_id.dropna())
heldout_events = set(heldout.event_id.dropna())

overlap = main_events & heldout_events

print("Main catalog events:", len(main_events))
print("Held-out catalog events:", len(heldout_events))
print("Overlapping events:", len(overlap))
print("Overlapping identifiers:", sorted(overlap))

print("\nHELD-OUT EVENT SUMMARY")
print(
    heldout.groupby("event_id")
    .agg(
        catalog_records=("event_id", "size"),
        valid_records=("is_valid", "sum"),
        median_invalid_fraction=(
            "invalid_pixel_frac", "median"
        )
    )
    .round(3)
    .to_string()
)

print("\nDECISION:")
if not overlap:
    print("PASS: No event-ID overlap between catalogs.")
else:
    print("CAUTION: Overlapping events require exclusion.")


In [ ]:

# ============================================================
# STEP 95B — EXTERNAL EVENT INDEPENDENCE
# Fast, metadata-only verification
# No raster download or training
# ============================================================

from pathlib import Path
import pandas as pd
import json
from huggingface_hub import hf_hub_download
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
OUT = ROOT / "v4/final_analysis/generalization_study"
OUT.mkdir(parents=True, exist_ok=True)

REPO = "links-ads/geoid-flood"

def cached_parquet(filename):
    return pd.read_parquet(
        hf_hub_download(
            repo_id=REPO,
            repo_type="dataset",
            filename=filename,
            local_files_only=True
        )
    )

main = cached_parquet(
    "geoid-flood/tile_catalog.parquet"
)
held = cached_parquet(
    "geoid-flood-heldout/tile_catalog.parquet"
)

main_ids = set(main.event_id.dropna().astype(str))
held_ids = set(held.event_id.dropna().astype(str))

overlap = main_ids & held_ids
independent = held_ids - main_ids

print("=" * 70)
print("STEP 95B — EXTERNAL EVENT INDEPENDENCE")
print("=" * 70)

print("Main catalog events:", len(main_ids))
print("Held-out catalog events:", len(held_ids))
print("Overlapping event IDs:", len(overlap))
print("Overlap:", sorted(overlap))
print("Eligible held-out events:", len(independent))

eligible = held[
    held.event_id.astype(str).isin(independent)
].copy()

if "is_valid" in eligible.columns:
    eligible = eligible[
        eligible.is_valid.fillna(False).astype(bool)
    ]

if "invalid_pixel_frac" in eligible.columns:
    eligible = eligible[
        eligible.invalid_pixel_frac.fillna(1) <= 0.95
    ]

summary = (
    eligible.groupby("event_id")
    .agg(
        eligible_tiles=("event_id", "size"),
        median_invalid_fraction=(
            "invalid_pixel_frac", "median"
        )
    )
    .reset_index()
    .sort_values(
        ["eligible_tiles", "event_id"],
        ascending=[False, True]
    )
)

assert not set(summary.event_id) & main_ids

CSV = OUT / "step95B_eligible_heldout_events.csv"
summary.to_csv(CSV, index=False)

report = {
    "main_event_count": len(main_ids),
    "heldout_event_count": len(held_ids),
    "overlapping_events": sorted(overlap),
    "eligible_event_count": len(independent),
    "eligible_tile_records": len(eligible),
    "note": (
        "Event-ID independence verified against "
        "the main GEOID-Flood catalog only. "
        "Input and label compatibility remain "
        "to be evaluated."
    )
}

JSON = OUT / "step95B_event_independence.json"
JSON.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\nELIGIBLE EVENT SUMMARY")
print(summary.round(3).to_string(index=False))

print("\nSAVED")
print(CSV)
print(JSON)

print("\nSTATUS")
if len(independent) > 0 and len(eligible) > 0:
    print("PASS — eligible independent event records found")
else:
    print("BLOCKED — no eligible independent event records")


In [ ]:

# ================================================================
# STEP 95C — FINAL GEOID-FLOOD DATA REPAIR & VALIDATION
# UA-FloodNet | External Generalization
#
# 1. Locate downloaded external sample
# 2. Repair Hugging Face symbolic links
# 3. Resolve missing cached files if necessary
# 4. Verify all raster files
# 5. Inspect satellite data and labels
# 6. Save detailed report to Google Drive
#
# NO MODEL TRAINING
# NO CHECKPOINT MODIFICATION
# NO DELETION OF EXISTING DATA
# ================================================================

import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. ENVIRONMENT
# ------------------------------------------------------------

def install_if_missing(module, package=None):
    try:
        __import__(module)
    except ImportError:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "-q",
                "install",
                package or module
            ],
            check=True
        )

install_if_missing("rasterio")
install_if_missing("huggingface_hub")

import rasterio
from huggingface_hub import hf_hub_download
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")

if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"

OUT = (
    ROOT
    / "v4"
    / "final_analysis"
    / "generalization_study"
)

OUT.mkdir(parents=True, exist_ok=True)

LOCAL_DATA = Path(
    "/content/geoid_flood_sample_local"
)

DATA_TREE = LOCAL_DATA / "geoid-flood"

CACHE_ROOT = Path(
    "/root/.cache/huggingface/hub/"
    "datasets--links-ads--geoid-flood"
)

BLOB_DIR = CACHE_ROOT / "blobs"

REPO = "links-ads/geoid-flood"

LAYERS = [
    "s1grd",
    "s1rtc",
    "s2l2a",
    "dem",
    "label",
    "cloudmask",
    "floodmask",
    "permwater",
    "validity"
]

EXPECTED_COUNTS = {
    "s1grd": 94,
    "s1rtc": 94,
    "s2l2a": 47,
    "dem": 47,
    "label": 47,
    "cloudmask": 47,
    "floodmask": 47,
    "permwater": 47,
    "validity": 47
}

print("=" * 78)
print("STEP 95C — FINAL EXTERNAL DATA REPAIR & VALIDATION")
print("=" * 78)

print("Python:", sys.version.split()[0])
print("Dataset:", LOCAL_DATA)
print("Google Drive output:", OUT)
print("Cache exists:", CACHE_ROOT.exists())

assert DATA_TREE.is_dir(), (
    "GEOID-Flood sample was not found. "
    "The previous sample download must be available."
)

# ------------------------------------------------------------
# 2. INVENTORY EXISTING DOWNLOADED FILES
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("PHASE 1 — FILE INVENTORY")
print("=" * 78)

files = sorted(
    p for p in LOCAL_DATA.rglob("*")
    if p.is_file() or p.is_symlink()
)

print("File entries:", len(files))

broken_links = [
    p for p in files
    if p.is_symlink() and not p.exists()
]

print("Broken symbolic links:", len(broken_links))

# ------------------------------------------------------------
# 3. REPAIR BROKEN CACHE SYMBOLIC LINKS
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("PHASE 2 — REPAIR SYMBOLIC LINKS")
print("=" * 78)

repaired = []
unresolved = []

for path in broken_links:

    raw_target = os.readlink(path)

    # The original cache link normally ends in a blob hash.
    blob_name = Path(raw_target).name

    candidate = BLOB_DIR / blob_name

    if candidate.is_file():

        # Preserve the real cached blob.
        # Modify only the broken link in the local sample.
        path.unlink()
        path.symlink_to(candidate)

        repaired.append(
            str(path.relative_to(LOCAL_DATA))
        )

    else:

        unresolved.append({
            "path": str(path.relative_to(LOCAL_DATA)),
            "blob_name": blob_name,
            "cache_target": str(candidate)
        })

print("Repaired using cached blobs:", len(repaired))
print("Unresolved links:", len(unresolved))

# ------------------------------------------------------------
# 4. RECOVER MISSING CACHED FILES
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("PHASE 3 — RECOVER UNRESOLVED FILES")
print("=" * 78)

recovered = []
recovery_errors = []

for item in unresolved:

    relative_path = item["path"]

    # The official sample is stored under sample/geoid-flood/.
    # The local target is under geoid-flood/.
    local_relative = Path(relative_path)

    if not local_relative.parts:
        recovery_errors.append({
            "path": relative_path,
            "error": "Invalid relative path"
        })
        continue

    repository_path = (
        Path("sample") / local_relative
    ).as_posix()

    target = LOCAL_DATA / local_relative

    print("Recovering:", repository_path)

    try:

        downloaded_path = hf_hub_download(
            repo_id=REPO,
            repo_type="dataset",
            filename=repository_path
        )

        source = Path(downloaded_path)

        assert source.is_file(), (
            "Downloaded source is not readable."
        )

        target.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        # Copy into local storage without moving the
        # Hugging Face cached source.
        temporary = target.with_name(
            target.name + ".repairing"
        )

        if temporary.exists() or temporary.is_symlink():
            temporary.unlink()

        shutil.copyfile(source, temporary)

        assert temporary.stat().st_size > 0

        # Replace only the unresolved local link.
        os.replace(temporary, target)

        recovered.append(relative_path)

    except Exception as exc:

        recovery_errors.append({
            "path": relative_path,
            "repository_path": repository_path,
            "error": repr(exc)
        })

print("Recovered from Hugging Face:", len(recovered))
print("Recovery errors:", len(recovery_errors))

# ------------------------------------------------------------
# 5. CHECK REMAINING BROKEN LINKS
# ------------------------------------------------------------

remaining_broken = [
    p for p in LOCAL_DATA.rglob("*")
    if p.is_symlink() and not p.exists()
]

print("\nRemaining broken links:", len(remaining_broken))

for path in remaining_broken[:10]:
    print("BROKEN:", path.relative_to(LOCAL_DATA))

# ------------------------------------------------------------
# 6. VERIFY ALL GEOTIFF FILES
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("PHASE 4 — COMPLETE RASTER VALIDATION")
print("=" * 78)

tifs = sorted(
    DATA_TREE.rglob("*.tif")
)

print("GeoTIFF paths found:", len(tifs))

validation_rows = []
raster_failures = []

for layer in LAYERS:

    layer_files = [
        path for path in tifs
        if layer in [
            part.lower()
            for part in path.relative_to(DATA_TREE).parts
        ]
    ]

    print(
        f"\n{layer.upper()}: "
        f"{len(layer_files)} / "
        f"{EXPECTED_COUNTS[layer]} expected"
    )

    for path in layer_files:

        relative = str(
            path.relative_to(DATA_TREE)
        )

        record = {
            "layer": layer,
            "file": relative,
            "exists": path.exists(),
            "readable": False,
            "bands": None,
            "width": None,
            "height": None,
            "dtype": None,
            "crs": None,
            "nodata": None,
            "pixel_min": None,
            "pixel_median": None,
            "pixel_max": None,
            "error": ""
        }

        try:

            if not path.exists():
                raise FileNotFoundError(
                    "Broken or missing raster path"
                )

            with rasterio.open(path) as src:

                record["bands"] = src.count
                record["width"] = src.width
                record["height"] = src.height
                record["dtype"] = str(src.dtypes)
                record["crs"] = str(src.crs)
                record["nodata"] = str(src.nodata)

                # Read a small window to verify the actual
                # image data, not merely the TIFF header.
                h = min(64, src.height)
                w = min(64, src.width)

                pixels = src.read(
                    1,
                    window=rasterio.windows.Window(
                        0, 0, w, h
                    )
                )

                finite = pixels[
                    np.isfinite(pixels)
                ]

                if finite.size:

                    record["pixel_min"] = float(
                        np.min(finite)
                    )

                    record["pixel_median"] = float(
                        np.median(finite)
                    )

                    record["pixel_max"] = float(
                        np.max(finite)
                    )

                record["readable"] = True

        except Exception as exc:

            record["error"] = repr(exc)

            raster_failures.append({
                "file": relative,
                "error": repr(exc)
            })

        validation_rows.append(record)

# ------------------------------------------------------------
# 7. SUMMARIZE ACTUAL RASTER STATUS
# ------------------------------------------------------------

validation_df = pd.DataFrame(
    validation_rows
)

summary_rows = []

for layer in LAYERS:

    subset = validation_df[
        validation_df["layer"] == layer
    ]

    actual = len(subset)

    readable = int(
        subset["readable"].sum()
    )

    expected = EXPECTED_COUNTS[layer]

    summary_rows.append({
        "layer": layer,
        "expected_files": expected,
        "found_files": actual,
        "readable_files": readable,
        "failed_files": actual - readable,
        "complete": (
            actual == expected
            and readable == expected
        )
    })

summary = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 78)
print("RASTER VALIDATION SUMMARY")
print("=" * 78)

print(summary.to_string(index=False))

print("\nRaster read failures:", len(raster_failures))

for failure in raster_failures[:10]:
    print(
        failure["file"],
        failure["error"]
    )

# ------------------------------------------------------------
# 8. INSPECT REPRESENTATIVE BANDS AND VALUES
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("PHASE 5 — SATELLITE AND LABEL CHARACTERISTICS")
print("=" * 78)

sample_report = {}

for layer in LAYERS:

    subset = validation_df[
        (validation_df["layer"] == layer)
        & (validation_df["readable"])
    ]

    if subset.empty:
        continue

    relative = subset.iloc[0]["file"]

    path = DATA_TREE / relative

    with rasterio.open(path) as src:

        print(f"\nLAYER: {layer}")
        print("Example:", relative)
        print("Bands:", src.count)
        print(
            "Dimensions:",
            f"{src.width} x {src.height}"
        )
        print("CRS:", src.crs)
        print("Dtypes:", src.dtypes)
        print("NoData:", src.nodata)
        print("Band descriptions:", src.descriptions)

        h = min(256, src.height)
        w = min(256, src.width)

        values = src.read(
            window=rasterio.windows.Window(
                0, 0, w, h
            )
        )

        finite = values[
            np.isfinite(values)
        ]

        if finite.size:
            print(
                "Pixel min:",
                float(finite.min())
            )
            print(
                "Pixel median:",
                float(np.median(finite))
            )
            print(
                "Pixel max:",
                float(finite.max())
            )

        # Inspect the label classes actually present,
        # without assuming which values mean flooding.
        if layer in {
            "label",
            "validity",
            "floodmask",
            "permwater"
        }:

            unique, counts = np.unique(
                values,
                return_counts=True
            )

            classes = dict(zip(
                unique.tolist(),
                counts.tolist()
            ))

            print(
                "Observed pixel values:",
                dict(list(classes.items())[:30])
            )

        else:
            classes = None

        sample_report[layer] = {
            "example": relative,
            "bands": src.count,
            "width": src.width,
            "height": src.height,
            "crs": str(src.crs),
            "dtype": list(src.dtypes),
            "nodata": str(src.nodata),
            "band_descriptions": list(
                src.descriptions
            ),
            "observed_classes": classes
        }

# ------------------------------------------------------------
# 9. SAVE REPORTS TO GOOGLE DRIVE
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("PHASE 6 — SAVE REPRODUCIBILITY REPORTS")
print("=" * 78)

validation_csv = (
    OUT / "step95C_all_raster_validation.csv"
)

summary_csv = (
    OUT / "step95C_raster_inventory.csv"
)

report_json = (
    OUT / "step95C_final_validation_report.json"
)

validation_df.to_csv(
    validation_csv,
    index=False
)

summary.to_csv(
    summary_csv,
    index=False
)

complete = (
    not remaining_broken
    and not recovery_errors
    and not raster_failures
    and bool(summary["complete"].all())
)

report = {
    "study": "UA-FloodNet",
    "step": "95C",
    "source": "GEOID-Flood official sample",
    "repository": REPO,
    "local_directory": str(DATA_TREE),
    "total_rasters": len(tifs),
    "broken_links_before": len(broken_links),
    "repaired_links": len(repaired),
    "recovered_files": len(recovered),
    "remaining_broken_links": [
        str(p.relative_to(LOCAL_DATA))
        for p in remaining_broken
    ],
    "recovery_errors": recovery_errors,
    "raster_failures": raster_failures,
    "layer_summary": summary_rows,
    "sample_characteristics": sample_report,
    "validation_complete": bool(complete),
    "note": (
        "This step verifies external raster accessibility "
        "and structure. It does not demonstrate external "
        "generalization or model compatibility."
    )
}

report_json.write_text(
    json.dumps(
        report,
        indent=2,
        default=str
    ),
    encoding="utf-8"
)

print("Saved:", validation_csv)
print("Saved:", summary_csv)
print("Saved:", report_json)

# ------------------------------------------------------------
# 10. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("STEP 95C — FINAL STATUS")
print("=" * 78)

if complete:
    print("PASS — ALL EXTERNAL SAMPLE RASTERS VERIFIED")
    print("The dataset is accessible for input adaptation.")
else:
    print("PARTIAL — SOME FILES REQUIRE ATTENTION")
    print("Review the saved failure report.")

print("\nRaster count:", len(tifs))
print("Broken links remaining:", len(remaining_broken))
print("Raster read failures:", len(raster_failures))

print("\nExisting UA-FloodNet checkpoints: UNCHANGED")
print("Existing benchmark results: UNCHANGED")
print("Model training performed: NONE")
print("=" * 78)


In [ ]:

# ================================================================
# STEP 95D — GEOID-FLOOD EXTERNAL INPUT READINESS
# Inventory valid aligned 256x256 evaluation windows
# No retraining, no checkpoint modification
# ================================================================

from pathlib import Path
from collections import Counter, defaultdict
import json, re
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
DATA = Path("/content/geoid_flood_sample_local/geoid-flood")
OUT = ROOT / "v4/final_analysis/generalization_study"
OUT.mkdir(parents=True, exist_ok=True)

assert DATA.is_dir(), "Local sample missing"
assert len(list(DATA.rglob("*.tif"))) == 517

WINDOW_SIZE = 256
STRIDE = 256
MIN_VALID = 0.50
MIN_FLOODED_PIXELS = 25

print("=" * 76)
print("STEP 95D — EXTERNAL INPUT READINESS")
print("=" * 76)

# ------------------------------------------------------------
# 1. GROUP FILES BY AREA / SCENE
# ------------------------------------------------------------

# Expected layout:
# geoid-flood/EMSR712-10/s1grd/<scene>_s1grd_pre_...tif
# geoid-flood/EMSR712-10/s1grd/<scene>_s1grd_post_...tif
# geoid-flood/EMSR712-10/s2l2a/<scene>_s2l2a_pre_...tif
# geoid-flood/EMSR712-10/label/<scene>_label.tif

scene_pattern = re.compile(
    r"^(?P<scene>.+?)_"
    r"(?P<layer>s1grd|s1rtc|s2l2a|label|validity)"
    r"(?:_(?P<time>pre|post))?"
    r"(?:_.*)?\.tif$",
    re.IGNORECASE
)

groups = defaultdict(lambda: defaultdict(list))

for path in DATA.rglob("*.tif"):
    relative = path.relative_to(DATA)

    if len(relative.parts) < 3:
        continue

    area, layer = relative.parts[0], relative.parts[1]

    if layer not in {
        "s1grd", "s1rtc", "s2l2a", "label", "validity"
    }:
        continue

    match = scene_pattern.match(path.name)

    if not match:
        continue

    scene = match.group("scene")
    time = match.group("time")

    key = (
        layer + "_" + time
        if time is not None
        else layer
    )

    groups[(area, scene)][key].append(path)

print("Distinct scene groups:", len(groups))

# ------------------------------------------------------------
# 2. INSPECT COMPLETE SCENE GROUPS
# ------------------------------------------------------------

required = [
    "s1grd_pre",
    "s1grd_post",
    "s2l2a_pre",
    "label",
    "validity"
]

complete = []
incomplete = []

for (area, scene), files in sorted(groups.items()):

    missing = [
        key for key in required
        if len(files.get(key, [])) == 0
    ]

    if missing:
        incomplete.append({
            "area": area,
            "scene": scene,
            "missing": missing
        })
        continue

    # More than one SAR acquisition can exist per temporal
    # period. We must document rather than silently average.
    complete.append({
        "area": area,
        "scene": scene,
        "files": files
    })

print("Complete modality groups:", len(complete))
print("Incomplete groups:", len(incomplete))

if incomplete:
    print("First incomplete groups:", incomplete[:5])

if not complete:
    print("\nAvailable filename examples:")
    for p in sorted(DATA.rglob("*.tif"))[:25]:
        print(p.relative_to(DATA))
    raise RuntimeError(
        "No complete groups found. Inspect filename conventions."
    )

# ------------------------------------------------------------
# 3. BUILD VALID-LABEL WINDOW INVENTORY
# ------------------------------------------------------------

records = []
scene_reports = []

for item in complete:

    area = item["area"]
    scene = item["scene"]
    files = item["files"]

    # Choose the earliest sorted acquisition consistently.
    # This is a provisional audit choice, not the final
    # scientific temporal-selection protocol.
    selected = {
        key: sorted(files[key])[0]
        for key in required
    }

    with rasterio.open(selected["label"]) as label_ds, \
         rasterio.open(selected["validity"]) as valid_ds, \
         rasterio.open(selected["s1grd_pre"]) as sar_pre_ds, \
         rasterio.open(selected["s1grd_post"]) as sar_post_ds, \
         rasterio.open(selected["s2l2a_pre"]) as optical_ds:

        datasets = [
            valid_ds, sar_pre_ds,
            sar_post_ds, optical_ds
        ]

        aligned = all(
            ds.crs == label_ds.crs
            and ds.width == label_ds.width
            and ds.height == label_ds.height
            and ds.transform.almost_equals(label_ds.transform)
            for ds in datasets
        )

        scene_reports.append({
            "area": area,
            "scene": scene,
            "aligned": bool(aligned),
            "sar_pre": str(selected["s1grd_pre"]),
            "sar_post": str(selected["s1grd_post"]),
            "optical_pre": str(selected["s2l2a_pre"]),
            "label": str(selected["label"]),
            "validity": str(selected["validity"]),
            "sar_pre_bands": sar_pre_ds.count,
            "sar_post_bands": sar_post_ds.count,
            "optical_bands": optical_ds.count,
            "label_width": label_ds.width,
            "label_height": label_ds.height,
            "num_sar_pre_candidates": len(files["s1grd_pre"]),
            "num_sar_post_candidates": len(files["s1grd_post"])
        })

        if not aligned:
            print("NOT ALIGNED:", area, scene)
            continue

        if (sar_pre_ds.count < 1 or
            sar_post_ds.count < 1 or
            optical_ds.count < 3):
            print("INSUFFICIENT BANDS:", area, scene)
            continue

        H, W = label_ds.height, label_ds.width

        for y in range(0, H-WINDOW_SIZE+1, STRIDE):
            for x in range(0, W-WINDOW_SIZE+1, STRIDE):

                window = Window(
                    x, y, WINDOW_SIZE, WINDOW_SIZE
                )

                labels = label_ds.read(1, window=window)
                validity = valid_ds.read(1, window=window)

                # Validity must be confirmed from the actual
                # dataset rather than inferred from class 255.
                valid = (
                    (validity > 0)
                    & (labels != 255)
                )

                valid_pixels = int(valid.sum())
                valid_fraction = valid_pixels / labels.size

                unique, counts = np.unique(
                    labels[valid],
                    return_counts=True
                )

                class_counts = {
                    int(k): int(v)
                    for k, v in zip(unique, counts)
                }

                # Provisional class-2 indicator for prioritizing
                # windows; verify the label legend before scoring.
                flood_pixels = int(
                    np.logical_and(valid, labels == 2).sum()
                )

                records.append({
                    "area": area,
                    "scene": scene,
                    "x": x,
                    "y": y,
                    "size": WINDOW_SIZE,
                    "valid_fraction": valid_fraction,
                    "valid_pixels": valid_pixels,
                    "candidate_class2_pixels": flood_pixels,
                    "eligible_valid": valid_fraction >= MIN_VALID,
                    "eligible_class2": (
                        valid_fraction >= MIN_VALID
                        and flood_pixels >= MIN_FLOODED_PIXELS
                    ),
                    "class_counts": json.dumps(class_counts),
                    **{
                        key + "_path": str(path)
                        for key, path in selected.items()
                    }
                })

# ------------------------------------------------------------
# 4. SAVE RESULTS
# ------------------------------------------------------------

window_df = pd.DataFrame(records)
scene_df = pd.DataFrame(scene_reports)

assert len(window_df) > 0, (
    "No aligned evaluation windows found."
)

window_csv = OUT / "step95D_external_window_inventory.csv"
scene_csv = OUT / "step95D_scene_alignment_audit.csv"

window_df.to_csv(window_csv, index=False)
scene_df.to_csv(scene_csv, index=False)

print("\n" + "=" * 76)
print("STEP 95D — RESULTS")
print("=" * 76)

print("Scene groups:", len(scene_df))
print("Aligned groups:", int(scene_df.aligned.sum()))
print("Total windows:", len(window_df))

print(
    "Windows with >=50% valid labeled pixels:",
    int(window_df.eligible_valid.sum())
)

print(
    "Windows with >=50% validity and >=25 class-2 pixels:",
    int(window_df.eligible_class2.sum())
)

print("\nObserved valid-label class counts:")
counter = Counter()

for value in window_df.class_counts:
    for key, n in json.loads(value).items():
        counter[int(key)] += int(n)

print(dict(sorted(counter.items())))

print("\nTop candidate windows:")
print(
    window_df.sort_values(
        ["eligible_valid", "candidate_class2_pixels"],
        ascending=False
    )[
        [
            "area", "scene", "x", "y",
            "valid_fraction", "candidate_class2_pixels"
        ]
    ].head(15).to_string(index=False)
)

print("\nSaved:")
print(window_csv)
print(scene_csv)

if not window_df.eligible_valid.any():
    print(
        "\nCAUTION: No windows satisfy the validity threshold. "
        "Inspect the label and validity semantics before inference."
    )
else:
    print(
        "\nPASS: Candidate labeled windows identified. "
        "Input normalization and label semantics still "
        "require confirmation before model evaluation."
    )

print("\nNo checkpoints modified. No models retrained.")


In [ ]:

# ================================================================
# STEP 95E — INPUT COMPATIBILITY AND NORMALIZATION AUDIT
# Existing OMBRIA and GEOID data only
# No training; no invented band-order assumptions
# ================================================================

from pathlib import Path
import re, json, zipfile, hashlib
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
GEOID = Path("/content/geoid_flood_sample_local/geoid-flood")
SOURCE = Path("/content/geoid_flood_official")
OUT = ROOT / "v4/final_analysis/generalization_study"
OUT.mkdir(parents=True, exist_ok=True)

WINDOWS = pd.read_csv(
    OUT / "step95D_external_window_inventory.csv"
)

eligible = WINDOWS[
    WINDOWS.eligible_valid.astype(bool)
].copy()

assert len(eligible) == 456, (
    f"Expected 456 eligible windows, found {len(eligible)}"
)

print("=" * 72)
print("STEP 95E — EXTERNAL INPUT COMPATIBILITY")
print("=" * 72)

# 1. Examine code for Sentinel-2 band order.
print("\nSOURCE CODE: POSSIBLE BAND DEFINITIONS")
terms = re.compile(
    r"(?i)(B01|B02|B03|B04|B8A|band_names|"
    r"band_order|rgb_bands|s2_bands|bands\s*=)"
)

matches = []
for path in SOURCE.rglob("*.py"):
    try:
        lines = path.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()
    except Exception:
        continue
    for line_no, line in enumerate(lines, 1):
        if terms.search(line):
            matches.append({
                "file": str(path.relative_to(SOURCE)),
                "line": line_no,
                "text": line.strip()[:250]
            })

for record in matches[:80]:
    print(f"{record['file']}:{record['line']} "
          f"{record['text']}")

print("\nMatching lines:", len(matches))

# 2. Examine representative valid windows.
# One window per scene, to reduce redundant sampling.
selected = eligible.sort_values(
    ["area", "scene", "valid_fraction"],
    ascending=[True, True, False]
).drop_duplicates(["area", "scene"])

def summarize(array):
    arr = np.asarray(array, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if not arr.size:
        return {"n": 0}
    q = np.quantile(arr, [0, .01, .25, .5, .75, .99, 1])
    return dict(zip(
        ["min", "p01", "p25", "median",
         "p75", "p99", "max"],
        map(float, q)
    ))

records = []
class_counts = {0: 0, 1: 0, 2: 0, 255: 0}

for row in selected.itertuples(index=False):
    win = Window(int(row.x), int(row.y), 256, 256)

    with rasterio.open(row.s1grd_pre_path) as ds:
        vv_pre = ds.read(1, window=win)

    with rasterio.open(row.s1grd_post_path) as ds:
        vv_post = ds.read(1, window=win)

    with rasterio.open(row.s2l2a_pre_path) as ds:
        s2 = ds.read(window=win)

    with rasterio.open(row.label_path) as ds:
        label = ds.read(1, window=win)

    with rasterio.open(row.validity_path) as ds:
        validity = ds.read(1, window=win)

    valid = (validity == 1) & (label != 255)

    for cls in [0, 1, 2, 255]:
        class_counts[cls] += int(
            np.sum((label == cls) &
                   (valid if cls != 255 else ~valid))
        )

    # Two sensor scaling candidates, not yet model inputs.
    vv_pre_db = 10*np.log10(
        np.maximum(vv_pre.astype("float64"), 1e-8)
    )
    vv_post_db = 10*np.log10(
        np.maximum(vv_post.astype("float64"), 1e-8)
    )

    records.append({
        "area": row.area,
        "scene": row.scene,
        "vv_pre_linear": summarize(vv_pre),
        "vv_post_linear": summarize(vv_post),
        "vv_pre_db": summarize(vv_pre_db),
        "vv_post_db": summarize(vv_post_db),
        "s2_bands": [
            summarize(s2[i])
            for i in range(s2.shape[0])
        ],
        "label_classes": {
            str(cls): int(np.sum((label == cls) & valid))
            for cls in [0, 1, 2]
        }
    })

# 3. Compare with actual OMBRIA training distributions.
archive = (
    ROOT / "dataset_backup/OMBRIA_NPZ_candidate_seed42.zip"
)

ombria_records = []

with zipfile.ZipFile(archive) as z:
    sample_names = sorted(
        n for n in z.namelist()
        if n.startswith("train/") and n.endswith(".npz")
    )[:30]

    for name in sample_names:
        with z.open(name) as f:
            with np.load(f, allow_pickle=False) as d:
                for key in [
                    "sar_pre", "sar_post",
                    "ms_pre", "ms_post"
                ]:
                    ombria_records.append({
                        "file": name,
                        "field": key,
                        "stats": summarize(d[key])
                    })

report = {
    "step": "95E",
    "sample_event": "EMSR712",
    "eligible_windows": len(eligible),
    "scenes_sampled": len(selected),
    "source_band_definition_matches": matches,
    "external_scene_statistics": records,
    "ombria_reference_statistics": ombria_records,
    "observed_classes_in_sampled_windows": class_counts,
    "post_event_optical_available": False,
    "inference_ready": False,
    "reason": (
        "Verify Sentinel-2 band order, choose and document "
        "SAR scaling, and declare the missing-post-optical "
        "adaptation before checkpoint inference."
    )
}

destination = OUT / "step95E_input_compatibility.json"
destination.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 72)
print("STEP 95E RESULTS")
print("=" * 72)
print("Eligible windows:", len(eligible))
print("Scenes inspected:", len(selected))
print("Observed class counts:", class_counts)

if records:
    print("\nExample SAR VV linear:")
    print(records[0]["vv_pre_linear"])
    print("\nExample SAR VV dB:")
    print(records[0]["vv_pre_db"])
    print("\nSentinel-2 bands:", len(records[0]["s2_bands"]))

print("\nSaved:", destination)
print("No inference or training performed.")


In [ ]:

# ============================================================
# STEP 95F — GEOID-FLOOD ZERO-SHOT BENCHMARK INFERENCE
# Existing standard, Siamese and two-stage U-Net models
# Four seeds; 456 eligible windows
# Missing-post-optical stress-test protocol
# ============================================================

from pathlib import Path
import sys, os, importlib.util, json
import numpy as np
import pandas as pd
import torch
import rasterio
from rasterio.windows import Window
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
CODE = ROOT / "v4/code"
BASE = ROOT / "v4/final_analysis/benchmark_study"
OUT = ROOT / "v4/final_analysis/generalization_study"
OUT.mkdir(parents=True, exist_ok=True)

DATA = Path("/content/geoid_flood_sample_local/geoid-flood")
assert DATA.is_dir(), "External rasters missing"

WINDOWS = pd.read_csv(
    OUT / "step95D_external_window_inventory.csv"
)

# Some CSV readers return bool, others text.
def bool_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series
    return series.astype(str).str.lower().eq("true")

eligible = WINDOWS[
    bool_series(WINDOWS["eligible_valid"])
].copy()

assert len(eligible) == 456
assert set(eligible.area) == {"EMSR712-3", "EMSR712-10"}

ARCHS = {
    "standard_unet": "train_unet_benchmark.py",
    "siamese_unet": "train_siamese_unet.py",
    "two_stage_fusion_unet": "train_two_stage_fusion_unet.py"
}
SEEDS = [7, 17, 37, 42]
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
print("Eligible windows:", len(eligible))

# ------------------------------------------------------------
# 1. LOAD ACTUAL TRAINED ARCHITECTURE DEFINITIONS
# ------------------------------------------------------------

sys.path.insert(0, str(CODE))
models = {}

for architecture, script in ARCHS.items():
    path = BASE / script
    assert path.is_file(), path
    spec = importlib.util.spec_from_file_location(
        "external_" + architecture, str(path)
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    models[architecture] = module.UNet

# ------------------------------------------------------------
# 2. FIXED EXTERNAL INPUT ADAPTATION
# ------------------------------------------------------------

def read_external_window(row):
    window = Window(
        int(row.x), int(row.y), 256, 256
    )

    with rasterio.open(row.s1grd_pre_path) as ds:
        pre = ds.read(1, window=window).astype("float32")

    with rasterio.open(row.s1grd_post_path) as ds:
        post = ds.read(1, window=window).astype("float32")

    # Verified 12-band order: B01, B02, B03, B04, ...
    # Rasterio indices are 1-based: R=B04, G=B03, B=B02.
    with rasterio.open(row.s2l2a_pre_path) as ds:
        rgb = ds.read(
            [4, 3, 2], window=window
        ).astype("float32")

    with rasterio.open(row.label_path) as ds:
        label = ds.read(1, window=window)

    with rasterio.open(row.validity_path) as ds:
        validity = ds.read(1, window=window)

    valid = (
        (validity == 1)
        & (label != 255)
        & np.isin(label, [0, 1, 2])
        & np.isfinite(pre)
        & np.isfinite(post)
        & np.isfinite(rgb).all(axis=0)
        & (pre > 0)
        & (post > 0)
    )

    # Fixed SAR decibel scaling; not OMBRIA-equivalent.
    def scale_sar(image):
        db = 10 * np.log10(np.maximum(image, 1e-8))
        return np.clip((db + 25.0) / 25.0, 0, 1)

    sar_pre = scale_sar(pre)[None]
    sar_post = scale_sar(post)[None]

    rgb = np.clip(rgb / 10000.0, 0, 1)

    assert sar_pre.shape == (1, 256, 256)
    assert sar_post.shape == (1, 256, 256)
    assert rgb.shape == (3, 256, 256)

    return {
        "sar_pre": sar_pre,
        "sar_post": sar_post,
        "ms_pre": rgb,
        "ms_post": rgb.copy(),
        "mask": (label == 2).astype("float32")[None],
        "valid": valid[None]
    }

# ------------------------------------------------------------
# 3. LOAD EXACT EXISTING BEST CHECKPOINTS
# ------------------------------------------------------------

def load_checkpoint(architecture, seed):
    path = (
        BASE / architecture / f"seed{seed}" / "best.pt"
    )
    assert path.is_file(), path

    ckpt = torch.load(
        path, map_location="cpu", weights_only=True
    )

    assert ckpt["architecture"] == architecture
    assert int(ckpt["seed"]) == seed

    model = models[architecture](width=ckpt["width"])
    model.load_state_dict(ckpt["model"], strict=True)
    model.to(DEVICE).eval()
    return model

# ------------------------------------------------------------
# 4. METRIC ACCUMULATION
# ------------------------------------------------------------

@torch.inference_mode()
def evaluate_area(model, rows):
    tp = fp = fn = tn = 0
    valid_total = 0
    windows_used = 0

    for row in rows.itertuples(index=False):
        example = read_external_window(row)

        batch = {
            key: torch.from_numpy(
                np.ascontiguousarray(example[key])
            ).unsqueeze(0).to(DEVICE)
            for key in [
                "sar_pre", "sar_post",
                "ms_pre", "ms_post", "mask"
            ]
        }

        logits = model(batch)

        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        assert logits.shape == (1, 1, 256, 256), (
            f"Unexpected output shape: {logits.shape}"
        )

        pred = (
            logits.sigmoid() >= 0.5
        ).cpu().numpy()[0, 0]

        truth = example["mask"][0] >= 0.5
        valid = example["valid"][0]

        tp += int(np.sum(pred & truth & valid))
        fp += int(np.sum(pred & ~truth & valid))
        fn += int(np.sum(~pred & truth & valid))
        tn += int(np.sum(~pred & ~truth & valid))

        valid_total += int(valid.sum())
        windows_used += 1

    union = tp + fp + fn
    f1_denom = 2 * tp + fp + fn

    return {
        "windows": windows_used,
        "valid_pixels": valid_total,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "iou": tp / union if union else np.nan,
        "f1": 2 * tp / f1_denom if f1_denom else np.nan,
        "precision": tp / (tp + fp) if tp + fp else np.nan,
        "recall": tp / (tp + fn) if tp + fn else np.nan
    }

# ------------------------------------------------------------
# 5. RESUMABLE FOUR-SEED EVALUATION
# ------------------------------------------------------------

RESULTS = OUT / "step95F_external_baseline_results.csv"

COLUMNS = [
    "architecture", "seed", "area", "windows",
    "valid_pixels", "tp", "fp", "fn", "tn",
    "iou", "f1", "precision", "recall"
]

if RESULTS.exists():
    results = pd.read_csv(RESULTS)
    assert set(COLUMNS).issubset(results.columns)
    assert not results.duplicated(
        ["architecture", "seed", "area"]
    ).any()
else:
    results = pd.DataFrame(columns=COLUMNS)

for architecture in ARCHS:

    for seed in SEEDS:

        done = set(zip(
            results.architecture,
            results.seed,
            results.area
        ))

        areas_remaining = [
            area for area in sorted(eligible.area.unique())
            if (architecture, seed, area) not in done
        ]

        if not areas_remaining:
            print("SKIP", architecture, seed)
            continue

        model = load_checkpoint(architecture, seed)

        for area in areas_remaining:

            rows = eligible[eligible.area == area]

            print(
                f"Evaluating {architecture}, "
                f"seed {seed}, area {area}, "
                f"{len(rows)} windows...",
                flush=True
            )

            scores = evaluate_area(model, rows)

            record = {
                "architecture": architecture,
                "seed": seed,
                "area": area,
                **scores
            }

            results = pd.concat(
                [results, pd.DataFrame([record])],
                ignore_index=True
            )

            temporary = RESULTS.with_suffix(".tmp")
            results.to_csv(temporary, index=False)
            os.replace(temporary, RESULTS)

            print(
                f"  IoU={scores['iou']:.5f} "
                f"F1={scores['f1']:.5f} "
                f"valid_pixels={scores['valid_pixels']}",
                flush=True
            )

        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

# ------------------------------------------------------------
# 6. RESULTS SUMMARY
# ------------------------------------------------------------

results = pd.read_csv(RESULTS)

assert len(results) == 24
assert not results.duplicated(
    ["architecture", "seed", "area"]
).any()

# Aggregate counts before computing pooled IoU.
pooled = (
    results.groupby(["architecture", "seed"])[
        ["tp", "fp", "fn", "tn", "valid_pixels"]
    ]
    .sum()
    .reset_index()
)

pooled["iou"] = (
    pooled.tp / (pooled.tp + pooled.fp + pooled.fn)
)

pooled["f1"] = (
    2 * pooled.tp /
    (2 * pooled.tp + pooled.fp + pooled.fn)
)

POOLED_FILE = OUT / "step95F_external_baseline_seed_summary.csv"
pooled.to_csv(POOLED_FILE, index=False)

print("\n" + "=" * 72)
print("STEP 95F — EXTERNAL BENCHMARK RESULTS")
print("=" * 72)

print("\nPER AREA")
print(
    results.groupby(["architecture", "area"])[
        ["iou", "f1"]
    ].mean().round(5).to_string()
)

print("\nPOOLED FOUR-SEED RESULTS")
print(
    pooled.groupby("architecture")[
        ["iou", "f1"]
    ].agg(["mean", "std"]).round(5).to_string()
)

print("\nSaved:", RESULTS)
print("Saved:", POOLED_FILE)
print("No training or checkpoint modification performed.")


In [ ]:

# ============================================================
# STEP 95G — IDENTIFY ORIGINAL UA-FLOODNET LOADER
# Read-only; prepares exact external inference
# ============================================================

from pathlib import Path
import sys, inspect
import torch
import torch.nn as nn
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
CODE = ROOT / "v4/code"
sys.path.insert(0, str(CODE))

import ablation_model
import train_v4

print("=" * 70)
print("STEP 95G — ORIGINAL MODEL IDENTIFICATION")
print("=" * 70)

print("\nMODEL CLASSES")

for name, obj in vars(ablation_model).items():
    if (
        inspect.isclass(obj)
        and issubclass(obj, nn.Module)
        and obj.__module__ == ablation_model.__name__
    ):
        try:
            signature = inspect.signature(obj)
        except Exception:
            signature = "Unavailable"

        print(f"{name}: {signature}")

print("\nTRAINING MODEL REFERENCES")

for name, obj in vars(train_v4).items():
    if inspect.isclass(obj) and issubclass(obj, nn.Module):
        if name.lower() not in {
            "module", "modulelist", "sequential"
        }:
            try:
                print(name, inspect.signature(obj))
            except Exception:
                print(name)

CHECKPOINTS = {
    7: ROOT / "v4/multiseed/light_seed7/best.pt",
    17: ROOT / "v4/multiseed/light_seed17/best.pt",
    37: ROOT / "v4/multiseed/light_seed37/best.pt",
    42: ROOT / "v4/light/light_seed42_50ep.pt"
}

for seed, path in CHECKPOINTS.items():
    print(f"\nSEED {seed}")
    print("Path:", path)
    print("Exists:", path.is_file())

    if not path.is_file():
        continue

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    if isinstance(ckpt, dict):
        print("Checkpoint keys:", list(ckpt.keys()))

        for key in [
            "model", "model_state_dict", "state_dict"
        ]:
            if (
                key in ckpt
                and isinstance(ckpt[key], dict)
            ):
                state = ckpt[key]
                print("Weights key:", key)
                print("Parameter tensors:", len(state))
                print("First five parameter names:")
                for pname in list(state)[:5]:
                    print(" ", pname, tuple(state[pname].shape))
                break

        for key in [
            "config", "args", "width", "epoch",
            "seed", "architecture"
        ]:
            if key in ckpt:
                print(key, ":", str(ckpt[key])[:600])

print("\n" + "=" * 70)
print("STEP 95G COMPLETE")
print("No model weights or data modified.")


In [ ]:

# ============================================================
# STEP 95H-FIX — ORIGINAL UA-FLOODNET FORWARD SIGNATURE
# Fix inference call; preserve all existing checkpoints
# ============================================================

import torch
import pandas as pd
import numpy as np
import os

# Replace only the evaluation function.
# Reuse the original read_window(), load_model(),
# eligible, DEVICE, OUT, CHECKPOINTS and SEEDS.

@torch.inference_mode()
def evaluate_area(model, rows):

    tp = fp = fn = tn = 0
    valid_total = 0
    count = 0

    for row in rows.itertuples(index=False):

        item = read_window(row)

        sar_pre = torch.from_numpy(
            np.ascontiguousarray(item["sar_pre"])
        ).unsqueeze(0).to(DEVICE)

        sar_post = torch.from_numpy(
            np.ascontiguousarray(item["sar_post"])
        ).unsqueeze(0).to(DEVICE)

        ms_pre = torch.from_numpy(
            np.ascontiguousarray(item["ms_pre"])
        ).unsqueeze(0).to(DEVICE)

        ms_post = torch.from_numpy(
            np.ascontiguousarray(item["ms_post"])
        ).unsqueeze(0).to(DEVICE)

        # CORRECT: four separate positional arguments.
        output = model(
            sar_pre,
            sar_post,
            ms_pre,
            ms_post
        )

        logits = segmentation_logits(output)

        assert logits.shape == (1, 1, 256, 256), (
            f"Unexpected output shape: {tuple(logits.shape)}"
        )

        prediction = (
            logits.sigmoid() >= 0.5
        ).cpu().numpy()[0, 0]

        truth = item["mask"][0].astype(bool)
        valid = item["valid"][0].astype(bool)

        tp += int((prediction & truth & valid).sum())
        fp += int((prediction & ~truth & valid).sum())
        fn += int((~prediction & truth & valid).sum())
        tn += int((~prediction & ~truth & valid).sum())

        valid_total += int(valid.sum())
        count += 1

    iou_denom = tp + fp + fn
    f1_denom = 2 * tp + fp + fn

    return {
        "windows": count,
        "valid_pixels": valid_total,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "iou": tp / iou_denom if iou_denom else np.nan,
        "f1": (
            2 * tp / f1_denom
            if f1_denom else np.nan
        ),
        "precision": (
            tp / (tp + fp)
            if tp + fp else np.nan
        ),
        "recall": (
            tp / (tp + fn)
            if tp + fn else np.nan
        )
    }

# ============================================================
# RESUME ALL FOUR UA-FLOODNET SEEDS
# ============================================================

CSV = OUT / "step95H_UAFloodNet_external_results.csv"

COLUMNS = [
    "architecture", "seed", "area", "windows",
    "valid_pixels", "tp", "fp", "fn", "tn",
    "iou", "f1", "precision", "recall"
]

if CSV.exists():
    results = pd.read_csv(CSV)
    assert set(COLUMNS).issubset(results.columns)
else:
    results = pd.DataFrame(columns=COLUMNS)

assert not results.duplicated(
    ["architecture", "seed", "area"]
).any()

for seed in SEEDS:

    completed = set(zip(
        results.architecture,
        results.seed,
        results.area
    ))

    pending = [
        area
        for area in sorted(eligible.area.unique())
        if ("UA-FloodNet", seed, area) not in completed
    ]

    if not pending:
        print("SKIP completed seed:", seed)
        continue

    model = load_model(seed)

    for area in pending:

        rows = eligible[eligible.area == area]

        print(
            f"Evaluating UA-FloodNet | "
            f"Seed {seed} | {area} | "
            f"{len(rows)} windows",
            flush=True
        )

        scores = evaluate_area(model, rows)

        record = {
            "architecture": "UA-FloodNet",
            "seed": seed,
            "area": area,
            **scores
        }

        results = pd.concat(
            [results, pd.DataFrame([record])],
            ignore_index=True
        )

        temporary = CSV.with_suffix(".tmp")
        results.to_csv(temporary, index=False)
        os.replace(temporary, CSV)

        print(
            f"  IoU={scores['iou']:.5f} | "
            f"F1={scores['f1']:.5f} | "
            f"Valid pixels={scores['valid_pixels']}",
            flush=True
        )

    del model

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# ============================================================
# FINAL FOUR-MODEL COMPARISON
# ============================================================

ua = pd.read_csv(CSV)

assert len(ua) == 8, (
    f"Expected 8 UA-FloodNet results; found {len(ua)}"
)

baseline = pd.read_csv(
    OUT / "step95F_external_baseline_results.csv"
)

combined = pd.concat(
    [baseline, ua],
    ignore_index=True
)

assert len(combined) == 32
assert not combined.duplicated(
    ["architecture", "seed", "area"]
).any()

support = combined.groupby("area")[
    ["windows", "valid_pixels"]
].nunique()

assert (
    (support.windows == 1)
    & (support.valid_pixels == 1)
).all(), "Evaluation support is inconsistent across architectures"

combined.to_csv(
    OUT / "step95H_four_model_external_results.csv",
    index=False
)

pooled = (
    combined.groupby(["architecture", "seed"])[
        ["tp", "fp", "fn", "tn", "valid_pixels"]
    ]
    .sum()
    .reset_index()
)

pooled["iou"] = (
    pooled.tp / (pooled.tp + pooled.fp + pooled.fn)
)

pooled["f1"] = (
    2 * pooled.tp /
    (2 * pooled.tp + pooled.fp + pooled.fn)
)

pooled.to_csv(
    OUT / "step95H_four_model_external_seed_summary.csv",
    index=False
)

print("\n" + "=" * 74)
print("STEP 95H — COMPLETE FOUR-MODEL EXTERNAL COMPARISON")
print("=" * 74)

print(
    pooled.groupby("architecture")[["iou", "f1"]]
    .agg(["mean", "std"])
    .round(5)
    .to_string()
)

print("\nAll original checkpoints preserved.")
print("No retraining performed.")
print("Saved results to:", OUT)


In [ ]:

# =============================================================
# STEP 96 — COMPUTATIONAL EFFICIENCY BENCHMARK
# 4 architectures x 4 trained seeds
# CUDA inference latency, FPS, peak GPU memory, parameters
# No training; no dataset download; no checkpoint modification
# =============================================================

from pathlib import Path
import sys, importlib.util, gc, json, time
import numpy as np
import pandas as pd
import torch
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
CODE = ROOT / "v4/code"
BENCH = ROOT / "v4/final_analysis/benchmark_study"
OUT = ROOT / "v4/final_analysis/efficiency_study"
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(CODE))
from ablation_model import UAFloodNet

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
SEEDS = [7, 17, 37, 42]
BATCH_SIZE = 1
IMAGE_SIZE = 256
WARMUP = 20
REPEATS = 80

BASELINES = {
    "standard_unet": "train_unet_benchmark.py",
    "siamese_unet": "train_siamese_unet.py",
    "two_stage_fusion_unet": "train_two_stage_fusion_unet.py"
}

UA_CKPTS = {
    7: ROOT / "v4/multiseed/light_seed7/best.pt",
    17: ROOT / "v4/multiseed/light_seed17/best.pt",
    37: ROOT / "v4/multiseed/light_seed37/best.pt",
    42: ROOT / "v4/light/light_seed42_50ep.pt"
}

print("=" * 72)
print("STEP 96 — COMPUTATIONAL EFFICIENCY BENCHMARK")
print("=" * 72)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(DEVICE))
print("Input:", BATCH_SIZE, "x", IMAGE_SIZE, "x", IMAGE_SIZE)

# -------------------------------------------------------------
# 1. Load original architecture definitions
# -------------------------------------------------------------

CLASSES = {}

for name, filename in BASELINES.items():
    path = BENCH / filename
    assert path.is_file(), path
    spec = importlib.util.spec_from_file_location(
        "efficiency_" + name, str(path)
    )
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    CLASSES[name] = mod.UNet

def load_model(name, seed):
    if name == "UA-FloodNet":
        ckpt_path = UA_CKPTS[seed]
        ckpt = torch.load(
            ckpt_path, map_location="cpu",
            weights_only=True
        )
        assert int(ckpt["seed"]) == seed
        assert bool(ckpt["use_alignment"])
        assert bool(ckpt["use_reliability"])
        model = UAFloodNet(
            sar_channels=1,
            ms_channels=3,
            width=int(ckpt["width"]),
            max_shift=4.0,
            use_alignment=True,
            use_reliability=True
        )
    else:
        ckpt_path = (
            BENCH / name / f"seed{seed}" / "best.pt"
        )
        ckpt = torch.load(
            ckpt_path, map_location="cpu",
            weights_only=True
        )
        assert ckpt["architecture"] == name
        assert int(ckpt["seed"]) == seed
        model = CLASSES[name](width=ckpt["width"])

    model.load_state_dict(ckpt["model"], strict=True)
    return model.to(DEVICE).eval()

# -------------------------------------------------------------
# 2. Fixed synthetic inputs (timing excludes raster I/O)
# -------------------------------------------------------------

torch.manual_seed(2026)

batch = {
    "sar_pre": torch.rand(
        BATCH_SIZE, 1, IMAGE_SIZE, IMAGE_SIZE,
        device=DEVICE
    ),
    "sar_post": torch.rand(
        BATCH_SIZE, 1, IMAGE_SIZE, IMAGE_SIZE,
        device=DEVICE
    ),
    "ms_pre": torch.rand(
        BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE,
        device=DEVICE
    ),
    "ms_post": torch.rand(
        BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE,
        device=DEVICE
    )
}

def forward_model(model, name):
    if name == "UA-FloodNet":
        return model(
            batch["sar_pre"],
            batch["sar_post"],
            batch["ms_pre"],
            batch["ms_post"]
        )
    return model(batch)

def get_logits(output):
    if isinstance(output, torch.Tensor):
        return output
    if isinstance(output, (tuple, list)):
        return output[0]
    if isinstance(output, dict):
        for key in (
            "logits", "seg_logits", "segmentation_logits"
        ):
            if key in output:
                return output[key]
    raise TypeError(
        f"Unexpected model output: {type(output)}"
    )

# -------------------------------------------------------------
# 3. Benchmark every original seed
# -------------------------------------------------------------

RESULTS = OUT / "step96_seed_efficiency.csv"

columns = [
    "architecture", "seed", "parameters",
    "parameters_millions", "latency_mean_ms",
    "latency_median_ms", "latency_p95_ms",
    "images_per_second", "peak_allocated_gpu_mb",
    "device", "batch_size", "image_size"
]

if RESULTS.exists():
    records = pd.read_csv(RESULTS)
    assert set(columns).issubset(records.columns)
else:
    records = pd.DataFrame(columns=columns)

assert not records.duplicated(
    ["architecture", "seed"]
).any()

architectures = ["UA-FloodNet", *BASELINES.keys()]

for name in architectures:
    for seed in SEEDS:

        exists = (
            (records.architecture == name)
            & (records.seed == seed)
        ).any()

        if exists:
            print("SKIP:", name, seed)
            continue

        print("\nBenchmarking:", name, "seed", seed, flush=True)

        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

        model = load_model(name, seed)

        parameters = sum(
            p.numel() for p in model.parameters()
        )

        with torch.inference_mode():
            output = get_logits(forward_model(model, name))
            assert output.shape == (
                BATCH_SIZE, 1, IMAGE_SIZE, IMAGE_SIZE
            ), tuple(output.shape)

            for _ in range(WARMUP):
                _ = get_logits(forward_model(model, name))

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
                torch.cuda.reset_peak_memory_stats()

            times_ms = []

            for _ in range(REPEATS):
                if DEVICE.type == "cuda":
                    start = torch.cuda.Event(enable_timing=True)
                    end = torch.cuda.Event(enable_timing=True)
                    start.record()
                    prediction = get_logits(
                        forward_model(model, name)
                    )
                    end.record()
                    end.synchronize()
                    times_ms.append(
                        float(start.elapsed_time(end))
                    )
                else:
                    t0 = time.perf_counter()
                    prediction = get_logits(
                        forward_model(model, name)
                    )
                    times_ms.append(
                        (time.perf_counter() - t0) * 1000
                    )

            if DEVICE.type == "cuda":
                peak_mb = (
                    torch.cuda.max_memory_allocated()
                    / (1024**2)
                )
            else:
                peak_mb = np.nan

        times_ms = np.asarray(times_ms)
        mean_ms = float(times_ms.mean())

        row = {
            "architecture": name,
            "seed": seed,
            "parameters": int(parameters),
            "parameters_millions": parameters / 1e6,
            "latency_mean_ms": mean_ms,
            "latency_median_ms": float(
                np.median(times_ms)
            ),
            "latency_p95_ms": float(
                np.percentile(times_ms, 95)
            ),
            "images_per_second": (
                1000 * BATCH_SIZE / mean_ms
            ),
            "peak_allocated_gpu_mb": peak_mb,
            "device": str(DEVICE),
            "batch_size": BATCH_SIZE,
            "image_size": IMAGE_SIZE
        }

        records = pd.concat(
            [records, pd.DataFrame([row])],
            ignore_index=True
        )

        temp = RESULTS.with_suffix(".tmp")
        records.to_csv(temp, index=False)
        temp.replace(RESULTS)

        print(
            f"Params={parameters/1e6:.3f}M | "
            f"Latency={mean_ms:.2f}ms | "
            f"Throughput={row['images_per_second']:.1f} img/s | "
            f"Peak={peak_mb:.1f} MB",
            flush=True
        )

        del model, output, prediction
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

# -------------------------------------------------------------
# 4. Final reproducibility checks
# -------------------------------------------------------------

records = pd.read_csv(RESULTS)

assert len(records) == 16
assert not records.duplicated(
    ["architecture", "seed"]
).any()

summary = (
    records.groupby("architecture")[
        [
            "parameters_millions",
            "latency_mean_ms",
            "images_per_second",
            "peak_allocated_gpu_mb"
        ]
    ]
    .agg(["mean", "std"])
)

SUMMARY = OUT / "step96_efficiency_summary.csv"
summary.to_csv(SUMMARY)

protocol = {
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "warmup_iterations": WARMUP,
    "timed_iterations": REPEATS,
    "seeds": SEEDS,
    "device": str(DEVICE),
    "gpu_name": (
        torch.cuda.get_device_name(DEVICE)
        if DEVICE.type == "cuda" else None
    ),
    "dtype": "float32",
    "autocast": False,
    "timing": (
        "CUDA events; GPU synchronized per iteration"
        if DEVICE.type == "cuda"
        else "time.perf_counter"
    ),
    "memory": (
        "Peak allocated GPU memory during timed forward "
        "passes; includes model parameters and resident input "
        "tensors, excludes image loading and preprocessing"
    ),
    "note": (
        "This is forward-pass-only inference efficiency "
        "on synthetic fixed-size tensors. It is not end-to-end "
        "geospatial processing throughput."
    )
}

PROTOCOL = OUT / "step96_efficiency_protocol.json"
PROTOCOL.write_text(
    json.dumps(protocol, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 72)
print("STEP 96 COMPLETED")
print("=" * 72)
print(summary.round(3).to_string())
print("\nSaved:", RESULTS)
print("Saved:", SUMMARY)
print("Saved:", PROTOCOL)
print("Original model checkpoints unchanged.")


In [ ]:

# STEP 97A — HELD-OUT DATA DOWNLOAD PLAN
# Metadata only; no imagery download or retraining

from pathlib import Path
import pandas as pd
from huggingface_hub import HfApi
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

OUT = (
    MOUNT / "MyDrive/UA_FloodNet/v4/"
    "final_analysis/generalization_study"
)
OUT.mkdir(parents=True, exist_ok=True)

api = HfApi()
info = api.repo_info(
    repo_id="links-ads/geoid-flood",
    repo_type="dataset",
    files_metadata=True
)

layers = ["s1grd", "s2l2a", "label", "validity"]
rows = []

for item in info.siblings:
    path = item.rfilename
    parts = Path(path).parts

    if (
        path.endswith(".tar")
        and len(parts) >= 5
        and parts[0] == "geoid-flood-heldout"
        and parts[1] == "shards"
        and parts[2] == "test"
        and parts[3] in layers
    ):
        rows.append({
            "layer": parts[3],
            "path": path,
            "size_bytes": item.size
        })

manifest = pd.DataFrame(rows)
assert not manifest.empty, "No held-out shard files found."

summary = (
    manifest.groupby("layer", as_index=False)
    .agg(
        shards=("path", "size"),
        bytes=("size_bytes", "sum")
    )
)

summary["size_GB"] = summary["bytes"] / 1e9

print("=" * 68)
print("STEP 97A — INDEPENDENT HELD-OUT DOWNLOAD PLAN")
print("=" * 68)

print(summary.to_string(index=False))
print(
    "\nTOTAL DOWNLOAD:",
    round(summary["size_GB"].sum(), 2),
    "GB"
)

print(
    "\nFiles with unknown sizes:",
    int(manifest.size_bytes.isna().sum())
)

manifest_file = OUT / "step97A_heldout_shard_manifest.csv"
manifest.to_csv(manifest_file, index=False)

print("\nSaved:", manifest_file)
print("No imagery downloaded; no checkpoints modified.")


In [ ]:

# ================================================================
# STEP 97B — GEOID-FLOOD NINE-EVENT HELD-OUT DOWNLOAD
# 28 shards | ~50.87 GB compressed
# Resumable | Local Colab storage | Checkpoint-safe
#
# REQUIRED LAYERS:
#  s1grd (pre/post), s2l2a (pre), label, validity
# ================================================================

from pathlib import Path
import subprocess
import shutil
import sys
import os
import json
import pandas as pd

from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
OUT = ROOT / "v4/final_analysis/generalization_study"
OUT.mkdir(parents=True, exist_ok=True)

SCRIPT = Path(
    "/content/geoid_flood_official/scripts/get_data.py"
)

LOCAL = Path("/content/geoid_flood_heldout_local")
TREE = LOCAL / "geoid-flood-heldout"

CACHE = Path(
    "/root/.cache/huggingface/hub/"
    "datasets--links-ads--geoid-flood/blobs"
)

MANIFEST = OUT / "step97A_heldout_shard_manifest.csv"

assert SCRIPT.is_file(), SCRIPT
assert MANIFEST.is_file(), MANIFEST

manifest = pd.read_csv(MANIFEST)

assert len(manifest) == 28
assert manifest["size_bytes"].notna().all()

EXPECTED_LAYERS = {
    "s1grd", "s2l2a", "label", "validity"
}

assert set(manifest.layer) == EXPECTED_LAYERS

download_gb = manifest.size_bytes.sum() / 1e9

print("=" * 75)
print("STEP 97B — NINE-EVENT EXTERNAL DATASET")
print("=" * 75)

print("Shards:", len(manifest))
print("Compressed data:", round(download_gb, 2), "GB")
print("Destination:", LOCAL)

# ------------------------------------------------------------
# 1. STORAGE PREFLIGHT
# ------------------------------------------------------------

free_gib = shutil.disk_usage("/content").free / (1024**3)

print("\nAvailable local storage:", round(free_gib, 2), "GiB")

# Conservative allowance for Hugging Face cache,
# downloaded shards and extracted imagery.
if free_gib < 145:
    raise RuntimeError(
        f"Only {free_gib:.1f} GiB free. "
        "A large download could exhaust the Colab runtime. "
        "Use a larger runtime disk or a staged download."
    )

# ------------------------------------------------------------
# 2. VERIFY OFFICIAL DOWNLOAD SELECTION
# ------------------------------------------------------------

command = [
    sys.executable,
    str(SCRIPT),
    "--tree", "geoid-flood-heldout",
    "--split", "test",
    "--layer", "s1grd", "s2l2a", "label", "validity",
    "--dest", str(LOCAL),
    "--workers", "2",
    "--extract-workers", "2"
]

preview = subprocess.run(
    command + ["--list"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("\nOFFICIAL SELECTION PREVIEW")
print(preview.stdout[-5000:])

if preview.returncode != 0:
    raise RuntimeError("Downloader selection preview failed.")

# ------------------------------------------------------------
# 3. DOWNLOAD TO LOCAL FILESYSTEM
# ------------------------------------------------------------

LOCAL.mkdir(parents=True, exist_ok=True)

print("\nSTARTING HELD-OUT DATA DOWNLOAD")
print("Existing completed downloads will be preserved.")
print("Do not mount this destination on Google Drive.")

# Stream the actual downloader output.
# Do not suppress errors or capture large output in RAM.
subprocess.run(
    command,
    check=True
)

# ------------------------------------------------------------
# 4. REPAIR HUGGING FACE CACHE SYMBOLIC LINKS
# ------------------------------------------------------------

print("\nREPAIRING CACHE LINKS")

broken = [
    p for p in TREE.rglob("*")
    if p.is_symlink() and not p.exists()
]

repaired = []
unresolved = []

for p in broken:
    target = os.readlink(p)
    blob = CACHE / Path(target).name

    if blob.is_file():
        p.unlink()
        p.symlink_to(blob)
        repaired.append(str(p.relative_to(TREE)))
    else:
        unresolved.append(str(p.relative_to(TREE)))

print("Broken links found:", len(broken))
print("Repaired:", len(repaired))
print("Unresolved:", len(unresolved))

# ------------------------------------------------------------
# 5. INVENTORY ACTUAL EXTRACTED RASTERS
# ------------------------------------------------------------

layers = [
    "s1grd", "s2l2a", "label", "validity"
]

tifs = list(TREE.rglob("*.tif"))

rows = []

for layer in layers:
    selected = [
        p for p in tifs
        if layer in [
            part.lower()
            for part in p.relative_to(TREE).parts
        ]
    ]

    resolved = sum(p.is_file() for p in selected)

    rows.append({
        "layer": layer,
        "raster_paths": len(selected),
        "resolved_paths": resolved,
        "unresolved_paths": len(selected) - resolved
    })

inventory = pd.DataFrame(rows)

print("\nEXTRACTED RASTER INVENTORY")
print(inventory.to_string(index=False))

# ------------------------------------------------------------
# 6. READABILITY CHECK
# ------------------------------------------------------------

import rasterio

read_failures = []
readable = 0

for i, path in enumerate(tifs, 1):
    try:
        with rasterio.open(path) as ds:
            assert ds.count > 0
            assert ds.width > 0 and ds.height > 0
            _ = ds.read(
                1,
                window=rasterio.windows.Window(
                    0, 0,
                    min(32, ds.width),
                    min(32, ds.height)
                )
            )
        readable += 1
    except Exception as exc:
        read_failures.append({
            "path": str(path.relative_to(TREE)),
            "error": str(exc)
        })

    if i % 500 == 0:
        print(f"Verified raster readability: {i}/{len(tifs)}")

# ------------------------------------------------------------
# 7. SAVE REPORT
# ------------------------------------------------------------

inventory_file = OUT / "step97B_heldout_raster_inventory.csv"
inventory.to_csv(inventory_file, index=False)

# Detect event directories from actual imagery.
event_dirs = sorted({
    p.relative_to(TREE).parts[0].split("-")[0]
    for p in tifs
    if len(p.relative_to(TREE).parts) >= 3
    and p.relative_to(TREE).parts[0].startswith("EMSR")
})

complete = (
    len(tifs) > 0
    and not unresolved
    and not read_failures
    and all(inventory.unresolved_paths == 0)
    and all(inventory.raster_paths > 0)
)

report = {
    "step": "97B",
    "downloaded_shards_expected": len(manifest),
    "expected_compressed_gb": download_gb,
    "raster_files": len(tifs),
    "readable_rasters": readable,
    "event_ids_found": event_dirs,
    "broken_links_repaired": len(repaired),
    "unresolved_links": unresolved,
    "raster_read_failures": read_failures[:100],
    "all_files_readable": complete,
    "local_directory": str(TREE),
    "checkpoint_modified": False
}

REPORT = OUT / "step97B_heldout_download_report.json"
REPORT.write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 75)
print("STEP 97B — FINAL STATUS")
print("=" * 75)

print("Actual raster files:", len(tifs))
print("Readable rasters:", readable)
print("Detected event IDs:", event_dirs)
print("Raster read failures:", len(read_failures))
print("Remaining broken links:", len(unresolved))
print("Saved:", REPORT)

if complete:
    print("\nPASS — HELD-OUT IMAGERY READY FOR WINDOW AUDIT")
else:
    print("\nPARTIAL — REVIEW THE REPORT BEFORE EVALUATION")

print("UA-FloodNet checkpoints remain unchanged.")


In [ ]:

# ============================================================
# STEP 97C — NINE-EVENT HELD-OUT WINDOW AUDIT
# Exact source-file matching and spatial alignment
# Resumable; no training; no checkpoint modification
# ============================================================

from pathlib import Path
from collections import Counter
import json, os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
DATA = Path(
    "/content/geoid_flood_heldout_local/geoid-flood-heldout"
)
OUT = ROOT / "v4/final_analysis/generalization_study"
OUT.mkdir(parents=True, exist_ok=True)

assert DATA.is_dir(), "Held-out data directory missing"

EXPECTED_EVENTS = {
    "EMSR857", "EMSR858", "EMSR861",
    "EMSR864", "EMSR865", "EMSR866",
    "EMSR868", "EMSR870", "EMSR871"
}

SIZE = 256
STRIDE = 256
MIN_VALID_FRACTION = 0.50

SCENE_FILE = OUT / "step97C_scene_audit.csv"
WINDOW_FILE = OUT / "step97C_heldout_window_manifest.csv"

scene_columns = [
    "event_id", "area", "scene", "status",
    "sar_pre_path", "sar_post_path",
    "s2_pre_path", "label_path", "validity_path",
    "width", "height", "alignment_ok",
    "valid_window_count", "total_window_count",
    "flood_positive_window_count", "error"
]

window_columns = [
    "event_id", "area", "scene", "x", "y", "size",
    "valid_fraction", "valid_pixels",
    "flood_pixels", "permanent_water_pixels",
    "background_pixels",
    "sar_pre_path", "sar_post_path",
    "s2_pre_path", "label_path", "validity_path"
]

def read_csv_or_empty(path, columns):
    if path.exists():
        df = pd.read_csv(path)
        assert set(columns).issubset(df.columns)
        assert not df.duplicated(["area", "scene"] if
            "x" not in columns else
            ["area", "scene", "x", "y"]).any()
        return df[columns]
    return pd.DataFrame(columns=columns)

scenes = read_csv_or_empty(SCENE_FILE, scene_columns)
windows = read_csv_or_empty(WINDOW_FILE, window_columns)

def atomic_save(df, path):
    tmp = path.with_suffix(".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def choose_match(folder, pattern):
    if not folder.is_dir():
        return None, 0
    matches = sorted(
        p for p in folder.glob(pattern)
        if p.is_file()
    )
    # Never silently select among multiple acquisitions.
    if len(matches) == 1:
        return matches[0], 1
    return None, len(matches)

label_files = sorted(DATA.glob("EMSR*/label/*_label.tif"))

print("=" * 75)
print("STEP 97C — INDEPENDENT EVENT WINDOW AUDIT")
print("=" * 75)
print("Label rasters:", len(label_files))
print("Previously audited scenes:", len(scenes))

assert len(label_files) > 0

for index, label_path in enumerate(label_files, 1):

    area_dir = label_path.parent.parent
    area = area_dir.name
    event = area.split("-")[0]
    scene = label_path.stem.removesuffix("_label")

    assert event in EXPECTED_EVENTS, (
        f"Unexpected event: {event}"
    )

    if (
        (scenes["area"] == area)
        & (scenes["scene"] == scene)
    ).any():
        continue

    # Match exact source scene identifier and modality.
    sar_pre, n_pre = choose_match(
        area_dir / "s1grd",
        f"{scene}_s1grd_pre_*.tif"
    )
    sar_post, n_post = choose_match(
        area_dir / "s1grd",
        f"{scene}_s1grd_post_*.tif"
    )
    optical, n_opt = choose_match(
        area_dir / "s2l2a",
        f"{scene}_s2l2a_pre_*.tif"
    )

    validity_path = (
        area_dir / "validity" / f"{scene}_validity.tif"
    )

    paths = {
        "sar_pre_path": sar_pre,
        "sar_post_path": sar_post,
        "s2_pre_path": optical,
        "label_path": label_path,
        "validity_path": validity_path
    }

    status = "complete"
    error = ""
    aligned = False
    width = height = 0
    total_count = eligible_count = positive_count = 0
    scene_windows = []

    if (
        any(p is None for p in [sar_pre, sar_post, optical])
        or not validity_path.is_file()
    ):
        status = "missing_or_ambiguous_modality"
        error = (
            f"SAR pre={n_pre}, post={n_post}, "
            f"S2 pre={n_opt}, "
            f"validity={validity_path.is_file()}"
        )

    else:
        try:
            with rasterio.open(label_path) as lbl, \
                 rasterio.open(validity_path) as vld, \
                 rasterio.open(sar_pre) as sp, \
                 rasterio.open(sar_post) as sq, \
                 rasterio.open(optical) as op:

                width, height = lbl.width, lbl.height

                aligned = all(
                    ds.crs == lbl.crs
                    and ds.width == lbl.width
                    and ds.height == lbl.height
                    and ds.transform.almost_equals(lbl.transform)
                    for ds in [vld, sp, sq, op]
                )

                band_ok = (
                    lbl.count == 1
                    and vld.count == 1
                    and sp.count >= 1
                    and sq.count >= 1
                    and op.count >= 4
                )

                if not aligned or not band_ok:
                    status = "alignment_or_band_failure"
                    error = (
                        f"alignment={aligned}, bands="
                        f"{[lbl.count,vld.count,sp.count,sq.count,op.count]}"
                    )
                else:
                    for y in range(0, height-SIZE+1, STRIDE):
                        for x in range(0, width-SIZE+1, STRIDE):
                            total_count += 1
                            win = Window(x, y, SIZE, SIZE)

                            lab = lbl.read(1, window=win)
                            val = vld.read(1, window=win)

                            valid = (
                                (val == 1)
                                & np.isin(lab, [0,1,2])
                            )

                            count = int(valid.sum())
                            fraction = count / (SIZE * SIZE)

                            if fraction < MIN_VALID_FRACTION:
                                continue

                            flood = int(
                                ((lab == 2) & valid).sum()
                            )
                            perm = int(
                                ((lab == 1) & valid).sum()
                            )
                            background = int(
                                ((lab == 0) & valid).sum()
                            )

                            eligible_count += 1
                            positive_count += int(flood > 0)

                            scene_windows.append({
                                "event_id": event,
                                "area": area,
                                "scene": scene,
                                "x": x, "y": y, "size": SIZE,
                                "valid_fraction": fraction,
                                "valid_pixels": count,
                                "flood_pixels": flood,
                                "permanent_water_pixels": perm,
                                "background_pixels": background,
                                **{
                                    k: str(p)
                                    for k, p in paths.items()
                                }
                            })

        except Exception as exc:
            status = "read_error"
            error = repr(exc)
            scene_windows = []
            eligible_count = positive_count = 0

    scene_record = {
        "event_id": event,
        "area": area,
        "scene": scene,
        "status": status,
        **{
            k: str(p) if p is not None else ""
            for k, p in paths.items()
        },
        "width": width,
        "height": height,
        "alignment_ok": bool(aligned),
        "valid_window_count": eligible_count,
        "total_window_count": total_count,
        "flood_positive_window_count": positive_count,
        "error": error
    }

    # Save window data first; mark the scene audited afterward.
    if scene_windows:
        windows = pd.concat(
            [windows, pd.DataFrame(scene_windows)],
            ignore_index=True
        )
        atomic_save(windows, WINDOW_FILE)

    scenes = pd.concat(
        [scenes, pd.DataFrame([scene_record])],
        ignore_index=True
    )
    atomic_save(scenes, SCENE_FILE)

    if index % 100 == 0 or index == len(label_files):
        print(
            f"Processed {index}/{len(label_files)} labels; "
            f"eligible windows={len(windows)}",
            flush=True
        )

# ------------------------------------------------------------
# SUMMARY AND EVENT COVERAGE
# ------------------------------------------------------------

scenes = pd.read_csv(SCENE_FILE)

if WINDOW_FILE.exists():
    windows = pd.read_csv(WINDOW_FILE)
else:
    windows = pd.DataFrame(columns=window_columns)

print("\n" + "=" * 75)
print("STEP 97C — FINAL SCENE AND WINDOW RESULTS")
print("=" * 75)

print("\nSCENE STATUS")
print(scenes.status.value_counts().to_string())

print("\nEVENT-WISE EVALUATION COVERAGE")

if not windows.empty:
    event_summary = (
        windows.groupby("event_id")
        .agg(
            windows=("scene","size"),
            scenes=("scene","nunique"),
            valid_pixels=("valid_pixels","sum"),
            flooded_pixels=("flood_pixels","sum"),
            flood_positive_windows=(
                "flood_pixels",
                lambda s: int((s > 0).sum())
            )
        )
        .reset_index()
        .sort_values("event_id")
    )
else:
    event_summary = pd.DataFrame()

print(
    event_summary.to_string(index=False)
    if not event_summary.empty
    else "No eligible windows"
)

SUMMARY = OUT / "step97C_event_window_summary.csv"
event_summary.to_csv(SUMMARY,index=False)

observed_events = (
    set(windows.event_id.astype(str))
    if not windows.empty else set()
)

print("\nExpected events:", len(EXPECTED_EVENTS))
print("Events with eligible windows:", len(observed_events))
print("Total audited scenes:", len(scenes))
print("Total eligible windows:", len(windows))

print("\nSaved:")
print(SCENE_FILE)
print(WINDOW_FILE)
print(SUMMARY)

if observed_events == EXPECTED_EVENTS:
    print("\nPASS — All nine events have eligible windows.")
else:
    print(
        "\nCAUTION — Some events have no eligible windows. "
        "Review the scene audit before inference."
    )

print("No models retrained or checkpoints modified.")


In [ ]:

# ================================================================
# STEP 97D — NINE-EVENT EXTERNAL FLOOD SEGMENTATION
# 4 architectures x 4 seeds x 9 held-out events
# 14,151 windows per model | 226,416 total predictions
# Resumable; no training or checkpoint modification
# ================================================================

from pathlib import Path
from collections import defaultdict
from contextlib import ExitStack
import sys, os, gc, importlib.util, json
import numpy as np
import pandas as pd
import torch
import rasterio
from rasterio.windows import Window
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
CODE = ROOT / "v4/code"
BASE = ROOT / "v4/final_analysis/benchmark_study"
OUT = ROOT / "v4/final_analysis/generalization_study"
RUNS = OUT / "step97D_scene_results"
RUNS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(CODE))
from ablation_model import UAFloodNet

MANIFEST = OUT / "step97C_heldout_window_manifest.csv"
df = pd.read_csv(MANIFEST)

EXPECTED_EVENTS = [
    "EMSR857", "EMSR858", "EMSR861",
    "EMSR864", "EMSR865", "EMSR866",
    "EMSR868", "EMSR870", "EMSR871"
]
SEEDS = [7, 17, 37, 42]
ARCHITECTURES = [
    "UA-FloodNet",
    "standard_unet",
    "siamese_unet",
    "two_stage_fusion_unet"
]
BATCH_SIZE = 4
SAVE_EVERY_SCENES = 20

assert len(df) == 14151
assert set(df.event_id) == set(EXPECTED_EVENTS)
assert not df.duplicated(
    ["event_id", "area", "scene", "x", "y"]
).any()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
assert DEVICE.type == "cuda", "Use a GPU Colab runtime."

print("=" * 76)
print("STEP 97D — NINE-EVENT EXTERNAL EVALUATION")
print("=" * 76)
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("Events:", len(EXPECTED_EVENTS))
print("Windows per model:", len(df))
print("Architectures:", len(ARCHITECTURES))
print("Seeds:", len(SEEDS))

# ------------------------------------------------------------
# 1. LOAD EXACT EXISTING ARCHITECTURES
# ------------------------------------------------------------

baseline_scripts = {
    "standard_unet": "train_unet_benchmark.py",
    "siamese_unet": "train_siamese_unet.py",
    "two_stage_fusion_unet": "train_two_stage_fusion_unet.py"
}

classes = {}

for name, filename in baseline_scripts.items():
    path = BASE / filename
    assert path.is_file(), path

    spec = importlib.util.spec_from_file_location(
        "heldout_" + name, str(path)
    )
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    classes[name] = module.UNet

ua_checkpoints = {
    7: ROOT / "v4/multiseed/light_seed7/best.pt",
    17: ROOT / "v4/multiseed/light_seed17/best.pt",
    37: ROOT / "v4/multiseed/light_seed37/best.pt",
    42: ROOT / "v4/light/light_seed42_50ep.pt"
}

def load_model(name, seed):

    if name == "UA-FloodNet":
        path = ua_checkpoints[seed]
    else:
        path = BASE / name / f"seed{seed}/best.pt"

    ckpt = torch.load(
        path, map_location="cpu", weights_only=True
    )
    assert int(ckpt["seed"]) == seed

    if name == "UA-FloodNet":
        assert ckpt["use_alignment"]
        assert ckpt["use_reliability"]
        model = UAFloodNet(
            sar_channels=1,
            ms_channels=3,
            width=int(ckpt["width"]),
            max_shift=4.0,
            use_alignment=True,
            use_reliability=True
        )
    else:
        assert ckpt["architecture"] == name
        model = classes[name](
            width=int(ckpt["width"])
        )

    model.load_state_dict(
        ckpt["model"], strict=True
    )
    return model.to(DEVICE).eval()

def get_logits(output):
    if torch.is_tensor(output):
        return output
    if isinstance(output, (tuple, list)):
        return output[0]
    if isinstance(output, dict):
        for key in (
            "logits", "seg_logits", "segmentation_logits"
        ):
            if key in output:
                return output[key]
    raise TypeError(f"Unexpected output: {type(output)}")

# ------------------------------------------------------------
# 2. SHARED GEOID INPUT ADAPTATION — IDENTICAL TO STEP 95F
# ------------------------------------------------------------

def sar_scale(a):
    db = 10.0 * np.log10(
        np.maximum(a.astype(np.float32), 1e-8)
    )
    return np.clip((db + 25) / 25, 0, 1).astype(np.float32)

def read_scene_windows(rows):
    """
    Read one matched scene, yielding aligned windows.
    Missing post-event optical is represented by duplicated
    pre-event RGB, exactly as in Step 95F/95H.
    """
    first = rows.iloc[0]
    fields = {
        "sar_pre": "sar_pre_path",
        "sar_post": "sar_post_path",
        "rgb_pre": "s2_pre_path",
        "label": "label_path",
        "validity": "validity_path"
    }

    with ExitStack() as stack:
        datasets = {
            key: stack.enter_context(
                rasterio.open(str(first[col]))
            )
            for key, col in fields.items()
        }

        for row in rows.itertuples(index=False):
            win = Window(
                int(row.x), int(row.y),
                256, 256
            )

            pre = datasets["sar_pre"].read(
                1, window=win
            ).astype(np.float32)

            post = datasets["sar_post"].read(
                1, window=win
            ).astype(np.float32)

            rgb = datasets["rgb_pre"].read(
                [4, 3, 2], window=win
            ).astype(np.float32)

            label = datasets["label"].read(
                1, window=win
            )
            validity = datasets["validity"].read(
                1, window=win
            )

            valid = (
                (validity == 1)
                & np.isin(label, [0, 1, 2])
                & np.isfinite(pre)
                & np.isfinite(post)
                & np.isfinite(rgb).all(axis=0)
                & (pre > 0)
                & (post > 0)
            )

            rgb = np.clip(
                rgb / 10000.0, 0, 1
            ).astype(np.float32)

            inputs = {
                "sar_pre": sar_scale(pre)[None],
                "sar_post": sar_scale(post)[None],
                "ms_pre": rgb,
                "ms_post": rgb.copy()
            }

            truth = (label == 2)
            yield inputs, truth, valid

def model_forward(model, architecture, batch):
    if architecture == "UA-FloodNet":
        output = model(
            batch["sar_pre"],
            batch["sar_post"],
            batch["ms_pre"],
            batch["ms_post"]
        )
    else:
        output = model(batch)
    return get_logits(output)

# ------------------------------------------------------------
# 3. EVALUATE SCENE IN BATCHES
# ------------------------------------------------------------

@torch.inference_mode()
def evaluate_scene(model, architecture, rows):
    counts = dict(tp=0, fp=0, fn=0, tn=0)
    windows = 0
    valid_pixels = 0

    pending = []

    def flush_batch(items):
        nonlocal windows, valid_pixels

        batch = {}
        for key in (
            "sar_pre", "sar_post",
            "ms_pre", "ms_post"
        ):
            values = np.stack([
                item[0][key] for item in items
            ]).astype(np.float32)

            batch[key] = torch.from_numpy(
                np.ascontiguousarray(values)
            ).to(DEVICE)

        logits = model_forward(
            model, architecture, batch
        )

        assert logits.shape == (
            len(items), 1, 256, 256
        ), tuple(logits.shape)

        predictions = (
            logits.sigmoid() >= 0.5
        ).cpu().numpy()[:, 0]

        for i, (_, truth, valid) in enumerate(items):
            pred = predictions[i]

            counts["tp"] += int(
                np.sum(pred & truth & valid)
            )
            counts["fp"] += int(
                np.sum(pred & ~truth & valid)
            )
            counts["fn"] += int(
                np.sum(~pred & truth & valid)
            )
            counts["tn"] += int(
                np.sum(~pred & ~truth & valid)
            )
            valid_pixels += int(valid.sum())
            windows += 1

    for item in read_scene_windows(rows):
        pending.append(item)

        if len(pending) >= BATCH_SIZE:
            flush_batch(pending)
            pending.clear()

    if pending:
        flush_batch(pending)

    assert windows == len(rows)

    return {
        "windows": windows,
        "valid_pixels": valid_pixels,
        **counts
    }

# ------------------------------------------------------------
# 4. RESUMABLE SCENE-LEVEL RESULTS
# ------------------------------------------------------------

COLUMNS = [
    "architecture", "seed",
    "event_id", "area", "scene",
    "windows", "valid_pixels",
    "tp", "fp", "fn", "tn"
]

def atomic_write(data, path):
    temporary = path.with_suffix(".tmp")
    data.to_csv(temporary, index=False)
    os.replace(temporary, path)

scene_groups = list(df.groupby(
    ["event_id", "area", "scene"],
    sort=True
))

print("Distinct scenes:", len(scene_groups))

for architecture in ARCHITECTURES:
    for seed in SEEDS:

        output = (
            RUNS / f"{architecture}_seed{seed}.csv"
        )

        if output.exists():
            results = pd.read_csv(output)
            assert set(COLUMNS).issubset(results.columns)
            results = results[COLUMNS]
        else:
            results = pd.DataFrame(columns=COLUMNS)

        assert not results.duplicated(
            ["event_id", "area", "scene"]
        ).any()

        completed = set(zip(
            results.event_id.astype(str),
            results.area.astype(str),
            results.scene.astype(str)
        ))

        todo = [
            (key, rows)
            for key, rows in scene_groups
            if key not in completed
        ]

        if not todo:
            print("COMPLETE:", architecture, seed)
            continue

        print(
            f"\nMODEL {architecture} SEED {seed}: "
            f"{len(todo)} scenes remaining",
            flush=True
        )

        gc.collect()
        torch.cuda.empty_cache()

        model = load_model(architecture, seed)

        buffer = []

        for i, (key, rows) in enumerate(todo, 1):
            event, area, scene = key

            scores = evaluate_scene(
                model, architecture, rows
            )

            buffer.append({
                "architecture": architecture,
                "seed": seed,
                "event_id": event,
                "area": area,
                "scene": scene,
                **scores
            })

            if (
                len(buffer) >= SAVE_EVERY_SCENES
                or i == len(todo)
            ):
                results = pd.concat(
                    [results, pd.DataFrame(buffer)],
                    ignore_index=True
                )

                assert not results.duplicated(
                    ["event_id", "area", "scene"]
                ).any()

                atomic_write(results, output)
                buffer.clear()

                print(
                    f"  Saved {i}/{len(todo)} "
                    f"remaining scenes; "
                    f"total saved={len(results)}",
                    flush=True
                )

        del model
        torch.cuda.empty_cache()
        gc.collect()

# ------------------------------------------------------------
# 5. COMBINE ALL SCENE RESULTS
# ------------------------------------------------------------

parts = []

for architecture in ARCHITECTURES:
    for seed in SEEDS:
        file = RUNS / f"{architecture}_seed{seed}.csv"
        assert file.is_file(), file

        part = pd.read_csv(file)

        assert len(part) == len(scene_groups), (
            architecture, seed, len(part)
        )
        parts.append(part)

all_scenes = pd.concat(parts, ignore_index=True)

assert len(all_scenes) == (
    len(scene_groups) * len(SEEDS) * len(ARCHITECTURES)
)

ALL_PATH = OUT / "step97D_all_scene_results.csv"
all_scenes.to_csv(ALL_PATH, index=False)

# ------------------------------------------------------------
# 6. EVENT-LEVEL CONFUSION MATRICES AND METRICS
# ------------------------------------------------------------

group_cols = ["architecture", "seed", "event_id"]

event = (
    all_scenes.groupby(group_cols)[
        ["windows", "valid_pixels",
         "tp", "fp", "fn", "tn"]
    ]
    .sum()
    .reset_index()
)

assert len(event) == 4 * 4 * 9

event["iou"] = (
    event.tp / (event.tp + event.fp + event.fn)
)
event["f1"] = (
    2 * event.tp /
    (2 * event.tp + event.fp + event.fn)
)

assert event.iou.notna().all()

EVENT_PATH = OUT / "step97D_event_level_results.csv"
event.to_csv(EVENT_PATH, index=False)

# Compute per-seed macro event IoU:
# Each of the nine events receives equal weight.
macro = (
    event.groupby(["architecture", "seed"])[
        ["iou", "f1"]
    ]
    .mean()
    .reset_index()
    .rename(columns={
        "iou": "macro_event_iou",
        "f1": "macro_event_f1"
    })
)

# Also compute pixel-pooled IoU for transparency.
pooled = (
    event.groupby(["architecture", "seed"])[
        ["tp", "fp", "fn", "tn", "valid_pixels"]
    ]
    .sum()
    .reset_index()
)

pooled["pooled_iou"] = (
    pooled.tp / (pooled.tp + pooled.fp + pooled.fn)
)

pooled["pooled_f1"] = (
    2 * pooled.tp /
    (2 * pooled.tp + pooled.fp + pooled.fn)
)

summary = macro.merge(
    pooled[
        ["architecture", "seed",
         "pooled_iou", "pooled_f1"]
    ],
    on=["architecture", "seed"],
    validate="one_to_one"
)

SUMMARY_PATH = (
    OUT / "step97D_nine_event_seed_summary.csv"
)
summary.to_csv(SUMMARY_PATH, index=False)

print("\n" + "=" * 76)
print("STEP 97D COMPLETED — NINE-EVENT EXTERNAL EVALUATION")
print("=" * 76)

print("\nFOUR-SEED SUMMARY")
print(
    summary.groupby("architecture")[
        [
            "macro_event_iou",
            "macro_event_f1",
            "pooled_iou",
            "pooled_f1"
        ]
    ]
    .agg(["mean", "std"])
    .round(5)
    .to_string()
)

print("\nPER-EVENT MEAN IoU")
print(
    event.pivot_table(
        index="event_id",
        columns="architecture",
        values="iou",
        aggfunc="mean"
    ).round(5).to_string()
)

print("\nSaved:", ALL_PATH)
print("Saved:", EVENT_PATH)
print("Saved:", SUMMARY_PATH)
print("No checkpoints modified. No retraining performed.")


In [ ]:

# ============================================================
# STEP 98 — FINAL NINE-EVENT STATISTICS AND FIGURES
# Paired event comparisons | 9 events | 4 training seeds
# Colored scientific figures | PNG 600 dpi, PDF, SVG
# No training or checkpoint changes
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_1samp, t
from statsmodels.stats.multitest import multipletests
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
BASE = ROOT / "v4/final_analysis/generalization_study"
OUT = BASE / "step98_final_statistics"
OUT.mkdir(parents=True, exist_ok=True)

event = pd.read_csv(
    BASE / "step97D_event_level_results.csv"
)

seed_summary = pd.read_csv(
    BASE / "step97D_nine_event_seed_summary.csv"
)

manifest = pd.read_csv(
    BASE / "step97C_heldout_window_manifest.csv"
)

NAMES = {
    "UA-FloodNet": "UA-FloodNet",
    "standard_unet": "Standard U-Net",
    "siamese_unet": "Siamese U-Net",
    "two_stage_fusion_unet": "Two-stage fusion"
}

COLORS = {
    "UA-FloodNet": "#7357B5",
    "Standard U-Net": "#1676B8",
    "Siamese U-Net": "#E4872A",
    "Two-stage fusion": "#159C77"
}

ORDER = [
    "UA-FloodNet",
    "Standard U-Net",
    "Siamese U-Net",
    "Two-stage fusion"
]

SEEDS = [7, 17, 37, 42]
EVENTS = sorted(manifest.event_id.unique())

event["model"] = event.architecture.map(NAMES)
seed_summary["model"] = seed_summary.architecture.map(NAMES)

assert len(EVENTS) == 9
assert len(manifest) == 14151
assert len(event) == 144
assert len(seed_summary) == 16
assert event.model.notna().all()
assert not event.duplicated(
    ["event_id", "model", "seed"]
).any()

# Every model and seed must cover identical events.
for model in ORDER:
    for seed in SEEDS:
        part = event[
            (event.model == model) &
            (event.seed == seed)
        ]
        assert set(part.event_id) == set(EVENTS)

# Verify that support (windows and valid pixels)
# is consistent across all architectures and seeds.
support = event.groupby("event_id")[
    ["windows", "valid_pixels"]
].nunique()
assert (support == 1).all().all()

# ------------------------------------------------------------
# 1. VERIFIED DESCRIPTIVE TABLE
# ------------------------------------------------------------

description = (
    seed_summary.groupby("model")[
        ["macro_event_iou", "macro_event_f1",
         "pooled_iou", "pooled_f1"]
    ]
    .agg(["mean", "std"])
    .reindex(ORDER)
)

description.to_csv(
    OUT / "Table_98A_Nine_Event_Descriptives.csv"
)

print("=" * 72)
print("STEP 98 — NINE-EVENT SCIENTIFIC VALIDATION")
print("=" * 72)

print("\nFOUR-SEED DESCRIPTIVES")
print(description.round(5).to_string())

# ------------------------------------------------------------
# 2. NINE INDEPENDENT EVENT-LEVEL EFFECTS
# ------------------------------------------------------------

event_means = (
    event.groupby(["event_id", "model"])["iou"]
    .mean()
    .unstack("model")
    .reindex(index=EVENTS, columns=ORDER)
)

assert event_means.notna().all().all()

event_means.to_csv(
    OUT / "Table_98B_Per_Event_Mean_IoU.csv"
)

reference = event_means["UA-FloodNet"]

comparison_rows = []

for model in ORDER[1:]:
    differences = (
        event_means[model] - reference
    ).to_numpy(dtype=float)

    n = len(differences)
    mean = float(np.mean(differences))
    sd = float(np.std(differences, ddof=1))
    se = sd / np.sqrt(n)

    if sd == 0:
        p = 1.0 if mean == 0 else 0.0
        lower = upper = mean
    else:
        p = float(
            ttest_1samp(differences, 0).pvalue
        )
        critical = t.ppf(0.975, df=n-1)
        lower = mean - critical * se
        upper = mean + critical * se

    comparison_rows.append({
        "comparison": f"{model} - UA-FloodNet",
        "model": model,
        "n_events": n,
        "mean_difference": mean,
        "ci95_lower": lower,
        "ci95_upper": upper,
        "p_unadjusted": p,
        "events_baseline_higher": int(
            (differences > 0).sum()
        )
    })

statistics = pd.DataFrame(comparison_rows)

statistics["p_holm"] = multipletests(
    statistics.p_unadjusted,
    method="holm"
)[1]

statistics["holm_significant"] = (
    statistics.p_holm < 0.05
)

statistics.to_csv(
    OUT / "Table_98C_Paired_Event_Comparisons.csv",
    index=False
)

print("\nPAIRED EVENT-LEVEL COMPARISONS")
print(
    statistics.round(6).to_string(index=False)
)

# ------------------------------------------------------------
# 3. FIGURE A — EVENT-BY-MODEL IoU HEATMAP
# ------------------------------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

fig, ax = plt.subplots(figsize=(9.8, 4.9))

values = event_means.to_numpy()

im = ax.imshow(
    values, cmap="YlGnBu",
    vmin=0, vmax=max(0.6, float(values.max())),
    aspect="auto"
)

ax.set_xticks(range(len(ORDER)), ORDER)
ax.set_yticks(range(len(EVENTS)), EVENTS)
ax.set_xlabel("Architecture")
ax.set_ylabel("Held-out flood event")

for i in range(len(EVENTS)):
    for j in range(len(ORDER)):
        v = values[i, j]
        ax.text(
            j, i, f"{v:.3f}",
            ha="center", va="center",
            fontsize=8.5,
            color="white" if im.norm(v) > 0.52 else "#24313B"
        )

ax.spines[:].set_visible(False)

cbar = fig.colorbar(
    im, ax=ax, shrink=0.82, pad=0.025
)
cbar.set_label("Flood IoU")

fig.tight_layout()

for extension in ("png", "pdf", "svg"):
    fig.savefig(
        OUT / f"Figure_98A_Event_IoU_Heatmap.{extension}",
        dpi=600 if extension == "png" else None,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 4. FIGURE B — COLORED PAIRED EVENT EFFECTS
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9.3, 4.8))

for i, model in enumerate(ORDER[1:]):
    diff = (
        event_means[model] - reference
    ).to_numpy()

    color = COLORS[model]
    y = np.full(len(diff), i, dtype=float)

    # Fixed stagger prevents perfectly overlapping points.
    offsets = np.linspace(-0.18, 0.18, len(diff))

    ax.scatter(
        diff, y + offsets,
        color=color, s=38, alpha=0.73,
        edgecolors="white", linewidths=0.6,
        label=model, zorder=3
    )

    row = statistics[
        statistics.model == model
    ].iloc[0]

    lo = float(row.ci95_lower)
    hi = float(row.ci95_upper)
    mean = float(row.mean_difference)

    ax.plot(
        [lo, hi], [i+0.27, i+0.27],
        color=color, linewidth=2.6
    )
    ax.plot(
        [lo, lo], [i+0.21, i+0.33],
        color=color, linewidth=1.5
    )
    ax.plot(
        [hi, hi], [i+0.21, i+0.33],
        color=color, linewidth=1.5
    )
    ax.scatter(
        mean, i+0.27,
        color=color, s=100,
        marker="D", edgecolors="white",
        linewidths=1, zorder=5
    )

ax.axvline(
    0, color="#596777",
    linewidth=1.2, linestyle="--"
)

ax.set_yticks(range(3), ORDER[1:])
ax.set_ylim(-0.5, 2.65)
ax.invert_yaxis()

ax.set_xlabel(
    "Event-level IoU difference (baseline − UA-FloodNet)"
)

ax.grid(
    axis="x",
    alpha=0.18,
    linestyle=":"
)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)

fig.tight_layout()

for extension in ("png", "pdf", "svg"):
    fig.savefig(
        OUT / f"Figure_98B_Paired_Event_Effects.{extension}",
        dpi=600 if extension == "png" else None,
        bbox_inches="tight",
        facecolor="white"
    )

plt.show()
plt.close(fig)

print("\n" + "=" * 72)
print("STEP 98 COMPLETE")
print("=" * 72)
print("Verified independent event IDs:", len(EVENTS))
print("Paired comparisons:", len(statistics))
print(
    "Holm-significant comparisons:",
    int(statistics.holm_significant.sum())
)
print("All figures and tables saved to:", OUT)
print("No models retrained or checkpoints modified.")


In [ ]:

from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet/v4/final_analysis"
EXT = ROOT / "generalization_study"
OUT = ROOT / "final_professor_package"
OUT.mkdir(parents=True, exist_ok=True)

event = pd.read_csv(EXT / "step97D_event_level_results.csv")
seed = pd.read_csv(EXT / "step97D_nine_event_seed_summary.csv")
stats = pd.read_csv(
    EXT / "step98_final_statistics/"
    "Table_98C_Paired_Event_Comparisons.csv"
)
eff = pd.read_csv(
    ROOT / "efficiency_study/step96_seed_efficiency.csv"
)

assert len(event) == 144
assert len(seed) == 16
assert len(stats) == 3
assert len(eff) == 16
assert event.event_id.nunique() == 9

assert not event.duplicated(
    ["architecture", "seed", "event_id"]
).any()

assert not seed.duplicated(
    ["architecture", "seed"]
).any()

assert (event[["tp", "fp", "fn", "tn"]] >= 0).all().all()

assert np.all(
    event.tp + event.fp + event.fn + event.tn
    == event.valid_pixels
)

calculated_iou = (
    event.tp / (event.tp + event.fp + event.fn)
)
calculated_f1 = (
    2 * event.tp / (2 * event.tp + event.fp + event.fn)
)

assert np.allclose(event.iou, calculated_iou, atol=1e-10)
assert np.allclose(event.f1, calculated_f1, atol=1e-10)

macro = event.groupby(
    ["architecture", "seed"]
)[["iou", "f1"]].mean().reset_index()

check = seed.merge(
    macro,
    on=["architecture", "seed"],
    validate="one_to_one"
)

assert np.allclose(
    check.macro_event_iou, check.iou, atol=1e-10
)
assert np.allclose(
    check.macro_event_f1, check.f1, atol=1e-10
)

# Independent recalculation of the pooled metrics.
pooled = event.groupby(
    ["architecture", "seed"]
)[["tp", "fp", "fn"]].sum().reset_index()

pooled["pooled_iou_recalc"] = (
    pooled.tp / (pooled.tp + pooled.fp + pooled.fn)
)

checked = seed.merge(
    pooled,
    on=["architecture", "seed"],
    validate="one_to_one"
)

assert np.allclose(
    checked.pooled_iou,
    checked.pooled_iou_recalc,
    atol=1e-10
)

# Preserve hashes of source evidence.
source_files = [
    EXT / "step97D_event_level_results.csv",
    EXT / "step97D_nine_event_seed_summary.csv",
    EXT / "step98_final_statistics/"
          "Table_98C_Paired_Event_Comparisons.csv",
    ROOT / "efficiency_study/step96_seed_efficiency.csv",
    ROOT / "benchmark_study/evaluation_93/"
           "baseline_four_seed_summary.csv",
    ROOT / "ablation_study/evaluation_88B/"
           "four_seed_architecture_summary.csv"
]

manifest = {
    "study": "UA-FloodNet",
    "event_level_records": len(event),
    "event_ids": sorted(event.event_id.unique().tolist()),
    "seed_summary_records": len(seed),
    "efficiency_records": len(eff),
    "numerical_consistency": "PASS",
    "limitations": [
        "OMBRIA geographic/event independence unverified",
        "GEOID post-event optical unavailable",
        "External evaluation uses duplicated pre-event optical",
        "Cross-dataset radiometric adaptation differs from OMBRIA",
        "Held-out GEOID event IDs disjoint from GEOID main catalog",
        "Held-out geographic overlap with OMBRIA unverified"
    ],
    "sha256": {
        str(p.relative_to(ROOT)): hashlib.sha256(
            p.read_bytes()
        ).hexdigest()
        for p in source_files
    }
}

manifest_file = OUT / "final_evidence_manifest.json"
manifest_file.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("=" * 70)
print("STEP 99A — FINAL SCIENTIFIC AUDIT")
print("=" * 70)
print("Event records verified:", len(event))
print("Independent GEOID event IDs:", event.event_id.nunique())
print("Seed summaries verified:", len(seed))
print("Efficiency records verified:", len(eff))
print("Numerical consistency: PASS")
print("Evidence manifest:", manifest_file)
print("No source results or checkpoints changed.")


In [ ]:

# ================================================================
# STEP 99B — FINAL TWO GEOID-FLOOD FIGURES
# Based strictly on original Step 97D and Step 98 results
#
# FIGURE 1: Nine-event flood IoU heatmap
# FIGURE 2: Paired event differences with 95% confidence intervals
#
# Exports: 900 DPI PNG + PDF + SVG
# No training, no checkpoint changes, no new statistical tests
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from google.colab import drive

MOUNT = Path("/content/ua_ablation_drive")
if not (MOUNT / "MyDrive").is_dir():
    drive.mount(str(MOUNT))

ROOT = MOUNT / "MyDrive/UA_FloodNet"
EXT = ROOT / "v4/final_analysis/generalization_study"
STATS = EXT / "step98_final_statistics"
OUT = STATS / "FINAL_TWO_FIGURES_900DPI"
OUT.mkdir(parents=True, exist_ok=True)

event = pd.read_csv(
    EXT / "step97D_event_level_results.csv"
)
paired = pd.read_csv(
    STATS / "Table_98C_Paired_Event_Comparisons.csv"
)

NAMES = {
    "UA-FloodNet": "UA-FloodNet",
    "standard_unet": "Standard U-Net",
    "siamese_unet": "Siamese U-Net",
    "two_stage_fusion_unet": "Two-stage fusion"
}

ORDER = [
    "UA-FloodNet",
    "Standard U-Net",
    "Siamese U-Net",
    "Two-stage fusion"
]

COLORS = {
    "Standard U-Net": "#2879C4",
    "Siamese U-Net": "#E78A2D",
    "Two-stage fusion": "#18A37F"
}

event["model"] = event["architecture"].map(NAMES)

assert len(event) == 144
assert event["model"].notna().all()
assert event["event_id"].nunique() == 9
assert not event.duplicated(
    ["event_id", "model", "seed"]
).any()

events = sorted(event["event_id"].unique())

means = (
    event.groupby(["event_id", "model"])["iou"]
    .mean()
    .unstack("model")
    .reindex(index=events, columns=ORDER)
)

assert means.notna().all().all()
assert len(paired) == 3

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white"
})

def save_figure(fig, name):
    for ext in ("png", "pdf", "svg"):
        path = OUT / f"{name}.{ext}"
        kwargs = {"dpi": 900} if ext == "png" else {}
        fig.savefig(
            path,
            bbox_inches="tight",
            pad_inches=0.12,
            facecolor="white",
            **kwargs
        )
        print("SAVED:", path.name)

# ================================================================
# FIGURE 1 — NINE-EVENT IoU HEATMAP
# ================================================================

fig, ax = plt.subplots(figsize=(10.6, 6.0))

values = means.to_numpy(dtype=float)

vmax = max(0.6, float(values.max()))

image = ax.imshow(
    values,
    cmap="YlGnBu",
    vmin=0,
    vmax=vmax,
    aspect="auto",
    interpolation="nearest"
)

ax.set_xticks(np.arange(len(ORDER)))
ax.set_xticklabels(ORDER, fontsize=10)
ax.set_yticks(np.arange(len(events)))
ax.set_yticklabels(events, fontsize=10)

ax.set_xlabel("Flood segmentation architecture", labelpad=12)
ax.set_ylabel("Independent held-out flood event", labelpad=10)

# Fine cell boundaries
ax.set_xticks(
    np.arange(-0.5, len(ORDER), 1),
    minor=True
)
ax.set_yticks(
    np.arange(-0.5, len(events), 1),
    minor=True
)
ax.grid(
    which="minor",
    color="white",
    linewidth=1.7
)
ax.tick_params(which="minor", bottom=False, left=False)

for i in range(len(events)):
    for j in range(len(ORDER)):
        value = values[i, j]
        ax.text(
            j, i, f"{value:.3f}",
            ha="center",
            va="center",
            fontsize=10,
            fontweight="medium",
            color="white" if value >= 0.37 else "#172B3D"
        )

for spine in ax.spines.values():
    spine.set_visible(False)

colorbar = fig.colorbar(
    image,
    ax=ax,
    fraction=0.032,
    pad=0.025
)
colorbar.set_label("Flood IoU", fontsize=10)

fig.subplots_adjust(
    left=0.13,
    right=0.90,
    top=0.97,
    bottom=0.14
)

save_figure(
    fig,
    "Figure_98A_Nine_Event_IoU_Heatmap_FINAL"
)

plt.show()
plt.close(fig)

# ================================================================
# FIGURE 2 — PAIRED-EVENT EFFECT ESTIMATION PLOT
# ================================================================

fig, ax = plt.subplots(figsize=(10.8, 5.4))

reference = means["UA-FloodNet"]

comparison_order = [
    "Standard U-Net",
    "Siamese U-Net",
    "Two-stage fusion"
]

for i, name in enumerate(comparison_order):

    difference = (
        means[name] - reference
    ).to_numpy(dtype=float)

    color = COLORS[name]

    # All nine observed event-level paired effects.
    vertical_offsets = np.linspace(-0.17, 0.17, 9)

    ax.scatter(
        difference,
        np.full(9, i) + vertical_offsets,
        s=58,
        color=color,
        alpha=0.80,
        edgecolor="white",
        linewidth=0.85,
        zorder=4
    )

    record = paired.loc[
        paired["model"] == name
    ]

    assert len(record) == 1

    row = record.iloc[0]

    estimate = float(row["mean_difference"])
    low = float(row["ci95_lower"])
    high = float(row["ci95_upper"])
    p_holm = float(row["p_holm"])

    # Check against actual event differences.
    assert np.isclose(
        estimate,
        np.mean(difference),
        atol=1e-6
    ), f"Statistics mismatch: {name}"

    # 95% paired-event CI
    y_ci = i + 0.28

    ax.plot(
        [low, high],
        [y_ci, y_ci],
        color=color,
        linewidth=2.7,
        solid_capstyle="round",
        zorder=5
    )

    ax.plot(
        [low, low],
        [y_ci-0.07, y_ci+0.07],
        color=color,
        linewidth=1.7
    )
    ax.plot(
        [high, high],
        [y_ci-0.07, y_ci+0.07],
        color=color,
        linewidth=1.7
    )

    ax.scatter(
        [estimate],
        [y_ci],
        color=color,
        marker="D",
        s=120,
        edgecolor="white",
        linewidth=1.2,
        zorder=6
    )

    # Indicate Holm significance, without altering p-values.
    if p_holm < 0.05:
        ax.text(
            high + 0.012,
            y_ci,
            "*",
            color=color,
            fontsize=19,
            fontweight="bold",
            va="center",
            ha="left"
        )

ax.axvline(
    0,
    color="#64748B",
    linestyle=(0, (4, 3)),
    linewidth=1.2,
    zorder=1
)

ax.set_yticks(
    np.arange(3),
    comparison_order
)

ax.set_ylim(-0.48, 2.63)
ax.invert_yaxis()

ax.set_xlabel(
    "Difference in event-level IoU (baseline − UA-FloodNet)",
    labelpad=12
)

ax.set_ylabel("Benchmark architecture", labelpad=12)

ax.grid(
    axis="x",
    color="#D9E1E8",
    linestyle=":",
    linewidth=0.8
)

ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)

legend = [
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor="#75859A",
        markeredgecolor="white",
        markersize=8,
        label="Individual flood event"
    ),
    Line2D(
        [0], [0],
        marker="D",
        color="#40536B",
        linewidth=2,
        markersize=7,
        label="Mean paired effect ± 95% CI"
    )
]

ax.legend(
    handles=legend,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.13),
    ncol=2,
    frameon=False,
    fontsize=9
)

fig.subplots_adjust(
    left=0.20,
    right=0.97,
    top=0.85,
    bottom=0.19
)

save_figure(
    fig,
    "Figure_98B_Paired_Event_Effects_FINAL"
)

plt.show()
plt.close(fig)

# ================================================================
# FINAL PACKAGE VALIDATION
# ================================================================

expected = [
    OUT / f"{name}.{ext}"
    for name in [
        "Figure_98A_Nine_Event_IoU_Heatmap_FINAL",
        "Figure_98B_Paired_Event_Effects_FINAL"
    ]
    for ext in ["png", "pdf", "svg"]
]

assert all(p.is_file() and p.stat().st_size > 0
           for p in expected)

print("\n" + "=" * 72)
print("STEP 99B — FINAL FIGURES COMPLETE")
print("=" * 72)
print("Scientific figures: 2")
print("Total exported files:", len(expected))
print("Raster resolution: 900 DPI")
print("Vector masters: PDF and SVG")
print("Output directory:", OUT)
print("All source statistics preserved.")
print("No checkpoints or training results modified.")
